# KDIC V1.5 + Hybrid 7:3 Min-Max + BGE Reranker + Parent-Child + 문맥 CLARIFY + 답변 D안

이 노트북은 기존 9:1 D안 챗봇에서 다음 항목만 변경·보강한 버전입니다.

- Dense Structured V2 : BM25 Nori-none = `7 : 3`
- Min/Max 결합 후보 depth `20`
- `BAAI/bge-reranker-v2-m3` CrossEncoder로 후보 20개 재정렬 후 Top-5 선택
- Reranker Top-5 Child를 `parent_doc_id` 기준 Parent 문맥으로 확장하고, 동일 Parent는 Evidence Pack에서 1회만 사용
- 업무가 없는 짧은 질문은 검색 전에 `CLARIFY`
- 단일 업무가 확인된 후속질문은 이전 업무와 결합한 독립 검색 질의로 복원
- 복수 업무로 모호하면 선택지를 제시하고 검색·Reranker·답변 D안을 중단
- 기존 V1.5 교차업무 분해와 답변 D안은 유지


## 실행 순서

1. 코랩에 `KDIC_output.zip`과 `kdic_dense_structured_v2_embeddings.jsonl`을 업로드합니다.
2. 모든 셀을 위에서 아래로 실행합니다.
3. HCX API 키를 입력합니다.
4. 마지막 입력창에 질문합니다.

같은 질문의 HCX-007 분해 결과는 `/content/kdic_v15_chat_decomposition_cache.jsonl`에 저장됩니다.

## 1. 의존성 설치

Elasticsearch 서버는 `8.15.3`, Python 클라이언트는 실제 배포되어 있는 같은 minor 계열의 `8.15.1`을 사용합니다. `elasticsearch==8.15.3`이라는 Python 패키지는 배포되어 있지 않으므로 해당 핀을 사용하면 설치 셀에서 바로 실패합니다.


In [1]:
!pip -q install "openai>=1.68,<2" "elasticsearch==8.15.1" "tqdm>=4.66,<5" "ipywidgets>=8.1,<9" "markdown>=3.6,<4" "pandas>=2.0,<3" "requests>=2.31,<3" "sentence-transformers>=3.0,<4"


## 2. Elasticsearch 8.15.3 + Nori 준비

이 셀은 Colab에서 자주 발생하는 다음 문제를 피하도록 구성했습니다.

- Elasticsearch를 root로 실행해서 발생하는 시작 실패
- 일반 사용자가 `/content`에 PID 파일을 쓰지 못하는 권한 오류
- 노트북 재실행 때 `elasticsearch.yml` 설정이 계속 중복되는 문제
- 기존 9200 포트 프로세스와의 충돌
- 부분 다운로드·부분 압축 해제로 인한 실행 파일 손상
- Nori 플러그인이 없는 상태에서 인덱스를 생성하는 문제
- Colab cgroup v2 경로가 샌드박스 밖으로 해석되어 `AccessControlException`이 발생하는 문제

설치 폴더, 데이터, 로그, PID 파일을 모두 A안 전용 경로로 분리합니다. Colab의 cgroup 경로는
Elasticsearch 컨테이너 실행 방식과 동일하게 루트(`/`)로 명시하여, Elasticsearch가
`/sys/fs/cgroup/../../jupyter-children/cpu.stat` 같은 잘못된 경로를 읽지 않도록 합니다.


In [2]:
%%bash
set -Eeuo pipefail

ES_VERSION="8.15.3"
ES_USER="kdic_es_a"
INSTALL_ROOT="/content/kdic_es_a_dist"
ES_HOME="${INSTALL_ROOT}/elasticsearch-${ES_VERSION}"
ES_RUNTIME="/content/kdic_es_a_runtime"
ES_ARCHIVE="${INSTALL_ROOT}/elasticsearch-${ES_VERSION}.tar.gz"
ES_PID_FILE="${ES_RUNTIME}/elasticsearch.pid"
ES_LOG_FILE="${ES_RUNTIME}/logs/kdic-a.log"
ES_HTTP_URL="http://127.0.0.1:9220"

show_diagnostics() {
  echo "[Elasticsearch 진단]" >&2
  if [ -f "${ES_PID_FILE}" ]; then
    echo "PID file: $(cat "${ES_PID_FILE}" 2>/dev/null || true)" >&2
  fi
  if [ -f "${ES_LOG_FILE}" ]; then
    tail -n 160 "${ES_LOG_FILE}" >&2 || true
  elif [ -d "${ES_RUNTIME}/logs" ]; then
    tail -n 160 "${ES_RUNTIME}"/logs/*.log >&2 2>/dev/null || true
  fi
}
trap show_diagnostics ERR

case "$(uname -m)" in
  x86_64) ES_ARCH="x86_64" ;;
  aarch64|arm64) ES_ARCH="aarch64" ;;
  *) echo "지원하지 않는 CPU 아키텍처: $(uname -m)" >&2; exit 1 ;;
esac

mkdir -p "${INSTALL_ROOT}" "${ES_RUNTIME}/data" "${ES_RUNTIME}/logs" "${ES_RUNTIME}/tmp"

if ! id "${ES_USER}" >/dev/null 2>&1; then
  useradd --system --create-home --home-dir "/content/${ES_USER}" --shell /usr/sbin/nologin "${ES_USER}"
fi

if curl -fsS "${ES_HTTP_URL}" >/dev/null 2>&1; then
  RUNNING_VERSION="$(curl -fsS "${ES_HTTP_URL}" | python3 -c 'import json,sys; print(json.load(sys.stdin)["version"]["number"])')"
  if [ "${RUNNING_VERSION}" != "${ES_VERSION}" ]; then
    echo "9220 포트에 Elasticsearch ${RUNNING_VERSION}가 실행 중입니다. Colab 런타임을 재시작하세요." >&2
    exit 1
  fi
  if ! curl -fsS "${ES_HTTP_URL}/_nodes/plugins" | python3 -c 'import json,sys; d=json.load(sys.stdin); assert any(p.get("name")=="analysis-nori" for n in d["nodes"].values() for p in n.get("plugins", []))'; then
    echo "실행 중인 9220 Elasticsearch에 analysis-nori가 없습니다. Colab 런타임을 재시작하세요." >&2
    exit 1
  fi
  echo "Elasticsearch ${RUNNING_VERSION} + analysis-nori 재사용"
  exit 0
fi

# A안 전용 PID는 남아 있지만 HTTP가 열리지 않으면 해당 프로세스만 정리한 뒤 재시작합니다.
if [ -f "${ES_PID_FILE}" ]; then
  OLD_PID="$(cat "${ES_PID_FILE}" 2>/dev/null || true)"
  if [ -n "${OLD_PID}" ] && kill -0 "${OLD_PID}" 2>/dev/null; then
    kill "${OLD_PID}" 2>/dev/null || true
    for _ in $(seq 1 15); do
      if ! kill -0 "${OLD_PID}" 2>/dev/null; then
        break
      fi
      sleep 1
    done
    if kill -0 "${OLD_PID}" 2>/dev/null; then
      kill -9 "${OLD_PID}" 2>/dev/null || true
    fi
  fi
  rm -f "${ES_PID_FILE}"
fi

if [ ! -x "${ES_HOME}/bin/elasticsearch" ]; then
  DOWNLOAD_URL="https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-${ES_VERSION}-linux-${ES_ARCH}.tar.gz"
  TEMP_ARCHIVE="${ES_ARCHIVE}.part"
  rm -f "${TEMP_ARCHIVE}"
  curl -fL --retry 5 --retry-delay 3 --connect-timeout 20 \
    "${DOWNLOAD_URL}" -o "${TEMP_ARCHIVE}"
  tar -tzf "${TEMP_ARCHIVE}" >/dev/null
  mv "${TEMP_ARCHIVE}" "${ES_ARCHIVE}"
  tar -xzf "${ES_ARCHIVE}" -C "${INSTALL_ROOT}"
fi

chown -R "${ES_USER}:${ES_USER}" "${ES_HOME}" "${ES_RUNTIME}" "/content/${ES_USER}"

if ! runuser -u "${ES_USER}" -- "${ES_HOME}/bin/elasticsearch-plugin" list | grep -qx "analysis-nori"; then
  runuser -u "${ES_USER}" -- "${ES_HOME}/bin/elasticsearch-plugin" install --batch analysis-nori
fi

CONFIG_FILE="${ES_HOME}/config/elasticsearch.yml"
python3 - "${CONFIG_FILE}" "${ES_RUNTIME}" <<'PY'
from pathlib import Path
import sys

config_path = Path(sys.argv[1])
runtime = Path(sys.argv[2])
config_path.write_text(
    "\n".join([
        "cluster.name: kdic-colab-answer-a",
        "node.name: kdic-colab-answer-a-node",
        f"path.data: {runtime / 'data'}",
        f"path.logs: {runtime / 'logs'}",
        "network.host: 127.0.0.1",
        "http.port: 9220",
        "transport.port: 9320",
        "discovery.type: single-node",
        "xpack.security.enabled: false",
        "xpack.security.enrollment.enabled: false",
        "xpack.security.http.ssl.enabled: false",
        "xpack.security.transport.ssl.enabled: false",
        "xpack.ml.enabled: false",
        "ingest.geoip.downloader.enabled: false",
        "cluster.routing.allocation.disk.threshold_enabled: false",
        "node.store.allow_mmap: false",
        "bootstrap.memory_lock: false",
        "",
    ]),
    encoding="utf-8",
)
PY
chown "${ES_USER}:${ES_USER}" "${CONFIG_FILE}"

if [ -f "${ES_PID_FILE}" ]; then
  OLD_PID="$(cat "${ES_PID_FILE}" 2>/dev/null || true)"
  if [ -n "${OLD_PID}" ] && kill -0 "${OLD_PID}" 2>/dev/null; then
    echo "기존 Elasticsearch PID ${OLD_PID}의 시작을 기다립니다."
  else
    rm -f "${ES_PID_FILE}"
  fi
fi

if [ ! -f "${ES_PID_FILE}" ]; then
  runuser -u "${ES_USER}" -- env \
    ES_JAVA_OPTS="-Xms512m -Xmx512m -Djava.io.tmpdir=${ES_RUNTIME}/tmp -Des.cgroups.hierarchy.override=/" \
    "${ES_HOME}/bin/elasticsearch" -d -p "${ES_PID_FILE}"
fi

for _ in $(seq 1 120); do
  if curl -fsS "${ES_HTTP_URL}" >/dev/null 2>&1; then
    break
  fi
  if [ -f "${ES_PID_FILE}" ]; then
    PID="$(cat "${ES_PID_FILE}" 2>/dev/null || true)"
    if [ -n "${PID}" ] && ! kill -0 "${PID}" 2>/dev/null; then
      echo "Elasticsearch 프로세스가 시작 중 종료되었습니다." >&2
      exit 1
    fi
  fi
  sleep 1
done

curl -fsS "${ES_HTTP_URL}" >/dev/null
curl -fsS "${ES_HTTP_URL}/_nodes/jvm" | python3 -c '
import json, sys
data = json.load(sys.stdin)
args = [arg for node in data["nodes"].values() for arg in node["jvm"].get("input_arguments", [])]
assert "-Des.cgroups.hierarchy.override=/" in args, args
'
curl -fsS "${ES_HTTP_URL}/_nodes/plugins" | python3 -c 'import json,sys; d=json.load(sys.stdin); assert any(p.get("name")=="analysis-nori" for n in d["nodes"].values() for p in n.get("plugins", []))'
curl -fsS -X POST "${ES_HTTP_URL}/_analyze" \
  -H 'Content-Type: application/json' \
  -d '{"tokenizer":{"type":"nori_tokenizer","decompound_mode":"none"},"text":"예금자보호제도"}' >/dev/null

echo "Elasticsearch ${ES_VERSION} + analysis-nori 준비 완료: ${ES_HTTP_URL}"


[2026-08-19T03:37:59,650][INFO ][o.e.n.NativeAccess       ] [kdic-colab-answer-a-node] Using native vector library; to disable start with -Dorg.elasticsearch.nativeaccess.enableVectorLibrary=false
[2026-08-19T03:37:59,878][INFO ][o.e.n.NativeAccess       ] [kdic-colab-answer-a-node] Using [jdk] native provider and native methods for [Linux]
[2026-08-19T03:38:00,231][INFO ][o.a.l.i.v.PanamaVectorizationProvider] [kdic-colab-answer-a-node] Java vector incubator API enabled; uses preferredBitSize=512; FMA enabled
[2026-08-19T03:38:01,093][INFO ][o.e.n.Node               ] [kdic-colab-answer-a-node] version[8.15.3], pid[10810], build[tar/f97532e680b555c3a05e73a74c28afb666923018/2024-10-09T22:08:00.328917561Z], OS[Linux/6.6.122+/amd64], JVM[Oracle Corporation/OpenJDK 64-Bit Server VM/22.0.1/22.0.1+8-16]
[2026-08-19T03:38:01,094][INFO ][o.e.n.Node               ] [kdic-colab-answer-a-node] JVM home [/content/kdic_es_a_dist/elasticsearch-8.15.3/jdk], using bundled JDK [true]
[2026-08-19T03:38

Aug 19, 2026 3:37:59 AM sun.util.locale.provider.LocaleProviderAdapter <clinit>


## 1. 설정

In [4]:
from __future__ import annotations

import getpass
import hashlib
import json
import math
import os
import re
import shutil
import zipfile
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Literal

import ipywidgets as widgets
import numpy as np
from elasticsearch import Elasticsearch, helpers
from IPython.display import JSON, Markdown, clear_output, display
from openai import BadRequestError, OpenAI
from tqdm.auto import tqdm

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass


# ---------- 데이터 / 캐시 ----------
# 경로를 직접 지정하지 않으면 ZIP 업로드 창이 열립니다.
DATA_SOURCE: str | None = None
FORCE_REUPLOAD_ZIP: bool = False  # True로 바꾸면 기존 업로드된 ZIP을 무시하고 새로 업로드합니다.
DENSE_CACHE_FILENAME = "kdic_dense_structured_v2_embeddings.jsonl"
DENSE_CACHE_PATH = Path("/content") / DENSE_CACHE_FILENAME

# ---------- HCX ----------
HCX_BASE_URL = "https://clovastudio.stream.ntruss.com/v1/openai"
HCX_EMBEDDING_MODEL = "bge-m3"
HCX_CHAT_MODEL = "HCX-005"
HCX_ENCODING_FORMAT = "float"
HCX_REQUEST_TIMEOUT = 120.0
HCX_MAX_RETRIES = 4

# ---------- 확정 검색 조건 ----------
DENSE_WEIGHT = 0.7
BM25_WEIGHT = 0.3
QUERY_FUSION_RRF_K = 10
CANDIDATE_DEPTH = 20
FINAL_TOP_K = 5

# ---------- Reranker ----------
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
RERANKER_CANDIDATE_DEPTH = 20
RERANKER_BATCH_SIZE = 8
RERANKER_MAX_LENGTH = 512

# ---------- Parent-Child ----------
# 검색 순위는 Child 청크 기준으로 유지하고, Reranker Top-K 확정 뒤
# parent_doc_id가 같은 형제 청크를 Evidence Context로 확장합니다.
PARENT_CHILD_ENABLED = True
# None이면 전체 Parent를 사용합니다. 숫자를 넣으면 문자 수 기준으로
# matched child 우선 + 가까운 sibling 순서로 선택합니다.
PARENT_CONTEXT_MAX_CHARS: int | None = None

# ---------- 버전 / Elasticsearch ----------
DENSE_INPUT_VERSION = "kdic-dense-structured-v2-title-section-content-newline"
DENSE_CACHE_VERSION = "kdic-hcx-dense-structured-v2-cache-v1"
ES_EXPECTED_VERSION = "8.15.3"
ES_URL = "http://127.0.0.1:9220"
ES_ANALYZER_NAME = "kdic_nori_none"
ES_INDEX_SCHEMA_VERSION = "kdic-hybrid-bm25-dense-v3"
FORCE_REBUILD_BM25_INDEX = False

assert math.isclose(DENSE_WEIGHT + BM25_WEIGHT, 1.0)
assert QUERY_FUSION_RRF_K > 0 and CANDIDATE_DEPTH > 0 and FINAL_TOP_K > 0
assert RERANKER_CANDIDATE_DEPTH == CANDIDATE_DEPTH
assert RERANKER_CANDIDATE_DEPTH >= FINAL_TOP_K

print({
    "answer_method": "D_ANSWER_SKELETON",
    "dense": "HCX bge-m3 Dense-structured-v2",
    "sparse": "Elasticsearch BM25 + Nori-none",
    "weights": [DENSE_WEIGHT, BM25_WEIGHT],
    "fusion": "MINMAX",
    "query_fusion_rrf_k": QUERY_FUSION_RRF_K,
    "candidate_depth": CANDIDATE_DEPTH,
    "final_top_k": FINAL_TOP_K,
    "reranker": RERANKER_MODEL_NAME,
    "parent_child": PARENT_CHILD_ENABLED,
    "parent_context_max_chars": PARENT_CONTEXT_MAX_CHARS,
    "evidence_pack": True,
    "answer_skeleton": True,
    "fact_index": False,
    "fact_sheet": False,
})


# ---------- V1.5 질의분석 ----------
HCX_DECOMPOSITION_MODEL = "HCX-007"
V15_ORIGINAL_WEIGHT = 0.40
V15_SUBQUERY_TOTAL_WEIGHT = 0.60
V15_MIN_CONFIDENCE = 0.80
V15_MAX_SUBQUERIES = 4
V15_CACHE_PATH = Path("/content/kdic_v15_chat_decomposition_cache.jsonl")

assert math.isclose(V15_ORIGINAL_WEIGHT + V15_SUBQUERY_TOTAL_WEIGHT, 1.0)


{'answer_method': 'D_ANSWER_SKELETON', 'dense': 'HCX bge-m3 Dense-structured-v2', 'sparse': 'Elasticsearch BM25 + Nori-none', 'weights': [0.7, 0.3], 'fusion': 'MINMAX', 'query_fusion_rrf_k': 10, 'candidate_depth': 20, 'final_top_k': 5, 'reranker': 'BAAI/bge-reranker-v2-m3', 'parent_child': True, 'parent_context_max_chars': None, 'evidence_pack': True, 'answer_skeleton': True, 'fact_index': False, 'fact_sheet': False}


## 4. KDIC ZIP 업로드와 청크 로딩

필수 파일은 ZIP 내부의 `processed/chunks.jsonl`입니다. 기존 `chunk_embeddings_hcx.jsonl`은 `content` 중심 임베딩이므로 Dense-structured-v2 캐시로 사용하지 않습니다.


In [5]:
def _read_jsonl(path: Path) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"JSONL 파싱 실패: {path}, line={line_number}") from error
            if not isinstance(record, dict):
                raise TypeError(f"JSONL 레코드가 객체가 아닙니다: {path}, line={line_number}")
            records.append(record)
    return records


def _safe_extract_zip(zip_path: Path, destination: Path) -> Path:
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path, "r") as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f"손상된 ZIP 항목: {bad_member}")
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f"안전하지 않은 ZIP 경로: {member.filename}")
        archive.extractall(destination)
    return destination


def _find_unique_file(root: Path, filename: str) -> Path:
    matches = list(root.rglob(filename))
    processed = [path for path in matches if path.parent.name == "processed"]
    candidates = processed or matches
    if not candidates:
        raise FileNotFoundError(f"{filename}을 찾지 못했습니다: {root}")
    if len(candidates) != 1:
        raise RuntimeError(f"{filename} 후보가 여러 개입니다: {candidates}")
    return candidates[0]


def resolve_data_source(configured_path: str | None) -> Path:
    if configured_path:
        path = Path(configured_path)
        if path.exists():
            return path
        raise FileNotFoundError(f"DATA_SOURCE 경로가 없습니다: {path}")

    try:
        from google.colab import files
    except ImportError as error:
        raise FileNotFoundError(
            "DATA_SOURCE에 KDIC_output ZIP 또는 압축 해제 폴더 경로를 지정하세요."
        ) from error

    existing_zips = sorted(Path("/content").glob("*.zip"))
    if not FORCE_REUPLOAD_ZIP and len(existing_zips) == 1:
        print(f"이미 업로드된 ZIP을 재사용합니다: {existing_zips[0]}")
        print("다른 파일을 새로 올리려면 설정 셀에서 FORCE_REUPLOAD_ZIP = True로 바꾸고 다시 실행하세요.")
        return existing_zips[0]

    print("KDIC_output ZIP 파일을 업로드하세요.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise RuntimeError(f"ZIP 파일을 정확히 1개 업로드해야 합니다: {list(uploaded)}")
    return Path("/content") / zip_names[0]


def prepare_data_root(source: Path) -> Path:
    if source.is_dir():
        return source
    if not zipfile.is_zipfile(source):
        raise ValueError(f"ZIP 파일이 아닙니다: {source}")
    digest = hashlib.sha256(source.read_bytes()).hexdigest()[:16]
    destination = Path("/content/kdic_data_a") / digest
    marker = destination / ".ready"
    if marker.exists():
        return destination
    if destination.exists():
        shutil.rmtree(destination)
    _safe_extract_zip(source, destination)
    marker.write_text("ready", encoding="utf-8")
    return destination


def _clean_text(value: Any) -> str:
    text = str(value or "").replace("\x00", "")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def load_chunks(data_root: Path) -> list[dict[str, Any]]:
    chunks_path = _find_unique_file(data_root, "chunks.jsonl")
    chunks = _read_jsonl(chunks_path)
    if not chunks:
        raise RuntimeError("chunks.jsonl이 비어 있습니다.")

    chunk_ids = [str(chunk.get("chunk_id") or "").strip() for chunk in chunks]
    if any(not chunk_id for chunk_id in chunk_ids):
        raise RuntimeError("빈 chunk_id가 있습니다.")
    if len(chunk_ids) != len(set(chunk_ids)):
        raise RuntimeError("중복 chunk_id가 있습니다.")
    if any(not _clean_text(chunk.get("content")) for chunk in chunks):
        raise RuntimeError("본문이 비어 있는 청크가 있습니다.")
    return chunks


def build_dense_structured_v2_text(chunk: dict[str, Any]) -> str:
    parts = [
        _clean_text(chunk.get("title")),
        _clean_text(chunk.get("section_title")),
        _clean_text(chunk.get("content")),
    ]
    text = "\n".join(part for part in parts if part)
    if not text:
        raise ValueError(f"Dense 입력이 비었습니다: {chunk.get('chunk_id')}")
    return text


DATA_PATH = resolve_data_source(DATA_SOURCE)
DATA_ROOT = prepare_data_root(DATA_PATH)
CHUNKS = load_chunks(DATA_ROOT)
CHUNKS_BY_ID = {str(chunk["chunk_id"]): chunk for chunk in CHUNKS}

# Parent-Child 인덱스: parent_doc_id가 없으면 document_id, 그것도 없으면 chunk_id를 사용합니다.
PARENT_CHILDREN_BY_ID: dict[str, list[dict[str, Any]]] = {}
CHUNK_PARENT_ID: dict[str, str] = {}
for chunk in CHUNKS:
    chunk_id = str(chunk["chunk_id"])
    parent_id = (
        _clean_text(chunk.get("parent_doc_id"))
        or _clean_text(chunk.get("document_id"))
        or chunk_id
    )
    CHUNK_PARENT_ID[chunk_id] = parent_id
    PARENT_CHILDREN_BY_ID.setdefault(parent_id, []).append(chunk)

for parent_id, children in PARENT_CHILDREN_BY_ID.items():
    children.sort(key=lambda row: (
        int(row.get("chunk_index") or 0),
        str(row.get("chunk_id") or ""),
    ))

dataset_hash = hashlib.sha256()
for chunk in CHUNKS:
    dataset_hash.update(str(chunk["chunk_id"]).encode("utf-8"))
    dataset_hash.update(b"\0")
    dataset_hash.update(build_dense_structured_v2_text(chunk).encode("utf-8"))
    dataset_hash.update(b"\0")
DATASET_FINGERPRINT = dataset_hash.hexdigest()
ES_INDEX_NAME = f"kdic-bm25-nori-none-a-v2-{DATASET_FINGERPRINT[:12]}"

print("데이터 경로:", DATA_ROOT)
print("청크 수:", len(CHUNKS))
print("Parent 문서 수:", len(PARENT_CHILDREN_BY_ID))
print("데이터 지문:", DATASET_FINGERPRINT[:16])
print("업무:", sorted({_clean_text(chunk.get("business_function")) for chunk in CHUNKS}))


KDIC_output ZIP 파일을 업로드하세요.


KeyboardInterrupt: 

## 6. HCX 클라이언트와 Dense-structured-v2 임베딩 캐시


In [ ]:
def load_hcx_api_key() -> str:
    key: str | None = None
    try:
        from google.colab import userdata
        key = userdata.get("HCX_API_KEY")
    except Exception:
        key = os.environ.get("HCX_API_KEY")
    if not key:
        key = getpass.getpass("HCX_API_KEY: ")

    key = str(key or "").strip()
    if not key:
        raise ValueError("HCX_API_KEY가 비어 있습니다.")
    if key.lower().startswith("bearer "):
        raise ValueError("HCX_API_KEY 앞에 'Bearer '를 붙이지 마세요.")
    if any(character.isspace() for character in key):
        raise ValueError("HCX_API_KEY 안에 공백 또는 줄바꿈이 있습니다.")
    return key


HCX_API_KEY = load_hcx_api_key()
HCX_CLIENT = OpenAI(
    api_key=HCX_API_KEY,
    base_url=HCX_BASE_URL,
    timeout=HCX_REQUEST_TIMEOUT,
    max_retries=HCX_MAX_RETRIES,
)


def embed_hcx_single(text: str) -> np.ndarray:
    cleaned = _clean_text(text)
    if not cleaned:
        raise ValueError("임베딩 입력이 비어 있습니다.")
    response = HCX_CLIENT.embeddings.create(
        model=HCX_EMBEDDING_MODEL,
        input=cleaned,
        encoding_format=HCX_ENCODING_FORMAT,
    )
    if len(response.data) != 1:
        raise RuntimeError(f"단일 임베딩 응답 개수가 1이 아닙니다: {len(response.data)}")
    vector = np.asarray(response.data[0].embedding, dtype=np.float32)
    if vector.ndim != 1 or vector.size == 0:
        raise RuntimeError(f"잘못된 임베딩 shape: {vector.shape}")
    if not np.all(np.isfinite(vector)):
        raise RuntimeError("임베딩에 NaN 또는 무한대가 있습니다.")
    return vector


print("HCX 클라이언트 준비 완료")
print("Dense 입력 예시:\n", build_dense_structured_v2_text(CHUNKS[0])[:500])


In [ ]:
def structured_input_sha256(chunk: dict[str, Any]) -> str:
    text = build_dense_structured_v2_text(chunk)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def _load_valid_dense_cache(path: Path) -> dict[str, dict[str, Any]]:
    if not path.exists():
        return {}
    valid: dict[str, dict[str, Any]] = {}
    for record in _read_jsonl(path):
        chunk_id = str(record.get("chunk_id") or "")
        chunk = CHUNKS_BY_ID.get(chunk_id)
        if chunk is None:
            continue
        if record.get("model") != HCX_EMBEDDING_MODEL:
            continue
        if record.get("input_version") != DENSE_INPUT_VERSION:
            continue
        if record.get("cache_version") != DENSE_CACHE_VERSION:
            continue
        if record.get("input_sha256") != structured_input_sha256(chunk):
            continue
        vector = np.asarray(record.get("embedding"), dtype=np.float32)
        if vector.ndim != 1 or vector.size == 0 or not np.all(np.isfinite(vector)):
            continue
        if int(record.get("dimensions") or 0) != vector.size:
            continue
        valid[chunk_id] = record
    return valid


def _write_dense_cache_atomic(path: Path, records_by_id: dict[str, dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    with temp_path.open("w", encoding="utf-8") as file:
        for chunk in CHUNKS:
            record = records_by_id.get(str(chunk["chunk_id"]))
            if record is not None:
                file.write(json.dumps(record, ensure_ascii=False) + "\n")
    os.replace(temp_path, path)


def prepare_dense_embeddings(
    cache_path: Path = DENSE_CACHE_PATH,
    checkpoint_every: int = 5,
    allow_generate_missing: bool = False,
) -> tuple[np.ndarray, list[str]]:
    cache = _load_valid_dense_cache(cache_path)
    missing = [chunk for chunk in CHUNKS if str(chunk["chunk_id"]) not in cache]
    print(f"Dense cache: valid={len(cache)}, missing={len(missing)}")

    if missing and not allow_generate_missing:
        raise RuntimeError(
            "Dense 캐시에 유효한 문서 임베딩이 부족하므로 자동 생성을 중단했습니다. "
            f"valid={len(cache)}, missing={len(missing)}. "
            "기존 캐시 파일을 올바르게 업로드하거나, 최초 생성일 때만 "
            "CREATE_DENSE_CACHE_ONCE=True로 바꾼 뒤 이 셀을 다시 실행하세요."
        )

    try:
        for index, chunk in enumerate(
            tqdm(missing, desc="Dense-structured-v2 embedding"),
            start=1,
        ):
            chunk_id = str(chunk["chunk_id"])
            input_text = build_dense_structured_v2_text(chunk)
            vector = embed_hcx_single(input_text)
            cache[chunk_id] = {
                "chunk_id": chunk_id,
                "model": HCX_EMBEDDING_MODEL,
                "encoding_format": HCX_ENCODING_FORMAT,
                "input_version": DENSE_INPUT_VERSION,
                "input_sha256": hashlib.sha256(input_text.encode("utf-8")).hexdigest(),
                "cache_version": DENSE_CACHE_VERSION,
                "dimensions": int(vector.size),
                "embedding": vector.tolist(),
                "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            }
            if index % checkpoint_every == 0:
                _write_dense_cache_atomic(cache_path, cache)
    finally:
        if cache:
            _write_dense_cache_atomic(cache_path, cache)

    ordered_vectors: list[np.ndarray] = []
    dimensions: set[int] = set()
    chunk_ids: list[str] = []
    for chunk in CHUNKS:
        chunk_id = str(chunk["chunk_id"])
        record = cache.get(chunk_id)
        if record is None:
            raise RuntimeError(f"Dense 캐시 누락: {chunk_id}")
        vector = np.asarray(record["embedding"], dtype=np.float32)
        norm = float(np.linalg.norm(vector))
        if norm == 0.0:
            raise RuntimeError(f"영벡터 임베딩: {chunk_id}")
        ordered_vectors.append(vector / norm)
        dimensions.add(int(vector.size))
        chunk_ids.append(chunk_id)

    if len(dimensions) != 1:
        raise RuntimeError(f"임베딩 차원 불일치: {dimensions}")
    return np.vstack(ordered_vectors), chunk_ids


def download_dense_cache_to_browser(
    cache_path: Path = DENSE_CACHE_PATH,
) -> None:
    if not cache_path.is_file() or cache_path.stat().st_size == 0:
        raise FileNotFoundError(f"다운로드할 Dense 캐시가 없습니다: {cache_path}")
    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError(f"Colab 외부에서는 이 파일을 직접 가져가세요: {cache_path}") from error
    print(f"Dense 캐시 다운로드를 시작합니다: {cache_path.name}")
    files.download(str(cache_path))


# 이 노트북은 임베딩 캐시를 업로드하지 않고 항상 새로 계산합니다.
CREATE_DENSE_CACHE_ONCE = True

DENSE_MATRIX, DENSE_CHUNK_IDS = prepare_dense_embeddings(
    allow_generate_missing=CREATE_DENSE_CACHE_ONCE,
)
DENSE_DIMENSION = int(DENSE_MATRIX.shape[1])
DENSE_VECTOR_BY_ID = dict(zip(DENSE_CHUNK_IDS, DENSE_MATRIX))

print("Dense matrix:", DENSE_MATRIX.shape)
print("Dense cache:", DENSE_CACHE_PATH)
download_dense_cache_to_browser()
print("Dense 캐시 다운로드 완료:", DENSE_CACHE_PATH.name)


## 5. Elasticsearch 연결 검증과 BM25 + Nori-none 인덱스

인덱스 이름에는 데이터 지문이 포함됩니다. 같은 데이터를 다시 실행하면 기존 정상 인덱스를 재사용하고, 문서 수·스키마·데이터 지문이 다르면 해당 A안 인덱스만 다시 만듭니다.


In [ ]:
def connect_elasticsearch() -> Elasticsearch:
    client = Elasticsearch(
        ES_URL,
        request_timeout=120,
        max_retries=5,
        retry_on_timeout=True,
    )
    try:
        info = client.info()
    except Exception as error:
        raise RuntimeError(
            "Elasticsearch 연결 실패입니다. 2번 준비 셀의 마지막 로그를 확인하세요. "
            f"원인={type(error).__name__}: {error}"
        ) from error

    running_version = str(info["version"]["number"])
    if running_version != ES_EXPECTED_VERSION:
        raise RuntimeError(
            f"Elasticsearch 버전 불일치: running={running_version}, expected={ES_EXPECTED_VERSION}"
        )

    nodes = client.nodes.info(metric="plugins")
    plugin_names = {
        str(plugin.get("name") or "")
        for node in nodes["nodes"].values()
        for plugin in node.get("plugins", [])
    }
    if "analysis-nori" not in plugin_names:
        raise RuntimeError(f"analysis-nori 플러그인이 없습니다: {sorted(plugin_names)}")
    return client


def _index_is_reusable(client: Elasticsearch) -> bool:
    if FORCE_REBUILD_BM25_INDEX:
        return False
    if not client.indices.exists(index=ES_INDEX_NAME):
        return False
    count = int(client.count(index=ES_INDEX_NAME)["count"])
    mapping = client.indices.get_mapping(index=ES_INDEX_NAME)
    metadata = mapping[ES_INDEX_NAME]["mappings"].get("_meta", {})
    return (
        count == len(CHUNKS)
        and metadata.get("schema_version") == ES_INDEX_SCHEMA_VERSION
        and metadata.get("dataset_fingerprint") == DATASET_FINGERPRINT
    )


def prepare_bm25_nori_none_index(client: Elasticsearch) -> None:
    if _index_is_reusable(client):
        print(f"기존 BM25 인덱스 재사용: {ES_INDEX_NAME}")
        return

    if client.indices.exists(index=ES_INDEX_NAME):
        client.indices.delete(index=ES_INDEX_NAME)

    client.indices.create(
        index=ES_INDEX_NAME,
        settings={
            "number_of_shards": 1,
            "number_of_replicas": 0,
            "similarity": {
                "kdic_bm25": {
                    "type": "BM25",
                    "k1": 1.2,
                    "b": 0.75,
                }
            },
            "analysis": {
                "tokenizer": {
                    "kdic_nori_none_tokenizer": {
                        "type": "nori_tokenizer",
                        "decompound_mode": "none",
                    }
                },
                "analyzer": {
                    ES_ANALYZER_NAME: {
                        "type": "custom",
                        "tokenizer": "kdic_nori_none_tokenizer",
                    }
                },
            },
        },
        mappings={
            "_meta": {
                "schema_version": ES_INDEX_SCHEMA_VERSION,
                "dataset_fingerprint": DATASET_FINGERPRINT,
            },
            "properties": {
                "chunk_id": {"type": "keyword"},
                "search_text": {
                    "type": "text",
                    "analyzer": ES_ANALYZER_NAME,
                    "search_analyzer": ES_ANALYZER_NAME,
                    "similarity": "kdic_bm25",
                },
                "embedding": {                      # ← 추가
                    "type": "dense_vector",     
                    "dims": DENSE_DIMENSION,
                    "index": True,
                    "similarity": "dot_product",    
                },
            },
        },
    )

    actions = (
        {
            "_op_type": "index",
            "_index": ES_INDEX_NAME,
            "_id": str(chunk["chunk_id"]),
            "_source": {
                "chunk_id": str(chunk["chunk_id"]),
                "search_text": build_dense_structured_v2_text(chunk),
                "embedding": DENSE_VECTOR_BY_ID[str(chunk["chunk_id"])].tolist(),
            },
        }
        for chunk in CHUNKS
    )
    bulk_client = client.options(request_timeout=120)
    success, errors = helpers.bulk(
        bulk_client,
        actions,
        chunk_size=100,
        max_retries=4,
        initial_backoff=1,
        max_backoff=8,
        raise_on_error=False,
        raise_on_exception=False,
    )
    client.indices.refresh(index=ES_INDEX_NAME)

    if errors:
        preview = json.dumps(errors[:3], ensure_ascii=False, default=str)[:3000]
        raise RuntimeError(f"BM25 인덱싱 실패 {len(errors)}건: {preview}")
    if int(success) != len(CHUNKS):
        raise RuntimeError(f"BM25 인덱싱 건수 불일치: success={success}, chunks={len(CHUNKS)}")

    actual_count = int(client.count(index=ES_INDEX_NAME)["count"])
    if actual_count != len(CHUNKS):
        raise RuntimeError(f"BM25 저장 건수 불일치: index={actual_count}, chunks={len(CHUNKS)}")

    analysis = client.indices.analyze(
        index=ES_INDEX_NAME,
        analyzer=ES_ANALYZER_NAME,
        text="예금자보호제도",
    )
    if not analysis.get("tokens"):
        raise RuntimeError("Nori 분석 결과가 비어 있습니다.")


ES = connect_elasticsearch()
prepare_bm25_nori_none_index(ES)

print("Elasticsearch:", ES.info()["version"]["number"])
print("BM25 인덱스:", ES_INDEX_NAME)
print("BM25 문서 수:", ES.count(index=ES_INDEX_NAME)["count"])
print("Nori-none 토큰:", [
    token["token"]
    for token in ES.indices.analyze(
        index=ES_INDEX_NAME,
        analyzer=ES_ANALYZER_NAME,
        text="예금자보호제도",
    )["tokens"]
])


## 2. V1.5 질의분석 모듈

In [ ]:
%%writefile kdic_integrated_eval_core.py
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any


@dataclass
class QueryPlan:
    need_id: str
    variant_id: str
    dense_query: str
    bm25_query: str
    filter_mode: str = "NONE"
    business_filters: list[str] = field(default_factory=list)
    soft_business_hints: list[str] = field(default_factory=list)
    query_weight: float = 1.0
    query_source: str = "ORIGINAL"


@dataclass
class AnalyzerCase:
    evaluation_id: str
    analyzer: str
    original_question: str
    route: str
    analysis_latency_ms: float
    plans: list[QueryPlan]
    raw_result: dict[str, Any]


def normalize_route(value: Any) -> str:
    text = str(value or "").strip().upper()
    mapping = {
        "SIMPLE_RETRIEVE": "RETRIEVE",
        "MULTI_RETRIEVE": "RETRIEVE",
        "OUT_OF_SCOPE": "OUT_OF_SCOPE",
        "OOS": "OUT_OF_SCOPE",
        "DIRECT": "DIRECT_RESPONSE",
        "DIRECT_RESPONSE": "DIRECT_RESPONSE",
        "CLARIFY": "CLARIFY",
        "RETRIEVE": "RETRIEVE",
    }
    return mapping.get(text, text)

In [ ]:
%%writefile kdic_lightweight_router_v1.py
from __future__ import annotations

"""KDIC 간편 라우터 V1.

설계 목표
---------
1. 명백한 DIRECT/OUT_OF_SCOPE/CLARIFY만 규칙으로 차단하고 나머지는 검색으로 보낸다.
2. 단일질의의 검색 문자열은 사용자 원문을 그대로 보존한다.
3. 실제로 독립 검색이 필요한 복합질의만 보수적으로 분해한다.
4. 업무 필터는 SOFT/NONE만 사용해 잘못된 HARD 필터를 구조적으로 막는다.
5. 외부 API나 모델을 호출하지 않아 라우팅 지연을 최소화한다.
"""

import json
import re
import time
import unicodedata
from dataclasses import dataclass
from typing import Any, Iterable, Mapping, Sequence


PIPELINE_VERSION = "KDIC_LIGHTWEIGHT_ROUTER_V1_2026_08_13"

BUSINESS_FUNCTIONS = (
    "예금자보호제도",
    "예금보험금 안내",
    "고객 미수령금 신청",
    "착오송금 반환 신청",
    "채무조정 안내",
    "은닉재산 신고",
)

INTENTS = (
    "AMOUNT",
    "ELIGIBILITY",
    "TIME",
    "APPLICATION",
    "OVERVIEW",
    "STATUS",
    "DOCUMENTS",
    "CONTACT",
)

BUSINESS_KEYWORDS: dict[str, tuple[str, ...]] = {
    "예금자보호제도": (
        "예금자보호", "보호한도", "보호 한도", "보호대상", "예금 보호",
        "예금은 얼마까지 보호", "금융상품이 보호", "금융회사가 보호 대상",
    ),
    "예금보험금 안내": (
        "예금보험금", "보험금 지급", "보험사고", "가지급금", "개산지급금",
        "1종 보험사고", "2종 보험사고",
    ),
    "고객 미수령금 신청": (
        "고객 미수령금", "미수령금", "파산배당금", "개산지급금 정산금",
        "지급대행점", "상속인 금융거래 조회", "상속인 금융거래 조회서비스",
    ),
    "착오송금 반환 신청": (
        "착오송금", "착오 송금", "잘못 보낸 돈", "잘못 송금", "반환지원",
        "매입계약", "지급명령", "강제집행", "송금인", "수취인",
        "계좌번호를 잘못", "엉뚱한 사람에게 보낸 돈",
    ),
    "채무조정 안내": (
        "채무조정", "신용회복지원", "파산선고", "채무감면", "개인회생",
        "개인파산", "워크아웃", "변제기간", "부채증명원", "채무정보", "면책",
    ),
    "은닉재산 신고": (
        "은닉재산", "은닉 재산", "금융부실관련자", "부실관련자", "차명재산",
        "차명 재산", "신고 포상금",
    ),
}

STRONG_BUSINESS_KEYWORDS: dict[str, tuple[str, ...]] = {
    "예금자보호제도": ("예금자보호", "보호한도", "보호 한도", "예금 보호"),
    "예금보험금 안내": ("예금보험금", "보험금 지급", "1종 보험사고", "2종 보험사고"),
    "고객 미수령금 신청": (
        "고객 미수령금", "미수령금", "파산배당금", "지급대행점", "상속인 금융거래 조회",
    ),
    "착오송금 반환 신청": (
        "착오송금", "착오 송금", "반환지원", "잘못 보낸 돈", "지급명령", "강제집행",
    ),
    "채무조정 안내": (
        "채무조정", "신용회복지원", "개인회생", "개인파산", "워크아웃", "부채증명원",
    ),
    "은닉재산 신고": (
        "은닉재산", "은닉 재산", "금융부실관련자", "차명재산", "차명 재산", "신고 포상금",
    ),
}

TYPO_MAP = (
    ("예금보헝금", "예금보험금"),
    ("예금보혐금", "예금보험금"),
    ("착오송금반한", "착오송금 반환"),
    ("반한지원", "반환지원"),
    ("반환지웜", "반환지원"),
    ("미수령금신정", "미수령금 신청"),
    ("통합신정", "통합신청"),
    ("검새", "검색"),
    ("발샐", "발생"),
    ("관게", "관계"),
    ("요정", "요청"),
    ("언재", "언제"),
    ("제외돼는", "제외되는"),
)

DIRECT_META_PATTERN = re.compile(
    r"^(?:안녕|안녕하세요|반갑습니다|반가워요|고마워요|고맙습니다|감사합니다|도움이 됐어요|"
    r"알겠습니다|무슨 질문을 할 수 있나요|지원하는 업무를 (?:알려주세요|목록으로 보여주세요)|"
    r"이 챗봇은 어떻게 사용하면 되나요|답변을 쉽게 설명해 줄 수 있나요|"
    r"전문가 수준으로 자세히 설명해 주세요|긴 설명보다 핵심 내용만 먼저 알려주세요|"
    r"질문을 잘못 입력했어요[.]? 다시 물어볼게요)[.!?]*$",
    re.I,
)

REFORMAT_PATTERN = re.compile(
    r"^(?:쉽게 설명해 주세요|핵심만 알려주세요|표로 정리해 주세요|더 자세히 설명해 주세요)[.!?]*$",
    re.I,
)

OOS_PATTERN = re.compile(
    r"(?:코스피|비트코인|주택담보대출\s*금리|대출금리|신용점수|실손보험|국민연금|"
    r"보이스피싱.*(?:경찰|신고)|은행\s*계좌를\s*새로|계좌\s*개설|해외송금\s*수수료|"
    r"카드\s*결제.*환불|전세대출|세금\s*환급|퇴직금|개인정보\s*유출|상속세|"
    r"환율|환전소|주식\s*(?:투자|포트폴리오)|신용카드\s*연회비|사업자등록|"
    r"(?:서울|오늘|내일|이번\s*주말)?.{0,8}(?:날씨|기온|미세먼지))",
    re.I,
)

GENERIC_BUSINESS_CLARIFY_PATTERN = re.compile(
    r"^(?:신청\s*(?:방법|자격|기한|과정|대상)|접수\s*(?:방법|절차)|제출해야\s*하는\s*서류|"
    r"필요한\s*서류|조회는\s*어디에서|온라인으로\s*신청|방문해서\s*접수|접수\s*후\s*처리\s*기간|"
    r"신청\s*자격과\s*제외\s*조건|신청\s*과정에서\s*수수료나\s*비용|"
    r"처리\s*결과는\s*어디에서\s*확인|이미\s*접수한\s*신청을\s*취소|"
    r"문의하거나\s*접수하려면\s*어느\s*기관|제가\s*신청\s*대상에\s*해당|"
    r"본인\s*대신\s*대리인이\s*신청|상속인이\s*신청하거나\s*받을\s*수|"
    r"처리\s*기간|문의처|신청\s*비용).*$",
    re.I,
)

TARGET_DEMONSTRATIVE_PATTERN = re.compile(
    r"(?:^|[\s,.(])(?:제가\s*(?:말한|가입한|가진|본)\s*)?"
    r"이\s*(?:금융상품|계좌|상품|돈|송금|거래|금액|채무|재산)"
    r"(?:을|를|이|가|도|은|는|의|이나|과|와|\s|[,.!?]|$)",
    re.I,
)

TARGET_REFERENCE_PHRASES = (
    "제가 가입한 상품", "어떤 송금 건", "어떤 예금에 대해", "어떤 돈을 신청",
    "어떤 예금이나 금융상품",
)

APPLICANT_REFERENCE_PHRASES = (
    "제 신청 유형", "제 신청 자격", "제 경우", "누구를 신청인", "누가 방문",
    "신고 주체 유형", "신청인란",
)

CASE_REFERENCE_PHRASES = (
    "제 상황", "현재 상황", "제 채무 상태", "제 신고 상황", "반려", "거절",
    "보완 요청", "진행되지 않", "여러 금융회사에 예금", "여러 계좌에 나뉘",
)

HIGH_PRECISION_INTENT_RULES: tuple[tuple[str, tuple[str, ...]], ...] = (
    ("DOCUMENTS", (
        r"필요한?\s*(?:서류|증빙)", r"제출(?:해야\s*하는|할)?\s*서류", r"구비\s*서류",
        r"신분증", r"위임장", r"준비(?:해야\s*할|할)?\s*서류", r"무엇을\s*(?:더\s*)?준비",
    )),
    ("STATUS", (
        r"어디(?:서|에서)\s*(?:확인|조회|검색)", r"조회\s*(?:방법|결과)", r"처리\s*결과",
        r"진행\s*(?:상황|상태)", r"지급\s*정보.*보는\s*방법", r"있는지.*조회",
    )),
    ("APPLICATION", (
        r"신청\s*(?:방법|절차)", r"접수\s*(?:방법|절차)", r"제출\s*방법", r"신고\s*채널",
        r"어떻게\s*(?:신청|접수|청구)", r"(?:온라인|방문|직접).*신청.*(?:가능|할\s*수)",
        r"취소.*방법", r"철회.*방법", r"무엇을\s*해야", r"어디에\s*접수",
    )),
    ("TIME", (
        r"언제(?:부터|까지)", r"신청.*(?:기한|기간|시점)", r"처리\s*기간", r"소요\s*(?:기간|시간)",
        r"얼마나\s*걸", r"언제\s*(?:지급|찾)",
    )),
    ("AMOUNT", (
        r"보호\s*한도", r"지급\s*금액", r"금액\s*계산", r"금액.*얼마(?:여야|이어야)",
        r"계산\s*(?:기준|방법)", r"수수료\s*(?:금액|비용)", r"얼마나\s*(?:감면|지급|보상|돌려|받|보호)",
        r"비용\s*차감", r"최종\s*보호금액",
    )),
    ("CONTACT", (
        r"연락처", r"전화번호", r"문의처", r"어디로\s*연락", r"어느\s*기관.*문의",
    )),
    ("ELIGIBILITY", (
        r"신청\s*(?:대상|자격|요건)", r"가능한\s*대상", r"제외되는?\s*경우", r"받을\s*수\s*있",
        r"신청할\s*수\s*있", r"포함되", r"어떤\s*경우.*(?:지급|지원|보호)",
        r"(?:대상|자격)에\s*해당", r"(?:예금|계좌|금융상품|상품|원금|이자|채권).{0,30}보호(?:가)?\s*되",
        r"지원\s*대상", r"누가\s*(?:신청|수령)", r"보호\s*대상",
    )),
    ("OVERVIEW", (
        r"무엇(?:인가요|인지|이며)", r"뭐예요", r"의미", r"정의", r"차이", r"종류", r"개요",
        r"설명", r"관계", r"왜\s*(?:발생|제외)", r"어떤\s*성격",
    )),
)

WEAK_INTENT_RULES: tuple[tuple[str, tuple[str, ...]], ...] = (
    ("DOCUMENTS", (r"서류", r"증빙", r"준비")),
    ("STATUS", (r"조회", r"확인", r"검색")),
    ("APPLICATION", (r"신청", r"접수", r"절차", r"제출", r"청구", r"신고")),
    ("TIME", (r"기간", r"기한", r"시점", r"언제")),
    ("AMOUNT", (r"한도", r"금액", r"계산", r"비용", r"포상금")),
    ("CONTACT", (r"연락", r"문의", r"전화")),
    ("ELIGIBILITY", (r"대상", r"자격", r"요건", r"조건", r"가능", r"보호")),
    ("OVERVIEW", (r"설명", r"관계", r"방식", r"종류", r"의미")),
)

# 두 정보가 서로 밀접한 하나의 검색 문서에서 함께 해결될 가능성이 높은 결합입니다.
# 이런 결합은 요구가 두 개여도 원문을 유지합니다.
COHESIVE_NO_SPLIT_PATTERNS = (
    re.compile(r"(?:대상|자격|조건).{0,28}(?:서류|준비)|(?:서류|준비).{0,28}(?:제출|신청|접수)\s*방법", re.I),
    re.compile(r"(?:누가|대리인|상속인|본인|법인).{0,35}(?:서류|준비)", re.I),
    re.compile(r"(?:한도|금액).{0,30}(?:포함|계산|합산|상계)|(?:포함|계산|합산|상계).{0,30}(?:한도|금액)", re.I),
    re.compile(r"(?:사유|이유).{0,25}(?:조건|해제)|(?:의미|무엇).{0,25}(?:차이|관계|종류)", re.I),
    re.compile(r"(?:언제까지|기한|기간).{0,25}(?:어디에서|어디에)\s*(?:신청|확인)", re.I),
    re.compile(r"(?:조회|확인)한?\s*(?:뒤|다음).{0,35}(?:신청|지급)", re.I),
    re.compile(r"(?:퇴직연금).{0,45}(?:연금저축).{0,45}(?:보호|한도)", re.I),
)

# 서로 다른 사전 용어가 잡혀도 목적이 관계·차이 또는 결합 가능성 확인이면 원문을 유지합니다.
CROSS_TERM_RELATION_KEEP_PATTERN = re.compile(
    r"(?:와|과|및).{0,38}(?:관계|차이)|(?:관계|차이).{0,38}(?:와|과|및)|"
    r"(?:와|과|및).{0,38}(?:함께|동시에)\s*(?:조회|확인|신청|보호).*(?:가능|할\s*수)",
    re.I,
)

# 독립된 대상·상황·처리 단계가 명시된 경우에만 같은 업무 안에서도 분해합니다.
STRONG_SAME_BUSINESS_SPLIT_PATTERNS = (
    re.compile(r"(?:때|경우)와.{0,55}(?:때|경우)", re.I),
    re.compile(r"(?:외화예금|간편송금|온라인\s*신청).{0,45}(?:후순위채권|해외\s*계좌|방문\s*신청)", re.I),
    re.compile(r"(?:1종\s*보험사고).{0,45}(?:2종\s*보험사고)", re.I),
    re.compile(r"(?:영업정지).{0,35}(?:기존\s*대출|대출\s*거래)", re.I),
    re.compile(r"(?:미리\s*신청|신청\s*전).{0,45}(?:실제\s*보험사고|접수\s*후)", re.I),
    re.compile(r"(?:지급되는\s*조건|지급\s*조건).{0,35}(?:실제\s*)?신청\s*절차", re.I),
    re.compile(r"(?:온라인\s*신청).{0,45}(?:지급대행점\s*)?방문\s*신청", re.I),
    re.compile(r"(?:신청(?:하기)?\s*전|신청\s*전에).{0,45}(?:접수|신청)\s*후", re.I),
    re.compile(r"(?:접수|신청).{0,25}(?:결과|진행\s*절차).{0,25}(?:확인|진행)", re.I),
    re.compile(r"(?:파산\s*금융회사).{0,40}(?:남은\s*)?미수령금.*신청", re.I),
    re.compile(r"(?:기간|얼마나\s*걸).{0,40}(?:비용\s*차감|차감\s*방식)", re.I),
    re.compile(r"(?:금융회사).{0,30}보호\s*대상.{0,30}(?:금융상품).{0,20}보호", re.I),
    re.compile(r"(?:미성년자).{0,35}보호되.{0,35}(?:누가|수령)", re.I),
    re.compile(r"(?:상속인).{0,30}(?:조회).{0,20}(?:뒤|다음).{0,30}(?:지급을\s*)?신청", re.I),
    re.compile(r"(?:제외되는\s*경우).{0,35}(?:제외되는\s*이유|왜\s*제외)", re.I),
)

CLAUSE_BOUNDARY_PATTERN = re.compile(
    r"\s*(?:[.!?;]+|,?\s*(?:그리고|또|혹시|별도로|그와\s*별개로|반면에|반면|뿐만\s*아니라)\s+)\s*",
    re.I,
)

CONJUNCTION_BOUNDARY_PATTERN = re.compile(
    r"\s*(?:,\s*|\s+)(?:그리고|또|혹시|별도로|반면에|반면)\s*",
    re.I,
)

NUMBER_PATTERN = re.compile(r"\d+(?:[.,]\d+)*(?:\s*(?:원|만원|억원|개월|년|일|%))?")
NEGATION_TERMS = ("아니", "못", "제외", "불가", "없", "않", "전혀")


@dataclass(frozen=True)
class RouterConfig:
    max_subqueries: int = 4
    include_original_anchor_for_multi: bool = True
    allow_hard_filter: bool = False
    min_subquery_chars: int = 5


def normalize_query(text: Any) -> dict[str, Any]:
    original = str(text or "")
    value = unicodedata.normalize("NFKC", original)
    changes: list[str] = []
    cleaned = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", " ", value)
    if cleaned != value:
        changes.append("CONTROL_CHARACTER")
        value = cleaned
    cleaned = re.sub(r"([!?ㅋㅎㅠㅜ])\1{2,}", r"\1\1", value)
    if cleaned != value:
        changes.append("REPEATED_CHARACTER")
        value = cleaned
    for wrong, correct in TYPO_MAP:
        if wrong in value:
            value = value.replace(wrong, correct)
            changes.append("EXPLICIT_TYPO")
    cleaned = re.sub(r"\s+", " ", value).strip()
    if cleaned != value:
        changes.append("WHITESPACE")
    if not cleaned:
        raise ValueError("사용자 질의가 비어 있습니다.")
    return {"original_query": original, "normalized_query": cleaned, "changes": list(dict.fromkeys(changes))}


def _ordered_unique(values: Iterable[Any]) -> list[Any]:
    output: list[Any] = []
    seen: set[Any] = set()
    for value in values:
        if value is None or value in seen:
            continue
        seen.add(value)
        output.append(value)
    return output


def _compact(text: str) -> str:
    return re.sub(r"\s+", "", text).lower()


def find_business_matches(text: str) -> list[dict[str, Any]]:
    compact = _compact(text)
    found: list[dict[str, Any]] = []
    for business, keywords in BUSINESS_KEYWORDS.items():
        evidence = [keyword for keyword in keywords if _compact(keyword) in compact]
        if not evidence:
            continue
        strong = [term for term in STRONG_BUSINESS_KEYWORDS[business] if _compact(term) in compact]
        found.append({
            "business_function": business,
            "evidence": _ordered_unique(evidence),
            "strong_evidence": _ordered_unique(strong),
            "confidence": 0.99 if strong else 0.80,
        })
    return found


def find_businesses(text: str) -> list[str]:
    return [row["business_function"] for row in find_business_matches(text)]


def find_intent_matches(text: str) -> list[dict[str, Any]]:
    matches: list[dict[str, Any]] = []
    for source, rules in (("HIGH_PRECISION_RULE", HIGH_PRECISION_INTENT_RULES), ("WEAK_RULE", WEAK_INTENT_RULES)):
        for intent, patterns in rules:
            hit = next((m for pattern in patterns if (m := re.search(pattern, text, flags=re.I))), None)
            if hit and intent not in {row["intent"] for row in matches}:
                matches.append({
                    "intent": intent,
                    "source": source,
                    "evidence": hit.group(0),
                    "start": hit.start(),
                    "end": hit.end(),
                })
    return matches


def _parse_previous_turns(value: Any) -> list[dict[str, str]]:
    if value is None:
        return []
    if isinstance(value, str):
        text = value.strip()
        if not text or text.lower() == "nan":
            return []
        try:
            value = json.loads(text)
        except json.JSONDecodeError:
            return [{"user": text, "assistant": ""}]
    if not isinstance(value, list):
        return []
    output = []
    for row in value[-3:]:
        if isinstance(row, Mapping):
            user = str(row.get("user") or row.get("question") or "").strip()
            assistant = str(row.get("assistant") or row.get("answer") or "").strip()
            if user or assistant:
                output.append({"user": user, "assistant": assistant})
        elif str(row).strip():
            output.append({"user": str(row).strip(), "assistant": ""})
    return output


def build_context(previous_turns: Any = None, conversation_state: Mapping[str, Any] | None = None) -> dict[str, Any]:
    state = dict(conversation_state or {})
    turns = _parse_previous_turns(previous_turns if previous_turns is not None else state.get("recent_turns"))
    confirmed = dict(state.get("confirmed") or {}) if isinstance(state.get("confirmed"), Mapping) else {}
    context_text = " ".join(row["user"] for row in turns if row.get("user"))
    context_businesses = find_businesses(context_text)
    if len(context_businesses) == 1 and not confirmed.get("business_function"):
        confirmed["business_function"] = context_businesses[0]
    return {
        "used": bool(turns or confirmed),
        "recent_turns": turns,
        "confirmed": confirmed,
        "context_businesses": context_businesses,
    }


def detect_route(
    query: str,
    *,
    context: Mapping[str, Any],
) -> tuple[str, list[str], list[str], str | None]:
    """보수적 라우팅. 반환값은 route, reasons, missing, direct_action 순서입니다."""
    businesses = find_businesses(query)
    context_business = str((context.get("confirmed") or {}).get("business_function") or "")
    has_context = bool(context.get("used"))

    if DIRECT_META_PATTERN.fullmatch(query):
        return "DIRECT", ["EXPLICIT_META_OR_SOCIAL"], [], "META_OR_SOCIAL"
    if REFORMAT_PATTERN.fullmatch(query) and has_context:
        return "DIRECT", ["REFORMAT_PREVIOUS_ANSWER"], [], "REFORMAT_PREVIOUS_ANSWER"
    if OOS_PATTERN.search(query) and not businesses:
        return "OUT_OF_SCOPE", ["EXPLICIT_NON_KDIC_TOPIC"], [], None

    if GENERIC_BUSINESS_CLARIFY_PATTERN.fullmatch(query) and not businesses and not context_business:
        return "CLARIFY", ["BUSINESS_NOT_SPECIFIED"], ["business_function"], None

    has_resolved_context = has_context or bool(context_business)
    if any(phrase in query for phrase in APPLICANT_REFERENCE_PHRASES) and not has_resolved_context:
        return "CLARIFY", ["UNRESOLVED_APPLICANT_REFERENCE"], ["applicant_type"], None
    if (any(phrase in query for phrase in TARGET_REFERENCE_PHRASES) or TARGET_DEMONSTRATIVE_PATTERN.search(query)) and not has_resolved_context:
        return "CLARIFY", ["UNRESOLVED_TARGET_REFERENCE"], ["target_type"], None
    if any(phrase in query for phrase in CASE_REFERENCE_PHRASES) and not has_resolved_context:
        return "CLARIFY", ["PERSONAL_CASE_REQUIRES_DETAILS"], ["case_details"], None

    return "RETRIEVE", ["DEFAULT_FAIL_OPEN_RETRIEVAL"], [], None


def _is_cohesive_no_split(query: str) -> bool:
    return any(pattern.search(query) for pattern in COHESIVE_NO_SPLIT_PATTERNS)


def _has_strong_same_business_split_signal(query: str) -> tuple[bool, str | None]:
    for index, pattern in enumerate(STRONG_SAME_BUSINESS_SPLIT_PATTERNS, 1):
        if pattern.search(query):
            return True, f"SAME_BUSINESS_STRONG_PATTERN_{index:02d}"
    return False, None


def detect_complexity(query: str) -> dict[str, Any]:
    businesses = find_businesses(query)
    intents = find_intent_matches(query)
    strong_same, strong_rule = _has_strong_same_business_split_signal(query)
    sentence_clauses = [part.strip(" ,") for part in CLAUSE_BOUNDARY_PATTERN.split(query) if part.strip(" ,")]

    reasons: list[str] = []
    cross_relation_keep = bool(CROSS_TERM_RELATION_KEEP_PATTERN.search(query))
    if len(businesses) >= 2 and not cross_relation_keep:
        reasons.append("MULTIPLE_BUSINESS_FUNCTIONS")
    elif len(businesses) >= 2 and cross_relation_keep:
        reasons.append("CROSS_TERM_RELATION_KEEP_ORIGINAL")
    if strong_same:
        reasons.append(strong_rule or "STRONG_SAME_BUSINESS_SIGNAL")
    if len(sentence_clauses) >= 2:
        clause_businesses = [find_businesses(part) for part in sentence_clauses]
        if sum(bool(values) for values in clause_businesses) >= 2:
            reasons.append("INDEPENDENT_BUSINESS_CLAUSES")
        elif len(intents) >= 2 and not _is_cohesive_no_split(query):
            reasons.append("INDEPENDENT_INTENT_CLAUSES")

    is_multi = (len(businesses) >= 2 and not cross_relation_keep) or strong_same or "INDEPENDENT_INTENT_CLAUSES" in reasons
    if _is_cohesive_no_split(query) and len(businesses) <= 1 and not strong_same:
        is_multi = False
        reasons.append("COHESIVE_SAME_BUSINESS_KEEP_ORIGINAL")
    if not is_multi:
        reasons.append("NO_SAFE_SPLIT_EVIDENCE")

    return {
        "question_type": "MULTI" if is_multi else "SINGLE",
        "businesses": businesses,
        "intents": [row["intent"] for row in intents],
        "clause_count": len(sentence_clauses),
        "reasons": _ordered_unique(reasons),
    }


def _clean_clause(text: str) -> str:
    text = re.sub(r"^(?:그리고|또|혹시|별도로|그럼|그러면)\s*", "", text.strip(), flags=re.I)
    text = re.sub(r"\s+", " ", text).strip(" ,.;")
    if text and not re.search(r"[?요다까]$", text):
        text += " 관련 정보"
    return text


def _sentence_clauses(query: str) -> list[str]:
    return [_clean_clause(part) for part in CLAUSE_BOUNDARY_PATTERN.split(query) if _clean_clause(part)]


def _business_anchor_positions(query: str) -> list[tuple[int, int, str, str]]:
    positions: list[tuple[int, int, str, str]] = []
    compact_query = query.lower()
    for business, keywords in BUSINESS_KEYWORDS.items():
        for keyword in sorted(keywords, key=len, reverse=True):
            start = compact_query.find(keyword.lower())
            if start >= 0:
                positions.append((start, start + len(keyword), business, keyword))
                break
    positions.sort(key=lambda row: row[0])
    return positions


def _split_cross_business(query: str, businesses: Sequence[str]) -> list[str]:
    clauses = _sentence_clauses(query)
    if len(clauses) >= 2:
        enriched: list[str] = []
        for clause in clauses:
            local_businesses = find_businesses(clause)
            if local_businesses:
                enriched.append(clause)
        if len(enriched) >= 2:
            return enriched

    anchors = _business_anchor_positions(query)
    if len(anchors) < 2:
        return []
    output: list[str] = []
    for index, (start, _end, business, keyword) in enumerate(anchors):
        next_start = anchors[index + 1][0] if index + 1 < len(anchors) else len(query)
        previous_end = anchors[index - 1][1] if index > 0 else 0
        raw = query[previous_end:next_start]
        raw = re.sub(r"^(?:와|과|및|하고|,|\s)+", "", raw)
        raw = re.sub(r"(?:와|과|및|하고|,|\s)+$", "", raw)
        clause = _clean_clause(raw)
        if business not in find_businesses(clause):
            clause = f"{business} {clause}".strip()
        # 너무 짧거나 명사구뿐이면 전체 문장의 해당 업무 관련 의도 단서를 붙입니다.
        local_intents = find_intent_matches(clause)
        if not local_intents:
            global_intents = find_intent_matches(query)
            if global_intents:
                clause = f"{clause} {global_intents[min(index, len(global_intents)-1)]['evidence']}"
        output.append(_clean_clause(clause))
    return output


def _split_same_business(query: str, business: str | None) -> list[str]:
    # 처리 전후나 서로 다른 처리 대상을 한 문장에 묶은 대표 구조는 의미 단위로 직접 분리합니다.
    match = re.search(
        r"^(?P<context>.*?영업정지되면)\s*(?P<first>예금은.*?)(?:고|며)\s*(?P<second>기존\s*대출\s*거래.*?)(?:[?]?)$",
        query,
        flags=re.I,
    )
    if match:
        context = match.group("context").strip()
        return [
            _clean_clause(f"{context} {match.group('first')}"),
            _clean_clause(f"{context} {match.group('second')}"),
        ]

    match = re.search(
        r"^(?P<actor>상속인이\s*고인의)\s*(?P<target>미수령금)을?\s*조회한?\s*(?:뒤|다음)\s*"
        r"(?P<action>지급을\s*신청하는\s*방법).*?$",
        query,
        flags=re.I,
    )
    if match:
        prefix = f"{match.group('actor')} {match.group('target')}"
        return [
            _clean_clause(f"{prefix} 조회 방법"),
            _clean_clause(f"{prefix} {match.group('action')}"),
        ]

    clauses = _sentence_clauses(query)
    if len(clauses) >= 2:
        output = []
        for clause in clauses:
            if business and business not in find_businesses(clause):
                clause = f"{business} {clause}"
            output.append(_clean_clause(clause))
        return output

    # 쉼표·연결 표현을 먼저 이용합니다.
    candidates = [part.strip(" ,") for part in re.split(r"\s*(?:,|이고|이며|인지,?|는지와|과|와)\s*", query) if part.strip(" ,")]
    if len(candidates) >= 2:
        candidates = candidates[:3]
        output = []
        for clause in candidates:
            if len(clause) < 4:
                continue
            if business and business not in find_businesses(clause):
                clause = f"{business} {clause}"
            output.append(_clean_clause(clause))
        if len(output) >= 2:
            return output

    # 안전한 절단점을 못 찾으면 분해 실패로 두고 원문 fallback을 사용합니다.
    return []


def validate_decomposition(original: str, subqueries: Sequence[str], expected_businesses: Sequence[str]) -> dict[str, Any]:
    queries = [_clean_clause(str(value)) for value in subqueries if _clean_clause(str(value))]
    issues: list[str] = []
    if len(queries) < 2:
        issues.append("TOO_FEW_SUBQUERIES")
    if len(set(_compact(value) for value in queries)) != len(queries):
        issues.append("DUPLICATE_SUBQUERIES")
    if any(len(value) < 5 for value in queries):
        issues.append("SUBQUERY_TOO_SHORT")

    reconstructed = " ".join(queries)
    missing_businesses = [business for business in expected_businesses if business not in find_businesses(reconstructed)]
    if missing_businesses:
        issues.append("MISSING_BUSINESS_COVERAGE")

    original_numbers = NUMBER_PATTERN.findall(original)
    missing_numbers = [value for value in original_numbers if value not in reconstructed]
    if missing_numbers:
        issues.append("MISSING_NUMERIC_CONSTRAINT")

    original_negations = [term for term in NEGATION_TERMS if term in original]
    missing_negations = [term for term in original_negations if term not in reconstructed]
    if missing_negations:
        issues.append("MISSING_NEGATION")

    status = "COMPLETE" if not issues else ("PARTIAL" if len(queries) >= 2 else "FAILED")
    return {
        "status": status,
        "issues": issues,
        "subqueries": queries,
        "missing_businesses": missing_businesses,
        "missing_numbers": missing_numbers,
        "missing_negations": missing_negations,
    }


def decompose_query(query: str, complexity: Mapping[str, Any], config: RouterConfig) -> dict[str, Any]:
    businesses = list(complexity.get("businesses") or [])
    if complexity.get("question_type") != "MULTI":
        return {"status": "NOT_REQUIRED", "subqueries": [query], "issues": [], "fallback_to_original": False}

    if len(businesses) >= 2:
        candidates = _split_cross_business(query, businesses)
    else:
        candidates = _split_same_business(query, businesses[0] if businesses else None)
    candidates = _ordered_unique(candidates)[: config.max_subqueries]
    validation = validate_decomposition(query, candidates, businesses)
    validation["fallback_to_original"] = validation["status"] != "COMPLETE"
    return validation


def _business_filter_for_query(query: str) -> dict[str, Any]:
    matches = find_business_matches(query)
    if len(matches) == 1:
        match = matches[0]
        return {
            "mode": "SOFT",
            "value": None,
            "soft_hint": match["business_function"],
            "confidence": match["confidence"],
            "evidence": match["evidence"],
            "hard_filter_eligible": False,
            "hard_filter_denial_reasons": ["HARD_DISABLED_BY_ROUTER_POLICY"],
        }
    return {
        "mode": "NONE",
        "value": None,
        "soft_hint": None,
        "confidence": 0.0,
        "evidence": [],
        "hard_filter_eligible": False,
        "hard_filter_denial_reasons": [
            "HARD_DISABLED_BY_ROUTER_POLICY",
            "MULTIPLE_OR_UNKNOWN_BUSINESS_CANDIDATES",
        ],
    }


def _make_need(need_id: str, query: str, *, source: str) -> dict[str, Any]:
    business_matches = find_business_matches(query)
    intent_matches = find_intent_matches(query)
    return {
        "need_id": need_id,
        "query": query,
        "query_source": source,
        "business_function": business_matches[0]["business_function"] if len(business_matches) == 1 else None,
        "business_candidates": business_matches,
        "intents": [row["intent"] for row in intent_matches],
        "intent_evidence": intent_matches,
    }


def build_query_plans(
    original_query: str,
    route: str,
    complexity: Mapping[str, Any],
    decomposition: Mapping[str, Any],
    config: RouterConfig,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    if route != "RETRIEVE":
        return [], []

    is_multi = complexity.get("question_type") == "MULTI"
    decomposition_complete = decomposition.get("status") == "COMPLETE"
    if is_multi and decomposition_complete:
        retrieval_queries = list(decomposition.get("subqueries") or [])
        source = "CONSERVATIVE_DECOMPOSITION"
    else:
        retrieval_queries = [original_query]
        source = "ORIGINAL_PASSTHROUGH" if not is_multi else "ORIGINAL_FALLBACK"

    needs = [_make_need(f"N{index}", query, source=source) for index, query in enumerate(retrieval_queries, 1)]
    plans: list[dict[str, Any]] = []
    for need in needs:
        query = need["query"]
        plans.append({
            "need_id": need["need_id"],
            "retrieval_mode": "STANDARD",
            "semantic_query": query,
            "keyword_query": query,
            "query_source": need["query_source"],
            "business_filter": _business_filter_for_query(query),
            "fallback_policy": {
                "enabled": True,
                "on": ["NO_RESULTS", "LOW_TOP_SCORE", "LOW_COVERAGE"],
                "next_filter_modes": ["NONE"],
                "fail_open": True,
                "original_anchor_query": original_query if is_multi and config.include_original_anchor_for_multi else None,
            },
            "intent_boost": {
                "mode": "SOFT" if need["intents"] else "NONE",
                "values": need["intents"],
                "weight": 0.10 if need["intents"] else 0.0,
            },
        })
    return needs, plans


def query_plan_is_valid(result: Mapping[str, Any]) -> bool:
    route = str((result.get("analysis") or {}).get("route") or "")
    plans = result.get("query_plans") or []
    if route == "RETRIEVE":
        if not plans:
            return False
        for plan in plans:
            if not str(plan.get("semantic_query") or "").strip():
                return False
            if not str(plan.get("keyword_query") or "").strip():
                return False
            if (plan.get("business_filter") or {}).get("mode") == "HARD":
                return False
    elif plans:
        return False
    return True


class KDICLightweightRouterV1:
    def __init__(self, config: RouterConfig | None = None):
        self.config = config or RouterConfig()

    def run(
        self,
        query: str,
        *,
        previous_turns: Any = None,
        conversation_state: Mapping[str, Any] | None = None,
    ) -> dict[str, Any]:
        started = time.perf_counter()
        normalized = normalize_query(query)
        original = normalized["original_query"]
        normalized_text = normalized["normalized_query"]
        context = build_context(previous_turns, conversation_state)
        route, route_reasons, missing, direct_action = detect_route(normalized_text, context=context)

        if route == "RETRIEVE":
            complexity = detect_complexity(normalized_text)
            decomposition = decompose_query(normalized_text, complexity, self.config)
        else:
            complexity = {
                "question_type": "NONE",
                "businesses": [],
                "intents": [],
                "clause_count": 0,
                "reasons": ["NO_RETRIEVAL_ROUTE"],
            }
            decomposition = {
                "status": "NOT_APPLICABLE",
                "subqueries": [],
                "issues": [],
                "fallback_to_original": False,
            }

        needs, plans = build_query_plans(original, route, complexity, decomposition, self.config)
        analysis = {
            "route": route,
            "question_type": complexity["question_type"],
            "business_functions": complexity["businesses"],
            "intents": complexity["intents"],
            "needs": needs,
            "missing_information": missing,
            "decomposition_status": decomposition["status"],
        }
        result = {
            "pipeline_version": PIPELINE_VERSION,
            "analysis_status": "OK",
            "original_query": original,
            "normalized_query": normalized_text,
            "normalization_changes": normalized["changes"],
            "context": context,
            "route_reasons": route_reasons,
            "direct_action": direct_action,
            "complexity": complexity,
            "decomposition": decomposition,
            "analysis": analysis,
            "query_plans": plans,
            "validation_warnings": [],
            "runtime": {
                "api_request_count": 0,
                "prompt_tokens": 0,
                "completion_tokens": 0,
                "total_tokens": 0,
                "latency_ms": round((time.perf_counter() - started) * 1000, 3),
            },
        }
        if not query_plan_is_valid(result):
            result["analysis_status"] = "INVALID_PLAN"
            result["validation_warnings"].append("QUERY_PLAN_VALIDATION_FAILED")
        return result


def route_query(
    query: str,
    *,
    previous_turns: Any = None,
    conversation_state: Mapping[str, Any] | None = None,
    config: RouterConfig | None = None,
) -> dict[str, Any]:
    return KDICLightweightRouterV1(config).run(
        query,
        previous_turns=previous_turns,
        conversation_state=conversation_state,
    )


if __name__ == "__main__":
    examples = (
        "예금자보호 한도는 얼마인가요?",
        "예금자보호 한도는 얼마인가요? 그리고 착오송금 반환지원은 누가 신청할 수 있나요?",
        "신청 방법을 알려주세요.",
        "안녕하세요.",
    )
    router = KDICLightweightRouterV1()
    for example in examples:
        print(json.dumps(router.run(example), ensure_ascii=False, indent=2))


In [ ]:
%%writefile kdic_lightweight_query_ablation_core.py
from __future__ import annotations

import hashlib
import json
import re
import time
import uuid
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Mapping, Sequence

import pandas as pd
import requests

import kdic_lightweight_router_v1 as light_router
from kdic_integrated_eval_core import AnalyzerCase, QueryPlan, normalize_route


VERSION_V10 = "LIGHT_V1.0_ORIGINAL"
VERSION_V11 = "LIGHT_V1.1_RULE_FALLBACK_ORIGINAL"
VERSION_V12 = "LIGHT_V1.2_RULE_THEN_LLM"
VERSION_V13 = "LIGHT_V1.3_LLM_ON_COMPLEX"
VERSION_ORDER = (VERSION_V10, VERSION_V11, VERSION_V12, VERSION_V13)

VERSION_DESCRIPTIONS = {
    VERSION_V10: "공통 라우팅 후 RETRIEVE 질의는 원문 하나만 검색",
    VERSION_V11: "복합 가능성이 높으면 규칙 분해, 실패 시 원문 검색",
    VERSION_V12: "복합 가능성이 높으면 규칙 분해, 실패 시 LLM 구조화 분해, 다시 실패하면 원문 검색",
    VERSION_V13: "복합 가능성이 높으면 규칙을 건너뛰고 LLM 구조화 분해, 실패 시 원문 검색",
}

DECOMPOSITION_PROMPT_VERSION = "KDIC_DECOMPOSE_STRUCTURED_V1_2026_08_13"
NUMBER_PATTERN = re.compile(r"\d+(?:[.,]\d+)?")
NEGATION_TERMS = ("아닌", "아니", "않", "못", "제외", "불가", "없이", "없", "미해당")
TOKEN_PATTERN = re.compile(r"[가-힣A-Za-z0-9]+")


@dataclass(frozen=True)
class AblationConfig:
    original_anchor_weight: float = 0.60
    decomposition_weight: float = 0.40
    max_subqueries: int = 4
    llm_min_confidence: float = 0.80
    llm_model: str = "HCX-007"
    llm_endpoint: str = "https://clovastudio.stream.ntruss.com/v3/chat-completions/HCX-007"
    llm_timeout_seconds: float = 90.0
    llm_max_retries: int = 2
    request_delay_seconds: float = 0.0

    def __post_init__(self) -> None:
        total = self.original_anchor_weight + self.decomposition_weight
        if abs(total - 1.0) > 1e-9:
            raise ValueError(f"검색 질의 가중치 합은 1이어야 합니다: {total}")
        if self.max_subqueries < 2:
            raise ValueError("max_subqueries는 2 이상이어야 합니다.")


def _now_ms() -> float:
    return time.perf_counter() * 1000.0


def _ordered_unique(values: Sequence[Any]) -> list[str]:
    seen: set[str] = set()
    output: list[str] = []
    for value in values:
        text = str(value or "").strip()
        if text and text not in seen:
            output.append(text)
            seen.add(text)
    return output


def _parse_previous_turns(value: Any) -> Any:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, (list, dict)):
        return value
    text = str(value).strip()
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        return text


def normalize_gold_route(value: Any) -> str:
    route = normalize_route(value)
    if route in {"SIMPLE_RETRIEVE", "MULTI_RETRIEVE", "RETRIEVE_RELAXED"}:
        return "RETRIEVE"
    return route


def analyze_common(
    evaluation_id: str,
    question: str,
    *,
    previous_turns: Any = None,
    router_config: light_router.RouterConfig | None = None,
) -> dict[str, Any]:
    """네 버전이 공유하는 정규화·라우팅·복합가능성 판별을 한 번 수행한다."""
    config = router_config or light_router.RouterConfig()
    common_started = _now_ms()
    normalized = light_router.normalize_query(question)
    context = light_router.build_context(_parse_previous_turns(previous_turns), None)
    route_raw, route_reasons, missing, direct_action = light_router.detect_route(
        normalized["normalized_query"], context=context
    )
    route = normalize_route(route_raw)
    if route == "RETRIEVE":
        complexity = light_router.detect_complexity(normalized["normalized_query"])
    else:
        complexity = {
            "question_type": "NONE",
            "businesses": [],
            "intents": [],
            "clause_count": 0,
            "reasons": ["NO_RETRIEVAL_ROUTE"],
        }
    common_latency_ms = _now_ms() - common_started

    rule_started = _now_ms()
    if route == "RETRIEVE" and complexity.get("question_type") == "MULTI":
        rule_decomposition = light_router.decompose_query(
            normalized["normalized_query"], complexity, config
        )
    else:
        rule_decomposition = {
            "status": "NOT_REQUIRED" if route == "RETRIEVE" else "NOT_APPLICABLE",
            "subqueries": [normalized["normalized_query"]] if route == "RETRIEVE" else [],
            "issues": [],
            "fallback_to_original": False,
        }
    rule_latency_ms = _now_ms() - rule_started

    return {
        "evaluation_id": str(evaluation_id),
        "original_question": str(question).strip(),
        "normalized_question": normalized["normalized_query"],
        "normalization_changes": normalized.get("changes") or [],
        "context": context,
        "route": route,
        "route_reasons": route_reasons,
        "missing_information": missing,
        "direct_action": direct_action,
        "complexity": complexity,
        "complex_candidate": route == "RETRIEVE" and complexity.get("question_type") == "MULTI",
        "rule_decomposition": rule_decomposition,
        "common_latency_ms": round(common_latency_ms, 3),
        "rule_latency_ms": round(rule_latency_ms, 3),
    }


def llm_required(version: str, common: Mapping[str, Any]) -> bool:
    if common.get("route") != "RETRIEVE" or not common.get("complex_candidate"):
        return False
    if version == VERSION_V12:
        return (common.get("rule_decomposition") or {}).get("status") != "COMPLETE"
    return version == VERSION_V13


def decomposition_json_schema(max_subqueries: int = 4) -> dict[str, Any]:
    return {
        "type": "object",
        "properties": {
            "decomposable": {"type": "boolean"},
            "subqueries": {
                "type": "array",
                "maxItems": int(max_subqueries),
                "items": {
                    "type": "object",
                    "properties": {"query": {"type": "string"}},
                    "required": ["query"],
                    "additionalProperties": False,
                },
            },
            "confidence": {"type": "number", "minimum": 0, "maximum": 1},
            "reason": {"type": "string"},
        },
        "required": ["decomposable", "subqueries", "confidence", "reason"],
        "additionalProperties": False,
    }


def build_decomposition_messages(question: str) -> list[dict[str, str]]:
    system = """
당신은 예금보험공사 검색 파이프라인의 복합질의 분해기입니다.
이 작업은 질의 재작성이나 검색어 최적화가 아니라, 원문에 실제로 들어 있는 독립 정보 요구를 구조적으로 분리하는 작업입니다.

규칙:
1. 서로 따로 검색하고 답할 수 있는 정보 요구가 2개 이상일 때만 decomposable=true로 판단합니다.
2. 단일 업무의 하나의 응집된 질문, 용어 정의, 비교 관계 자체를 묻는 질문은 분리하지 않습니다.
3. 원문의 업무명, 대상, 조건, 숫자, 기간, 부정 표현을 빠뜨리거나 바꾸지 않습니다.
4. 원문에 없는 업무, 조건, 숫자, 예외, 의도를 추가하지 않습니다.
5. 문체 개선, 요약, 동의어 확장, 검색 키워드 생성은 하지 않습니다.
6. 각 하위질문은 단독으로 이해 가능한 한국어 질문이어야 합니다.
7. 분리할 수 없거나 확신이 낮으면 decomposable=false, subqueries=[]로 반환합니다.
8. 하위질문은 2개 이상 4개 이하로 제한합니다.
""".strip()
    user = f"원문 질문:\n{question}\n\n원문의 독립 정보 요구만 판별하고 JSON 스키마에 맞춰 반환하세요."
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def _extract_hcx_payload(response_json: Mapping[str, Any]) -> tuple[dict[str, Any], dict[str, int]]:
    result = response_json.get("result") or {}
    message = result.get("message") or {}
    content = message.get("content")
    if isinstance(content, Mapping):
        payload = dict(content)
    else:
        text = str(content or "").strip()
        if text.startswith("```"):
            text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.I | re.S).strip()
        payload = json.loads(text)
    usage = result.get("usage") or response_json.get("usage") or {}
    prompt_tokens = int(usage.get("promptTokens") or usage.get("prompt_tokens") or 0)
    completion_tokens = int(usage.get("completionTokens") or usage.get("completion_tokens") or 0)
    total_tokens = int(usage.get("totalTokens") or usage.get("total_tokens") or prompt_tokens + completion_tokens)
    return payload, {
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
    }


def _token_overlap_ratio(original: str, candidate: str) -> float:
    original_tokens = set(TOKEN_PATTERN.findall(original.lower()))
    candidate_tokens = set(TOKEN_PATTERN.findall(candidate.lower()))
    if not candidate_tokens:
        return 0.0
    return len(original_tokens & candidate_tokens) / len(candidate_tokens)


def validate_llm_decomposition(
    original: str,
    payload: Mapping[str, Any],
    *,
    expected_businesses: Sequence[str],
    config: AblationConfig,
) -> dict[str, Any]:
    issues: list[str] = []
    decomposable = payload.get("decomposable") is True
    confidence = float(payload.get("confidence") or 0.0)
    reason = str(payload.get("reason") or "").strip()
    raw_subqueries = payload.get("subqueries") or []
    if not isinstance(raw_subqueries, list):
        raw_subqueries = []
        issues.append("SUBQUERIES_NOT_ARRAY")
    raw_subquery_count = len(raw_subqueries)
    subqueries = _ordered_unique(
        [item.get("query") if isinstance(item, Mapping) else item for item in raw_subqueries]
    )[: config.max_subqueries]

    if not decomposable:
        return {
            "status": "DECLINED",
            "accepted": False,
            "subqueries": [],
            "confidence": confidence,
            "reason": reason,
            "issues": ["LLM_DECLINED_DECOMPOSITION"],
        }
    if confidence < config.llm_min_confidence:
        issues.append("LOW_LLM_CONFIDENCE")

    base_validation = light_router.validate_decomposition(original, subqueries, expected_businesses)
    issues.extend(base_validation.get("issues") or [])
    if raw_subquery_count > config.max_subqueries:
        issues.append("TOO_MANY_SUBQUERIES")

    reconstructed = " ".join(subqueries)
    original_numbers = set(NUMBER_PATTERN.findall(original))
    generated_numbers = set(NUMBER_PATTERN.findall(reconstructed))
    if generated_numbers - original_numbers:
        issues.append("INVENTED_NUMERIC_CONSTRAINT")
    original_negations = {term for term in NEGATION_TERMS if term in original}
    generated_negations = {term for term in NEGATION_TERMS if term in reconstructed}
    if generated_negations - original_negations:
        issues.append("INVENTED_NEGATION")
    generated_businesses = set(light_router.find_businesses(reconstructed))
    expected_business_set = set(expected_businesses)
    if expected_business_set and generated_businesses - expected_business_set:
        issues.append("INVENTED_BUSINESS")
    for subquery in subqueries:
        if _token_overlap_ratio(original, subquery) < 0.25:
            issues.append("LOW_SOURCE_TERM_OVERLAP")
            break

    issues = _ordered_unique(issues)
    accepted = not issues and 2 <= len(subqueries) <= config.max_subqueries
    return {
        "status": "COMPLETE" if accepted else "FAILED",
        "accepted": accepted,
        "subqueries": subqueries if accepted else [],
        "confidence": confidence,
        "reason": reason,
        "issues": issues,
    }


class HCXStructuredDecomposer:
    def __init__(
        self,
        api_key: str,
        *,
        config: AblationConfig | None = None,
        cache_path: str | Path | None = None,
        session: requests.Session | None = None,
    ) -> None:
        key = str(api_key or "").strip()
        if not key or key.lower().startswith("bearer ") or any(ch.isspace() for ch in key):
            raise ValueError("HCX_API_KEY에는 Bearer 접두사나 공백을 넣지 않습니다.")
        self.api_key = key
        self.config = config or AblationConfig()
        self.cache_path = Path(cache_path) if cache_path else None
        self.session = session or requests.Session()
        self.cache: dict[str, dict[str, Any]] = {}
        if self.cache_path and self.cache_path.exists():
            with self.cache_path.open(encoding="utf-8") as handle:
                for line in handle:
                    if line.strip():
                        row = json.loads(line)
                        self.cache[str(row["cache_key"])] = row

    def _cache_key(self, question: str, expected_businesses: Sequence[str]) -> str:
        raw = json.dumps(
            {
                "prompt_version": DECOMPOSITION_PROMPT_VERSION,
                "model": self.config.llm_model,
                "question": question,
                "expected_businesses": list(expected_businesses),
                "min_confidence": self.config.llm_min_confidence,
            },
            ensure_ascii=False,
            sort_keys=True,
        )
        return hashlib.sha256(raw.encode("utf-8")).hexdigest()

    def _append_cache(self, row: Mapping[str, Any]) -> None:
        if not self.cache_path:
            return
        self.cache_path.parent.mkdir(parents=True, exist_ok=True)
        with self.cache_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(dict(row), ensure_ascii=False) + "\n")

    def decompose(self, question: str, expected_businesses: Sequence[str]) -> dict[str, Any]:
        cache_key = self._cache_key(question, expected_businesses)
        if cache_key in self.cache:
            cached = dict(self.cache[cache_key])
            cached["cache_hit"] = True
            cached["actual_api_latency_ms"] = 0.0
            return cached

        body = {
            "messages": build_decomposition_messages(question),
            "topP": 0.1,
            "topK": 0,
            "maxCompletionTokens": 700,
            "temperature": 0.0,
            "repetitionPenalty": 1.0,
            "thinking": {"effort": "none"},
            "stop": [],
            "responseFormat": {"type": "json", "schema": decomposition_json_schema(self.config.max_subqueries)},
        }
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
            "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        }
        last_error: Exception | None = None
        for attempt in range(self.config.llm_max_retries + 1):
            started = _now_ms()
            try:
                response = self.session.post(
                    self.config.llm_endpoint,
                    headers=headers,
                    json=body,
                    timeout=self.config.llm_timeout_seconds,
                )
                response.raise_for_status()
                payload, usage = _extract_hcx_payload(response.json())
                api_latency_ms = _now_ms() - started
                validation = validate_llm_decomposition(
                    question,
                    payload,
                    expected_businesses=expected_businesses,
                    config=self.config,
                )
                row = {
                    "cache_key": cache_key,
                    "question": question,
                    "model": self.config.llm_model,
                    "prompt_version": DECOMPOSITION_PROMPT_VERSION,
                    "raw_payload": payload,
                    **validation,
                    **usage,
                    "effective_api_latency_ms": round(api_latency_ms, 3),
                    "actual_api_latency_ms": round(api_latency_ms, 3),
                    "cache_hit": False,
                    "error_type": "",
                    "error_message": "",
                }
                self.cache[cache_key] = row
                self._append_cache(row)
                if self.config.request_delay_seconds > 0:
                    time.sleep(self.config.request_delay_seconds)
                return dict(row)
            except Exception as error:
                last_error = error
                if attempt < self.config.llm_max_retries:
                    time.sleep(min(2 ** attempt, 4))

        row = {
            "cache_key": cache_key,
            "question": question,
            "model": self.config.llm_model,
            "prompt_version": DECOMPOSITION_PROMPT_VERSION,
            "raw_payload": {},
            "status": "ERROR",
            "accepted": False,
            "subqueries": [],
            "confidence": 0.0,
            "reason": "",
            "issues": ["LLM_REQUEST_FAILED"],
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0,
            "effective_api_latency_ms": 0.0,
            "actual_api_latency_ms": 0.0,
            "cache_hit": False,
            "error_type": type(last_error).__name__ if last_error else "UnknownError",
            "error_message": str(last_error or "unknown error"),
        }
        self.cache[cache_key] = row
        self._append_cache(row)
        return dict(row)


def _make_plans(original: str, subqueries: Sequence[str], config: AblationConfig) -> list[QueryPlan]:
    original_compact = re.sub(r"\s+", "", original).lower()
    valid_subqueries = [
        value for value in _ordered_unique(subqueries)
        if re.sub(r"\s+", "", value).lower() != original_compact
    ]
    if len(valid_subqueries) < 2:
        return [
            QueryPlan(
                need_id="FUSED",
                variant_id="ORIGINAL",
                dense_query=original,
                bm25_query=original,
                filter_mode="NONE",
                business_filters=[],
                soft_business_hints=[],
                query_weight=1.0,
                query_source="ORIGINAL",
            )
        ]
    sub_weight = config.decomposition_weight / len(valid_subqueries)
    plans = [
        QueryPlan(
            need_id="FUSED",
            variant_id="ORIGINAL_ANCHOR",
            dense_query=original,
            bm25_query=original,
            filter_mode="NONE",
            business_filters=[],
            soft_business_hints=[],
            query_weight=config.original_anchor_weight,
            query_source="ORIGINAL_ANCHOR",
        )
    ]
    for index, subquery in enumerate(valid_subqueries, 1):
        plans.append(
            QueryPlan(
                need_id="FUSED",
                variant_id=f"SUBQUERY_{index:02d}",
                dense_query=subquery,
                bm25_query=subquery,
                filter_mode="NONE",
                business_filters=[],
                soft_business_hints=[],
                query_weight=sub_weight,
                query_source="DECOMPOSED",
            )
        )
    return plans


def build_version_case(
    common: Mapping[str, Any],
    version: str,
    *,
    llm_record: Mapping[str, Any] | None = None,
    config: AblationConfig | None = None,
) -> AnalyzerCase:
    if version not in VERSION_ORDER:
        raise ValueError(f"지원하지 않는 버전: {version}")
    cfg = config or AblationConfig()
    route = str(common["route"])
    original = str(common["original_question"])
    rule = dict(common.get("rule_decomposition") or {})
    candidate = bool(common.get("complex_candidate"))
    subqueries: list[str] = []
    source = "ORIGINAL_POLICY"
    fallback_reason = ""
    policy_rule_used = False
    policy_llm_called = False

    if route == "RETRIEVE" and candidate:
        if version in {VERSION_V11, VERSION_V12}:
            policy_rule_used = True
            if rule.get("status") == "COMPLETE":
                subqueries = list(rule.get("subqueries") or [])
                source = "RULE"
            elif version == VERSION_V12:
                policy_llm_called = True
                if llm_record and llm_record.get("accepted"):
                    subqueries = list(llm_record.get("subqueries") or [])
                    source = "LLM"
                else:
                    source = "ORIGINAL_FALLBACK"
                    fallback_reason = "LLM_FAILED_OR_DECLINED"
            else:
                source = "ORIGINAL_FALLBACK"
                fallback_reason = "RULE_DECOMPOSITION_FAILED"
        elif version == VERSION_V13:
            policy_llm_called = True
            if llm_record and llm_record.get("accepted"):
                subqueries = list(llm_record.get("subqueries") or [])
                source = "LLM"
            else:
                source = "ORIGINAL_FALLBACK"
                fallback_reason = "LLM_FAILED_OR_DECLINED"

    if route == "RETRIEVE":
        plans = _make_plans(original, subqueries, cfg)
    else:
        plans = []

    analysis_latency_ms = float(common.get("common_latency_ms") or 0.0)
    if policy_rule_used:
        analysis_latency_ms += float(common.get("rule_latency_ms") or 0.0)
    if policy_llm_called and llm_record:
        analysis_latency_ms += float(llm_record.get("effective_api_latency_ms") or 0.0)

    raw_result = {
        "pipeline_version": version,
        "analysis_status": "OK",
        "original_query": original,
        "normalized_query": common.get("normalized_question"),
        "route_reasons": common.get("route_reasons") or [],
        "direct_action": common.get("direct_action"),
        "blocking_slot": (common.get("missing_information") or [None])[0],
        "complexity": common.get("complexity") or {},
        "complex_candidate": candidate,
        "rule_decomposition": rule,
        "llm_decomposition": dict(llm_record or {}),
        "decomposition_source": source,
        "final_subqueries": subqueries,
        "fallback_reason": fallback_reason,
        "model_needs": [
            {"business_function": value}
            for value in (common.get("complexity") or {}).get("businesses") or []
        ],
        "runtime": {
            "api_request_count": int(policy_llm_called),
            "prompt_tokens": int((llm_record or {}).get("prompt_tokens") or 0) if policy_llm_called else 0,
            "completion_tokens": int((llm_record or {}).get("completion_tokens") or 0) if policy_llm_called else 0,
            "total_tokens": int((llm_record or {}).get("total_tokens") or 0) if policy_llm_called else 0,
            "latency_ms": round(analysis_latency_ms, 3),
        },
    }
    return AnalyzerCase(
        evaluation_id=str(common["evaluation_id"]),
        analyzer=version,
        original_question=original,
        route=route,
        analysis_latency_ms=round(analysis_latency_ms, 3),
        plans=plans,
        raw_result=raw_result,
    )


def build_all_cases(
    eval_df: pd.DataFrame,
    *,
    decomposer: HCXStructuredDecomposer | None,
    config: AblationConfig | None = None,
    previous_turns_column: str = "previous_turns",
) -> tuple[dict[tuple[str, str], AnalyzerCase], pd.DataFrame]:
    cfg = config or AblationConfig()
    common_by_id: dict[str, dict[str, Any]] = {}
    for row in eval_df.to_dict(orient="records"):
        evaluation_id = str(row["evaluation_id"])
        common_by_id[evaluation_id] = analyze_common(
            evaluation_id,
            str(row["question"]),
            previous_turns=row.get(previous_turns_column),
        )

    llm_by_id: dict[str, dict[str, Any]] = {}
    required_ids = [
        evaluation_id
        for evaluation_id, common in common_by_id.items()
        if any(llm_required(version, common) for version in (VERSION_V12, VERSION_V13))
    ]
    if required_ids and decomposer is None:
        raise ValueError("V1.2/V1.3 평가에는 HCXStructuredDecomposer가 필요합니다.")
    for evaluation_id in required_ids:
        common = common_by_id[evaluation_id]
        llm_by_id[evaluation_id] = decomposer.decompose(
            str(common["normalized_question"]),
            list((common.get("complexity") or {}).get("businesses") or []),
        )

    cases: dict[tuple[str, str], AnalyzerCase] = {}
    audit_rows: list[dict[str, Any]] = []
    for evaluation_id, common in common_by_id.items():
        llm_record = llm_by_id.get(evaluation_id)
        for version in VERSION_ORDER:
            case = build_version_case(common, version, llm_record=llm_record, config=cfg)
            cases[(version, evaluation_id)] = case
            runtime = case.raw_result["runtime"]
            audit_rows.append({
                "evaluation_id": evaluation_id,
                "question": case.original_question,
                "version": version,
                "route": case.route,
                "complex_candidate": bool(common.get("complex_candidate")),
                "rule_status": (common.get("rule_decomposition") or {}).get("status"),
                "rule_subqueries": (common.get("rule_decomposition") or {}).get("subqueries") or [],
                "llm_policy_call": int(llm_required(version, common)),
                "llm_actual_api_call": int(bool(llm_record) and not bool(llm_record.get("cache_hit"))) if llm_required(version, common) else 0,
                "llm_cache_hit": bool((llm_record or {}).get("cache_hit")) if llm_required(version, common) else False,
                "llm_status": (llm_record or {}).get("status", "NOT_CALLED"),
                "llm_confidence": float((llm_record or {}).get("confidence") or 0.0),
                "llm_issues": (llm_record or {}).get("issues") or [],
                "decomposition_source": case.raw_result["decomposition_source"],
                "final_subqueries": case.raw_result["final_subqueries"],
                "query_plan_count": len(case.plans),
                "query_plan_weight_sum": round(sum(plan.query_weight for plan in case.plans), 10),
                "hard_filter_count": sum(plan.filter_mode == "HARD" for plan in case.plans),
                "analysis_latency_ms": case.analysis_latency_ms,
                "prompt_tokens": runtime["prompt_tokens"],
                "completion_tokens": runtime["completion_tokens"],
                "total_tokens": runtime["total_tokens"],
            })
    return cases, pd.DataFrame(audit_rows)


def summarize_router_ablation(audit_df: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for version, frame in audit_df.groupby("version", sort=False):
        retrieve = frame[frame["route"].eq("RETRIEVE")]
        rows.append({
            "version": version,
            "question_count": len(frame),
            "retrieve_count": int(frame["route"].eq("RETRIEVE").sum()),
            "clarify_count": int(frame["route"].eq("CLARIFY").sum()),
            "out_of_scope_count": int(frame["route"].eq("OUT_OF_SCOPE").sum()),
            "direct_response_count": int(frame["route"].eq("DIRECT_RESPONSE").sum()),
            "complex_candidate_count": int(frame["complex_candidate"].sum()),
            "decomposed_count": int(frame["decomposition_source"].isin(["RULE", "LLM"]).sum()),
            "rule_decomposed_count": int(frame["decomposition_source"].eq("RULE").sum()),
            "llm_decomposed_count": int(frame["decomposition_source"].eq("LLM").sum()),
            "original_fallback_count": int(frame["decomposition_source"].eq("ORIGINAL_FALLBACK").sum()),
            "llm_policy_call_count": int(frame["llm_policy_call"].sum()),
            "llm_total_tokens": int(frame["total_tokens"].sum()),
            "analysis_latency_ms_mean_all": float(frame["analysis_latency_ms"].mean()),
            "analysis_latency_ms_p95_all": float(frame["analysis_latency_ms"].quantile(0.95)),
            "retrieval_query_count_mean": float(retrieve["query_plan_count"].mean()) if len(retrieve) else 0.0,
            "hard_filter_count": int(frame["hard_filter_count"].sum()),
            "invalid_query_plan_weight_count": int((retrieve["query_plan_weight_sum"].sub(1.0).abs() > 1e-9).sum()),
        })
    summary = pd.DataFrame(rows)
    summary["version"] = pd.Categorical(summary["version"], VERSION_ORDER, ordered=True)
    return summary.sort_values("version").reset_index(drop=True)


def route_hard_gate_report(audit_df: pd.DataFrame, eval_df: pd.DataFrame) -> pd.DataFrame:
    gold = eval_df[["evaluation_id", "gold_route_v6"]].copy() if "gold_route_v6" in eval_df.columns else pd.DataFrame()
    if not gold.empty:
        gold["gold_route_v6"] = gold["gold_route_v6"].map(normalize_gold_route)
    rows: list[dict[str, Any]] = []
    for version, frame in audit_df.groupby("version", sort=False):
        merged = frame.merge(gold, on="evaluation_id", how="left") if not gold.empty else frame.assign(gold_route_v6="")
        route_known = merged["gold_route_v6"].fillna("").ne("")
        normal_retrieve = merged["gold_route_v6"].eq("RETRIEVE")
        predicted_retrieve = merged["route"].eq("RETRIEVE")
        query_valid_rate = float(merged.loc[predicted_retrieve, "query_plan_count"].gt(0).mean()) if predicted_retrieve.any() else 1.0
        wrong_oos = int((normal_retrieve & merged["route"].eq("OUT_OF_SCOPE")).sum())
        wrong_direct = int((normal_retrieve & merged["route"].eq("DIRECT_RESPONSE")).sum())
        route_accuracy = float((merged.loc[route_known, "route"] == merged.loc[route_known, "gold_route_v6"]).mean()) if route_known.any() else float("nan")
        predicted_clarify = merged["route"].eq("CLARIFY") & route_known
        clarify_precision = float(merged.loc[predicted_clarify, "gold_route_v6"].eq("CLARIFY").mean()) if predicted_clarify.any() else 1.0
        rows.extend([
            {"version": version, "gate": "실행 성공률", "value": 1.0, "threshold": ">=0.995", "passed": True},
            {"version": version, "gate": "검색 질의 생성 유효율", "value": query_valid_rate, "threshold": ">=0.99", "passed": query_valid_rate >= 0.99},
            {"version": version, "gate": "정상 질문의 잘못된 OOS", "value": wrong_oos, "threshold": "=0", "passed": wrong_oos == 0},
            {"version": version, "gate": "정상 질문의 잘못된 DIRECT", "value": wrong_direct, "threshold": "=0", "passed": wrong_direct == 0},
            {"version": version, "gate": "Hard Filter", "value": int(frame["hard_filter_count"].sum()), "threshold": "=0", "passed": int(frame["hard_filter_count"].sum()) == 0},
            {"version": version, "gate": "검색 불가능 질문의 추가질문 Precision", "value": clarify_precision, "threshold": ">=0.95", "passed": clarify_precision >= 0.95},
            {"version": version, "gate": "최종 라우팅 정확도", "value": route_accuracy, "threshold": "참고", "passed": True},
        ])
    return pd.DataFrame(rows)


def query_analysis_quality_summary(audit_df: pd.DataFrame, eval_df: pd.DataFrame) -> pd.DataFrame:
    """라우팅과 최종 분해 여부를 gold와 비교한다. gold 열이 없으면 해당 값은 NaN이다."""
    gold_columns = [column for column in ("evaluation_id", "gold_route_v6", "split_needed") if column in eval_df.columns]
    gold = eval_df[gold_columns].copy()
    if "gold_route_v6" in gold.columns:
        gold["gold_route"] = gold["gold_route_v6"].map(normalize_gold_route)
    else:
        gold["gold_route"] = ""
    if "split_needed" in gold.columns:
        gold["gold_multi"] = gold["split_needed"].map(
            lambda value: str(value).strip().lower() in {"1", "true", "y", "yes", "multi", "복합", "필요"}
            if not pd.isna(value) and str(value).strip() else pd.NA
        )
    else:
        gold["gold_multi"] = pd.NA

    rows: list[dict[str, Any]] = []
    for version, frame in audit_df.groupby("version", sort=False):
        merged = frame.merge(gold[["evaluation_id", "gold_route", "gold_multi"]], on="evaluation_id", how="left")
        known_route = merged["gold_route"].fillna("").ne("")
        route_accuracy = float(merged.loc[known_route, "route"].eq(merged.loc[known_route, "gold_route"]).mean()) if known_route.any() else float("nan")
        gold_retrieve = merged["gold_route"].eq("RETRIEVE")
        retrieve_recall = float(merged.loc[gold_retrieve, "route"].eq("RETRIEVE").mean()) if gold_retrieve.any() else float("nan")
        predicted_clarify = merged["route"].eq("CLARIFY") & known_route
        clarify_precision = float(merged.loc[predicted_clarify, "gold_route"].eq("CLARIFY").mean()) if predicted_clarify.any() else 1.0

        known_multi = merged["gold_multi"].notna() & gold_retrieve
        predicted_multi = merged["decomposition_source"].isin(["RULE", "LLM"])
        tp = int((known_multi & merged["gold_multi"].astype("boolean").fillna(False) & predicted_multi).sum())
        fp = int((known_multi & ~merged["gold_multi"].astype("boolean").fillna(False) & predicted_multi).sum())
        fn = int((known_multi & merged["gold_multi"].astype("boolean").fillna(False) & ~predicted_multi).sum())
        if not known_multi.any():
            precision = recall = f1 = float("nan")
        else:
            precision = tp / (tp + fp) if tp + fp else 1.0
            recall = tp / (tp + fn) if tp + fn else 1.0
            f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append({
            "version": version,
            "route_accuracy": route_accuracy,
            "retrieve_recall": retrieve_recall,
            "clarify_precision": clarify_precision,
            "multi_precision": precision,
            "multi_recall": recall,
            "multi_f1": f1,
            "multi_tp": tp,
            "multi_fp": fp,
            "multi_fn": fn,
            "analysis_latency_ms_mean": float(frame["analysis_latency_ms"].mean()),
            "llm_policy_call_count": int(frame["llm_policy_call"].sum()),
            "llm_total_tokens": int(frame["total_tokens"].sum()),
        })
    summary = pd.DataFrame(rows)
    summary["version"] = pd.Categorical(summary["version"], VERSION_ORDER, ordered=True)
    return summary.sort_values("version").reset_index(drop=True)


def case_signature(case: AnalyzerCase) -> str:
    payload = {
        "route": case.route,
        "plans": [asdict(plan) for plan in case.plans],
        "source": case.raw_result.get("decomposition_source"),
    }
    return hashlib.sha256(json.dumps(payload, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()[:16]


In [ ]:
%%writefile kdic_decomposition_quality_core.py
from __future__ import annotations

import hashlib
import json
import math
import re
import time
import uuid
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Callable, Mapping, Sequence

import pandas as pd
try:
    import requests
except ImportError:  # 로컬 정적 검증 환경에서는 HTTP 호출을 사용하지 않을 수 있습니다.
    requests = None  # type: ignore[assignment]

import kdic_lightweight_router_v1 as light_router


DATE_PREFIX = "2026-08-14"

BASELINE = "V1.5_BASELINE"
QUALITY = "V1.5_Q"
RETRY = "V1.5_R"
QUALITY_RETRY = "V1.5_QR"
CONDITION_ORDER = (BASELINE, QUALITY, RETRY, QUALITY_RETRY)
CONDITION_LABELS = {
    BASELINE: "V1.5 Baseline",
    QUALITY: "V1.5-Q 품질개선",
    RETRY: "V1.5-R 교정재시도",
    QUALITY_RETRY: "V1.5-QR 품질개선+재시도",
}

PROMPT_VERSION = "KDIC_DECOMPOSITION_QUALITY_V1_2026_08_14"
REPAIR_PROMPT_VERSION = "KDIC_DECOMPOSITION_REPAIR_V1_2026_08_14"

INTENT_VALUES = (
    "OVERVIEW", "ELIGIBILITY", "AMOUNT", "APPLICATION", "DOCUMENTS",
    "TIME", "STATUS", "CALCULATION", "EXCEPTION", "OTHER",
)
INTENT_PATTERNS: dict[str, tuple[str, ...]] = {
    "AMOUNT": (r"한도", r"얼마", r"금액", r"최대", r"최소", r"몇\s*원", r"비율"),
    "APPLICATION": (
        r"신청\s*(?:방법|절차)", r"신청하려면", r"접수\s*(?:방법|절차)?",
        r"어떻게\s*(?:신청|받|진행)",
    ),
    "DOCUMENTS": (r"서류", r"준비물", r"증빙", r"제출"),
    "TIME": (r"언제", r"기간", r"기한", r"시점", r"며칠", r"몇\s*개월"),
    "STATUS": (r"조회", r"확인", r"찾(?:는|을|아)", r"남았(?:는지|나요)"),
    "CALCULATION": (r"계산", r"산정", r"합산"),
    "EXCEPTION": (r"제외", r"예외", r"불가", r"해당하지", r"받지\s*못"),
    "ELIGIBILITY": (r"대상", r"자격", r"조건", r"누가", r"가능한지", r"받을\s*수\s*있"),
    "OVERVIEW": (r"무엇(?:인가요|인지|이죠)?", r"의미", r"차이", r"어떤\s*제도", r"설명"),
}
SUBJECT_TERMS = (
    "상속인", "본인", "대리인", "법인", "개인", "채무자", "송금인", "수취인",
    "미성년자", "친권자", "외국인", "고인", "피상속인", "금융회사",
)
NEGATION_TERMS = ("아닌", "아니", "않", "못", "제외", "불가", "없이", "없", "미해당", "뿐 아니라")
NUMBER_PATTERN = re.compile(r"\d+(?:[.,]\d+)?(?:\s*(?:원|만원|천만원|억원|%|퍼센트|년|개월|일|회))?")
TOKEN_PATTERN = re.compile(r"[가-힣A-Za-z0-9]+")
UNRESOLVED_REFERENCE_PATTERN = re.compile(r"(?:그것|그거|이것|해당\s*(?:제도|경우|업무)|그\s*제도|앞의\s*내용)")


@dataclass(frozen=True)
class QualityConfig:
    llm_endpoint: str = "https://clovastudio.stream.ntruss.com/v3/chat-completions/HCX-007"
    llm_model: str = "HCX-007"
    llm_timeout_seconds: float = 120.0
    transport_retries: int = 2
    semantic_retries: int = 1
    llm_min_confidence: float = 0.80
    max_subqueries: int = 4
    request_delay_seconds: float = 0.0
    original_weight: float = 0.40
    subquery_total_weight: float = 0.60

    def __post_init__(self) -> None:
        if abs(self.original_weight + self.subquery_total_weight - 1.0) > 1e-9:
            raise ValueError("원문과 하위질의 가중치 합은 1이어야 합니다.")
        if self.semantic_retries != 1:
            raise ValueError("이번 실험의 의미 교정 재시도는 정확히 1회로 고정합니다.")


def ordered_unique(values: Sequence[Any]) -> list[str]:
    result: list[str] = []
    seen: set[str] = set()
    for value in values:
        text = re.sub(r"\s+", " ", str(value or "")).strip()
        if text and text not in seen:
            seen.add(text)
            result.append(text)
    return result


def find_intents(text: str) -> list[str]:
    output: list[str] = []
    for intent, patterns in INTENT_PATTERNS.items():
        if any(re.search(pattern, text) for pattern in patterns):
            output.append(intent)
    if len(output) > 1 and "OVERVIEW" in output:
        output.remove("OVERVIEW")
    return output


def source_features(question: str, expected_businesses: Sequence[str]) -> dict[str, Any]:
    matches = light_router.find_business_matches(question)
    evidence_by_business = {
        str(row["business_function"]): ordered_unique(row.get("evidence") or [])
        for row in matches
    }
    return {
        "expected_businesses": ordered_unique(expected_businesses),
        "expected_intents": find_intents(question),
        "numbers": ordered_unique(NUMBER_PATTERN.findall(question)),
        "negations": [term for term in NEGATION_TERMS if term in question],
        "subjects": [term for term in SUBJECT_TERMS if term in question],
        "business_evidence": evidence_by_business,
    }


def quality_json_schema(max_subqueries: int) -> dict[str, Any]:
    business_values = sorted(light_router.BUSINESS_FUNCTIONS)
    return {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "decomposable": {"type": "boolean"},
            "confidence": {"type": "number", "minimum": 0, "maximum": 1},
            "reason": {"type": "string"},
            "subqueries": {
                "type": "array",
                "minItems": 0,
                "maxItems": max_subqueries,
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "query": {"type": "string"},
                        "business_function": {"type": "string", "enum": business_values},
                        "intent": {"type": "string", "enum": list(INTENT_VALUES)},
                        "preserved_terms": {"type": "array", "items": {"type": "string"}},
                        "preserved_constraints": {"type": "array", "items": {"type": "string"}},
                    },
                    "required": [
                        "query", "business_function", "intent",
                        "preserved_terms", "preserved_constraints",
                    ],
                },
            },
        },
        "required": ["decomposable", "confidence", "reason", "subqueries"],
    }


def baseline_json_schema(max_subqueries: int) -> dict[str, Any]:
    return {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "decomposable": {"type": "boolean"},
            "subqueries": {
                "type": "array",
                "maxItems": max_subqueries,
                "items": {
                    "oneOf": [
                        {"type": "string"},
                        {
                            "type": "object",
                            "additionalProperties": False,
                            "properties": {"query": {"type": "string"}},
                            "required": ["query"],
                        },
                    ]
                },
            },
            "confidence": {"type": "number", "minimum": 0, "maximum": 1},
            "reason": {"type": "string"},
        },
        "required": ["decomposable", "subqueries", "confidence", "reason"],
    }


def build_quality_messages(question: str, expected_businesses: Sequence[str]) -> list[dict[str, str]]:
    features = source_features(question, expected_businesses)
    system = """당신은 예금보험공사 검색용 교차업무 질의 구조화 분해기입니다.
라우터가 제시한 업무들은 확정 정답이 아니라 분해 필요성을 검토할 후보입니다.
서로 다른 업무 용어가 보여도 하나의 사건·절차·대상을 비교하거나 설명하는 단일 정보요구라면 decomposable=false와 빈 subqueries를 반환합니다.
서로 독립적으로 검색해야 할 업무별 정보요구가 둘 이상일 때만 decomposable=true로 분해합니다.
원문을 요약하거나 일반화하지 말고, 서로 다른 업무별 독립 검색 질의로만 분리합니다.
각 하위질의는 하나의 업무와 하나의 주된 요청 의도만 담당해야 합니다.
원문의 전문용어, 숫자, 금액, 기간, 부정·제외 표현, 사용자 주체와 조건을 보존합니다.
원문에 없는 업무·숫자·조건을 추가하지 않습니다.
'그것', '해당 경우'처럼 원문 없이 이해할 수 없는 표현을 사용하지 않습니다.
동일 의미의 하위질의를 중복 생성하지 않습니다.
반드시 지정된 JSON Schema만 출력합니다."""
    user = json.dumps({
        "question": question,
        "router_expected_businesses": list(expected_businesses),
        "source_features_to_preserve": features,
        "instruction": "먼저 실제 독립 정보요구가 둘 이상인지 판정하세요. 맞을 때만 각 업무를 담당하는 2~4개 하위질의로 분해하세요.",
    }, ensure_ascii=False, indent=2)
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_repair_messages(
    question: str,
    expected_businesses: Sequence[str],
    previous_payload: Mapping[str, Any],
    issues: Sequence[str],
    *,
    quality_mode: bool,
) -> list[dict[str, str]]:
    system = """당신은 검색용 질의 분해 결과 교정기입니다.
직전 결과 전체를 새로 창작하지 말고, 검증기가 지적한 오류만 수정합니다.
누락된 업무·요청·전문용어·숫자·부정·주체를 복원하고 새 정보는 만들지 않습니다.
라우터 업무 후보는 확정 정답이 아닙니다. 하나의 정보요구라면 decomposable=false로 판단하며 억지로 분해하지 않습니다.
교정 결과도 검증을 통과하지 못하면 폐기되므로 억지로 분해하지 않습니다.
반드시 지정된 JSON Schema만 출력합니다."""
    user = json.dumps({
        "question": question,
        "router_expected_businesses": list(expected_businesses),
        "source_features_to_preserve": source_features(question, expected_businesses),
        "previous_payload": dict(previous_payload),
        "validation_issues": list(issues),
        "output_mode": "quality_structured" if quality_mode else "baseline_structured",
        "instruction": "검증 오류를 정확히 수정하여 2~4개의 독립 검색 질의를 반환하세요.",
    }, ensure_ascii=False, indent=2)
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def extract_queries(payload: Mapping[str, Any]) -> list[str]:
    raw = payload.get("subqueries") or []
    if not isinstance(raw, list):
        return []
    return ordered_unique([
        item.get("query") if isinstance(item, Mapping) else item
        for item in raw
    ])


def _token_overlap_ratio(original: str, candidate: str) -> float:
    original_tokens = set(TOKEN_PATTERN.findall(original.lower()))
    candidate_tokens = set(TOKEN_PATTERN.findall(candidate.lower()))
    if not candidate_tokens:
        return 0.0
    return len(original_tokens & candidate_tokens) / len(candidate_tokens)


def validate_baseline_decomposition(
    question: str,
    payload: Mapping[str, Any],
    *,
    expected_businesses: Sequence[str],
    config: QualityConfig,
) -> dict[str, Any]:
    """기존 V1.5 검증 규칙을 독립적으로 재현한다.

    이 함수는 개선 검증기의 비교 기준이므로 기존 실험 모듈을 import하지 않는다.
    """
    issues: list[str] = []
    decomposable = payload.get("decomposable") is True
    confidence = float(payload.get("confidence") or 0.0)
    reason = str(payload.get("reason") or "").strip()
    raw_subqueries = payload.get("subqueries") or []
    if not isinstance(raw_subqueries, list):
        raw_subqueries = []
        issues.append("SUBQUERIES_NOT_ARRAY")
    raw_subquery_count = len(raw_subqueries)
    subqueries = extract_queries({"subqueries": raw_subqueries})[: config.max_subqueries]

    if not decomposable:
        return {
            "status": "DECLINED",
            "accepted": False,
            "subqueries": [],
            "candidate_subqueries": subqueries,
            "confidence": confidence,
            "reason": reason,
            "issues": ["LLM_DECLINED_DECOMPOSITION"],
            "checks": content_checks(question, payload, expected_businesses),
        }
    if confidence < config.llm_min_confidence:
        issues.append("LOW_LLM_CONFIDENCE")

    base_validation = light_router.validate_decomposition(
        question, subqueries, expected_businesses
    )
    issues.extend(base_validation.get("issues") or [])
    if raw_subquery_count > config.max_subqueries:
        issues.append("TOO_MANY_SUBQUERIES")

    reconstructed = " ".join(subqueries)
    original_numbers = set(NUMBER_PATTERN.findall(question))
    generated_numbers = set(NUMBER_PATTERN.findall(reconstructed))
    if generated_numbers - original_numbers:
        issues.append("INVENTED_NUMERIC_CONSTRAINT")
    original_negations = {term for term in NEGATION_TERMS if term in question}
    generated_negations = {term for term in NEGATION_TERMS if term in reconstructed}
    if generated_negations - original_negations:
        issues.append("INVENTED_NEGATION")
    generated_businesses = set(light_router.find_businesses(reconstructed))
    expected_business_set = set(expected_businesses)
    if expected_business_set and generated_businesses - expected_business_set:
        issues.append("INVENTED_BUSINESS")
    if any(_token_overlap_ratio(question, subquery) < 0.25 for subquery in subqueries):
        issues.append("LOW_SOURCE_TERM_OVERLAP")

    issues = ordered_unique(issues)
    accepted = not issues and 2 <= len(subqueries) <= config.max_subqueries
    return {
        "status": "COMPLETE" if accepted else "FAILED",
        "accepted": accepted,
        "subqueries": subqueries if accepted else [],
        "candidate_subqueries": subqueries,
        "confidence": confidence,
        "reason": reason,
        "issues": issues,
        "checks": content_checks(question, payload, expected_businesses),
    }


def content_checks(
    question: str,
    payload: Mapping[str, Any],
    expected_businesses: Sequence[str],
) -> dict[str, Any]:
    queries = extract_queries(payload)
    reconstructed = " ".join(queries)
    features = source_features(question, expected_businesses)
    generated_businesses = set(light_router.find_businesses(reconstructed))
    expected_business_set = set(features["expected_businesses"])
    generated_intents = set(find_intents(reconstructed))
    expected_intents = set(features["expected_intents"])
    numbers_preserved = all(value in reconstructed for value in features["numbers"])
    negations_preserved = all(value in reconstructed for value in features["negations"])
    subjects_preserved = all(value in reconstructed for value in features["subjects"])
    standalone = all(
        not UNRESOLVED_REFERENCE_PATTERN.search(query)
        or bool(light_router.find_businesses(query))
        for query in queries
    )
    atomic = all(len(set(light_router.find_businesses(query)) & expected_business_set) <= 1 for query in queries)
    return {
        "query_count": len(queries),
        "business_coverage": expected_business_set.issubset(generated_businesses),
        "request_coverage": expected_intents.issubset(generated_intents),
        "numeric_preservation": numbers_preserved,
        "negation_preservation": negations_preserved,
        "subject_preservation": subjects_preserved,
        "standalone_subqueries": standalone,
        "atomic_subqueries": atomic,
        "invented_business_count": len(generated_businesses - expected_business_set),
    }


def validate_quality_decomposition(
    question: str,
    payload: Mapping[str, Any],
    *,
    expected_businesses: Sequence[str],
    config: QualityConfig,
) -> dict[str, Any]:
    base = validate_baseline_decomposition(
        question, payload, expected_businesses=expected_businesses, config=config
    )
    issues = list(base.get("issues") or [])
    queries = extract_queries(payload)
    raw_items = payload.get("subqueries") if isinstance(payload.get("subqueries"), list) else []
    structured_items = [item for item in raw_items if isinstance(item, Mapping)]
    if len(structured_items) != len(raw_items):
        issues.append("MISSING_STRUCTURED_SUBQUERY_FIELDS")

    assigned_businesses: list[str] = []
    assigned_intents: list[str] = []
    pairs: list[tuple[str, str]] = []
    for item in structured_items:
        business = str(item.get("business_function") or "").strip()
        intent = str(item.get("intent") or "").strip()
        query = str(item.get("query") or "").strip()
        assigned_businesses.append(business)
        assigned_intents.append(intent)
        pairs.append((business, intent))
        if business not in expected_businesses:
            issues.append("INVENTED_OR_WRONG_ASSIGNED_BUSINESS")
        if intent not in INTENT_VALUES:
            issues.append("INVALID_ASSIGNED_INTENT")
        detected = set(light_router.find_businesses(query))
        if business and business not in detected:
            issues.append("ASSIGNED_BUSINESS_NOT_EXPLICIT_IN_QUERY")
        if len(detected & set(expected_businesses)) > 1:
            issues.append("NON_ATOMIC_SUBQUERY")
        if UNRESOLVED_REFERENCE_PATTERN.search(query) and not detected:
            issues.append("NON_STANDALONE_SUBQUERY")

    if set(expected_businesses) - set(assigned_businesses):
        issues.append("MISSING_ASSIGNED_BUSINESS_COVERAGE")
    expected_intents = set(find_intents(question))
    if expected_intents - set(assigned_intents):
        issues.append("MISSING_REQUEST_COVERAGE")
    if len(pairs) != len(set(pairs)):
        issues.append("DUPLICATE_BUSINESS_INTENT_PAIR")

    checks = content_checks(question, payload, expected_businesses)
    if not checks["request_coverage"]:
        issues.append("MISSING_REQUEST_TERMS_IN_QUERY")
    if not checks["subject_preservation"]:
        issues.append("MISSING_SUBJECT_CONSTRAINT")
    if not checks["standalone_subqueries"]:
        issues.append("NON_STANDALONE_SUBQUERY")
    if not checks["atomic_subqueries"]:
        issues.append("NON_ATOMIC_SUBQUERY")
    issues = ordered_unique(issues)
    accepted = bool(payload.get("decomposable") is True) and not issues and 2 <= len(queries) <= config.max_subqueries
    return {
        "status": "COMPLETE" if accepted else "FAILED",
        "accepted": accepted,
        "subqueries": queries if accepted else [],
        "candidate_subqueries": queries,
        "confidence": float(payload.get("confidence") or 0.0),
        "reason": str(payload.get("reason") or "").strip(),
        "issues": issues,
        "checks": checks,
    }


class HCXQualityCaller:
    def __init__(
        self,
        api_key: str,
        *,
        config: QualityConfig | None = None,
        cache_path: str | Path | None = None,
        session: Any | None = None,
    ) -> None:
        key = str(api_key or "").strip()
        if not key or key.lower().startswith("bearer ") or any(char.isspace() for char in key):
            raise ValueError("HCX_API_KEY 형식을 확인하세요.")
        self.api_key = key
        self.config = config or QualityConfig()
        self.cache_path = Path(cache_path) if cache_path else None
        if session is None and requests is None:
            raise RuntimeError("HCX 호출에는 requests 패키지가 필요합니다.")
        self.session = session or requests.Session()
        self.cache: dict[str, dict[str, Any]] = {}
        if self.cache_path and self.cache_path.exists():
            for line in self.cache_path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    row = json.loads(line)
                    self.cache[str(row["cache_key"])] = row

    def _append(self, row: Mapping[str, Any]) -> None:
        if not self.cache_path:
            return
        self.cache_path.parent.mkdir(parents=True, exist_ok=True)
        with self.cache_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(dict(row), ensure_ascii=False) + "\n")

    def _call(
        self,
        *,
        cache_payload: Mapping[str, Any],
        messages: Sequence[Mapping[str, str]],
        schema: Mapping[str, Any],
        validator: Callable[[Mapping[str, Any]], dict[str, Any]],
        prompt_version: str,
    ) -> dict[str, Any]:
        from kdic_lightweight_query_ablation_core import _extract_hcx_payload

        cache_key = hashlib.sha256(
            json.dumps(cache_payload, ensure_ascii=False, sort_keys=True).encode("utf-8")
        ).hexdigest()
        if cache_key in self.cache:
            row = dict(self.cache[cache_key])
            row["cache_hit"] = True
            row["actual_api_latency_ms"] = 0.0
            return row

        body = {
            "messages": list(messages),
            "topP": 0.1,
            "topK": 0,
            "maxCompletionTokens": 900,
            "temperature": 0.0,
            "repetitionPenalty": 1.0,
            "thinking": {"effort": "none"},
            "stop": [],
            "responseFormat": {"type": "json", "schema": dict(schema)},
        }
        last_error: Exception | None = None
        for attempt in range(self.config.transport_retries + 1):
            started = time.perf_counter()
            try:
                response = self.session.post(
                    self.config.llm_endpoint,
                    headers={
                        "Authorization": f"Bearer {self.api_key}",
                        "Content-Type": "application/json",
                        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
                    },
                    json=body,
                    timeout=self.config.llm_timeout_seconds,
                )
                response.raise_for_status()
                payload, usage = _extract_hcx_payload(response.json())
                validation = validator(payload)
                latency = (time.perf_counter() - started) * 1000
                row = {
                    "cache_key": cache_key,
                    "model": self.config.llm_model,
                    "prompt_version": prompt_version,
                    "raw_payload": payload,
                    **validation,
                    **usage,
                    "effective_api_latency_ms": round(latency, 3),
                    "actual_api_latency_ms": round(latency, 3),
                    "cache_hit": False,
                    "transport_attempts": attempt + 1,
                    "error_type": "",
                    "error_message": "",
                }
                self.cache[cache_key] = row
                self._append(row)
                if self.config.request_delay_seconds:
                    time.sleep(self.config.request_delay_seconds)
                return dict(row)
            except Exception as error:
                last_error = error
                if attempt < self.config.transport_retries:
                    time.sleep(min(2 ** attempt, 4))

        row = {
            "cache_key": cache_key,
            "model": self.config.llm_model,
            "prompt_version": prompt_version,
            "raw_payload": {},
            "status": "ERROR",
            "accepted": False,
            "subqueries": [],
            "candidate_subqueries": [],
            "confidence": 0.0,
            "reason": "",
            "issues": ["LLM_REQUEST_FAILED"],
            "checks": {},
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0,
            "effective_api_latency_ms": 0.0,
            "actual_api_latency_ms": 0.0,
            "cache_hit": False,
            "transport_attempts": self.config.transport_retries + 1,
            "error_type": type(last_error).__name__ if last_error else "UnknownError",
            "error_message": str(last_error or "unknown error"),
        }
        self.cache[cache_key] = row
        self._append(row)
        return dict(row)

    def quality_first(self, question: str, expected_businesses: Sequence[str]) -> dict[str, Any]:
        return self._call(
            cache_payload={
                "prompt_version": PROMPT_VERSION,
                "model": self.config.llm_model,
                "question": question,
                "expected_businesses": list(expected_businesses),
            },
            messages=build_quality_messages(question, expected_businesses),
            schema=quality_json_schema(self.config.max_subqueries),
            validator=lambda payload: validate_quality_decomposition(
                question, payload, expected_businesses=expected_businesses, config=self.config
            ),
            prompt_version=PROMPT_VERSION,
        )

    def repair(
        self,
        question: str,
        expected_businesses: Sequence[str],
        first_record: Mapping[str, Any],
        *,
        quality_mode: bool,
    ) -> dict[str, Any]:
        issues = list(first_record.get("issues") or [])
        previous_payload = dict(first_record.get("raw_payload") or {})
        if quality_mode:
            schema = quality_json_schema(self.config.max_subqueries)
            validator = lambda payload: validate_quality_decomposition(
                question, payload, expected_businesses=expected_businesses, config=self.config
            )
        else:
            schema = baseline_json_schema(self.config.max_subqueries)
            validator = lambda payload: validate_baseline_decomposition(
                question, payload,
                expected_businesses=expected_businesses,
                config=self.config,
            )
        return self._call(
            cache_payload={
                "prompt_version": REPAIR_PROMPT_VERSION,
                "model": self.config.llm_model,
                "question": question,
                "expected_businesses": list(expected_businesses),
                "quality_mode": quality_mode,
                "issues": issues,
                "previous_payload": previous_payload,
            },
            messages=build_repair_messages(
                question, expected_businesses, previous_payload, issues, quality_mode=quality_mode
            ),
            schema=schema,
            validator=validator,
            prompt_version=REPAIR_PROMPT_VERSION,
        )


def normalize_baseline_record(
    question: str,
    expected_businesses: Sequence[str],
    record: Mapping[str, Any],
) -> dict[str, Any]:
    output = dict(record)
    output.setdefault("candidate_subqueries", extract_queries(record.get("raw_payload") or {}))
    output.setdefault("checks", content_checks(question, record.get("raw_payload") or {}, expected_businesses))
    return output


def _condition_record(
    condition: str,
    first: Mapping[str, Any],
    final: Mapping[str, Any],
    *,
    retry_called: bool,
) -> dict[str, Any]:
    first_latency = float(first.get("effective_api_latency_ms") or 0.0)
    retry_latency = float(final.get("effective_api_latency_ms") or 0.0) if retry_called else 0.0
    first_tokens = int(first.get("total_tokens") or 0)
    retry_tokens = int(final.get("total_tokens") or 0) if retry_called else 0
    return {
        "condition": condition,
        "condition_label": CONDITION_LABELS[condition],
        "first_status": first.get("status"),
        "first_accepted": bool(first.get("accepted")),
        "first_confidence": float(first.get("confidence") or 0.0),
        "first_issues": list(first.get("issues") or []),
        "first_candidate_subqueries": list(first.get("candidate_subqueries") or first.get("subqueries") or []),
        "first_checks": dict(first.get("checks") or {}),
        "retry_called": retry_called,
        "retry_count": int(retry_called),
        "retry_success": bool(retry_called and final.get("accepted")),
        "retry_status": final.get("status") if retry_called else "NOT_CALLED",
        "retry_issues": list(final.get("issues") or []) if retry_called else [],
        "final_status": final.get("status"),
        "final_accepted": bool(final.get("accepted")),
        "final_confidence": float(final.get("confidence") or 0.0),
        "final_issues": list(final.get("issues") or []),
        "final_subqueries": list(final.get("subqueries") or []),
        "final_candidate_subqueries": list(final.get("candidate_subqueries") or final.get("subqueries") or []),
        "final_checks": dict(final.get("checks") or {}),
        "fallback_to_original": not bool(final.get("accepted")),
        "analysis_api_latency_ms": first_latency + retry_latency,
        "prompt_tokens": int(first.get("prompt_tokens") or 0) + (int(final.get("prompt_tokens") or 0) if retry_called else 0),
        "completion_tokens": int(first.get("completion_tokens") or 0) + (int(final.get("completion_tokens") or 0) if retry_called else 0),
        "total_tokens": first_tokens + retry_tokens,
        "api_request_count": 1 + int(retry_called),
        "first_raw_payload": dict(first.get("raw_payload") or {}),
        "retry_raw_payload": dict(final.get("raw_payload") or {}) if retry_called else {},
    }


def should_semantic_retry(record: Mapping[str, Any]) -> bool:
    """검증 가능한 생성 오류만 한 번 교정하고, 모델의 분해 거절은 존중한다."""
    status = str(record.get("status") or "").upper()
    issues = set(record.get("issues") or [])
    if bool(record.get("accepted")):
        return False
    if status in {"DECLINED", "ERROR"}:
        return False
    if "LLM_DECLINED_DECOMPOSITION" in issues or "LLM_REQUEST_FAILED" in issues:
        return False
    return True


def run_candidate_conditions(
    question: str,
    expected_businesses: Sequence[str],
    *,
    baseline_decomposer: Any,
    quality_caller: HCXQualityCaller,
) -> list[dict[str, Any]]:
    baseline_first = normalize_baseline_record(
        question, expected_businesses,
        baseline_decomposer.decompose(question, expected_businesses),
    )
    quality_first = quality_caller.quality_first(question, expected_businesses)

    if not should_semantic_retry(baseline_first):
        baseline_final = baseline_first
        baseline_retry_called = False
    else:
        baseline_final = quality_caller.repair(
            question, expected_businesses, baseline_first, quality_mode=False
        )
        baseline_retry_called = True

    if not should_semantic_retry(quality_first):
        quality_final = quality_first
        quality_retry_called = False
    else:
        quality_final = quality_caller.repair(
            question, expected_businesses, quality_first, quality_mode=True
        )
        quality_retry_called = True

    return [
        _condition_record(BASELINE, baseline_first, baseline_first, retry_called=False),
        _condition_record(QUALITY, quality_first, quality_first, retry_called=False),
        _condition_record(RETRY, baseline_first, baseline_final, retry_called=baseline_retry_called),
        _condition_record(QUALITY_RETRY, quality_first, quality_final, retry_called=quality_retry_called),
    ]


def make_query_plans(original: str, subqueries: Sequence[str], config: QualityConfig) -> list[Any]:
    from kdic_integrated_eval_core import QueryPlan

    cleaned = ordered_unique(subqueries)
    compact_original = re.sub(r"\s+", "", original).lower()
    cleaned = [item for item in cleaned if re.sub(r"\s+", "", item).lower() != compact_original]
    if len(cleaned) < 2:
        return [QueryPlan(
            need_id="FUSED", variant_id="ORIGINAL", dense_query=original, bm25_query=original,
            filter_mode="NONE", business_filters=[], soft_business_hints=[], query_weight=1.0,
            query_source="ORIGINAL_FALLBACK",
        )]
    each = config.subquery_total_weight / len(cleaned)
    plans = [QueryPlan(
        need_id="FUSED", variant_id="ORIGINAL_ANCHOR", dense_query=original, bm25_query=original,
        filter_mode="NONE", business_filters=[], soft_business_hints=[],
        query_weight=config.original_weight, query_source="ORIGINAL_ANCHOR",
    )]
    plans.extend(QueryPlan(
        need_id="FUSED", variant_id=f"SUBQUERY_{index:02d}", dense_query=query, bm25_query=query,
        filter_mode="NONE", business_filters=[], soft_business_hints=[],
        query_weight=each, query_source="DECOMPOSED",
    ) for index, query in enumerate(cleaned, 1))
    return plans


def build_condition_case(
    common: Mapping[str, Any],
    condition_record: Mapping[str, Any] | None,
    condition: str,
    *,
    config: QualityConfig,
) -> Any:
    from kdic_integrated_eval_core import AnalyzerCase

    route = str(common["route"])
    original = str(common["original_question"])
    cross_candidate = bool(common.get("complex_candidate")) and len(
        ordered_unique((common.get("complexity") or {}).get("businesses") or [])
    ) >= 2
    record = dict(condition_record or {})
    accepted = bool(cross_candidate and record.get("final_accepted"))
    subqueries = list(record.get("final_subqueries") or []) if accepted else []
    plans = make_query_plans(original, subqueries, config) if route == "RETRIEVE" else []
    analysis_latency = float(common.get("common_latency_ms") or 0.0) + float(record.get("analysis_api_latency_ms") or 0.0)
    raw_result = {
        "pipeline_version": condition,
        "analysis_status": "OK",
        "original_query": original,
        "normalized_query": common.get("normalized_question"),
        "route_reasons": common.get("route_reasons") or [],
        "complex_candidate": bool(common.get("complex_candidate")),
        "cross_business_candidate": cross_candidate,
        "businesses": (common.get("complexity") or {}).get("businesses") or [],
        "decomposition_condition": condition,
        "decomposition_record": record,
        "decomposition_source": "LLM" if accepted else ("ORIGINAL_FALLBACK" if cross_candidate else "ORIGINAL_POLICY"),
        "final_subqueries": subqueries,
        "fusion_policy": "WEIGHTED_RRF",
        "original_weight": config.original_weight if accepted else 1.0,
        "subquery_total_weight": config.subquery_total_weight if accepted else 0.0,
        "runtime": {
            "api_request_count": int(record.get("api_request_count") or 0),
            "prompt_tokens": int(record.get("prompt_tokens") or 0),
            "completion_tokens": int(record.get("completion_tokens") or 0),
            "total_tokens": int(record.get("total_tokens") or 0),
            "latency_ms": analysis_latency,
        },
    }
    return AnalyzerCase(
        evaluation_id=str(common["evaluation_id"]), analyzer=condition,
        original_question=original, route=route,
        analysis_latency_ms=round(analysis_latency, 3), plans=plans, raw_result=raw_result,
    )


def summarize_decomposition(audit_df: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for condition in CONDITION_ORDER:
        frame = audit_df[audit_df["condition"].eq(condition)]
        candidates = frame[frame["cross_business_candidate"]]
        true_cross = candidates[candidates["gold_cross_business"]]
        boundary = candidates[~candidates["gold_cross_business"]]
        retries = candidates[candidates["retry_called"]]
        rows.append({
            "condition": condition,
            "condition_label": CONDITION_LABELS[condition],
            "all_question_count": len(frame),
            "predicted_cross_business_count": len(candidates),
            "gold_cross_business_count": int(frame["gold_cross_business"].sum()),
            "cross_business_true_positive_count": int((frame["cross_business_candidate"] & frame["gold_cross_business"]).sum()),
            "cross_business_false_positive_count": int((frame["cross_business_candidate"] & ~frame["gold_cross_business"]).sum()),
            "cross_business_false_negative_count": int((~frame["cross_business_candidate"] & frame["gold_cross_business"]).sum()),
            "first_accept_count": int(candidates["first_accepted"].sum()),
            "final_accept_count": int(candidates["final_accepted"].sum()),
            "true_cross_first_accept_rate": float(true_cross["first_accepted"].mean()) if len(true_cross) else math.nan,
            "true_cross_final_accept_rate": float(true_cross["final_accepted"].mean()) if len(true_cross) else math.nan,
            "boundary_wrong_accept_count": int(boundary["final_accepted"].sum()),
            "retry_call_count": int(candidates["retry_called"].sum()),
            "retry_success_count": int(candidates["retry_success"].sum()),
            "retry_success_rate": float(retries["retry_success"].mean()) if len(retries) else math.nan,
            "fallback_count": int(candidates["fallback_to_original"].sum()),
            "business_coverage_rate": float(candidates["check_business_coverage"].mean()),
            "request_coverage_rate": float(candidates["check_request_coverage"].mean()),
            "constraint_preservation_rate": float((
                candidates["check_numeric_preservation"]
                & candidates["check_negation_preservation"]
                & candidates["check_subject_preservation"]
            ).mean()),
            "standalone_rate": float(candidates["check_standalone_subqueries"].mean()),
            "atomic_rate": float(candidates["check_atomic_subqueries"].mean()),
            "analysis_latency_ms_mean_all": float(frame["analysis_latency_ms"].mean()),
            "analysis_latency_ms_p95_all": float(frame["analysis_latency_ms"].quantile(.95)),
            "analysis_latency_ms_mean_candidates": float(candidates["analysis_latency_ms"].mean()),
            "total_tokens": int(frame["total_tokens"].sum()),
        })
    return pd.DataFrame(rows)


def decomposition_hard_gates(audit_df: pd.DataFrame, eval_df: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for condition in CONDITION_ORDER:
        frame = audit_df[audit_df["condition"].eq(condition)]
        candidates = frame[frame["cross_business_candidate"]]
        boundary = candidates[~candidates["gold_cross_business"]]
        retry_failed = candidates[candidates["retry_called"] & ~candidates["retry_success"]]
        accepted = candidates[candidates["final_accepted"]]
        gates = [
            ("실행 성공률", float(frame["execution_success"].mean()), ">=0.995", float(frame["execution_success"].mean()) >= .995),
            ("검색 질의 생성 유효율", float(frame.loc[frame["route"].eq("RETRIEVE"), "query_plan_valid"].mean()), ">=0.99", float(frame.loc[frame["route"].eq("RETRIEVE"), "query_plan_valid"].mean()) >= .99),
            ("정상 질문의 잘못된 OOS", int(frame["false_oos"].sum()), "=0", int(frame["false_oos"].sum()) == 0),
            ("정상 질문의 잘못된 DIRECT", int(frame["false_direct"].sum()), "=0", int(frame["false_direct"].sum()) == 0),
            ("Hard Filter", int(frame["hard_filter_count"].sum()), "=0", int(frame["hard_filter_count"].sum()) == 0),
            ("비교차 업무 분해 승인", int(boundary["final_accepted"].sum()), "=0", int(boundary["final_accepted"].sum()) == 0),
            ("승인 결과 검증 오류", int(accepted["final_issues"].map(bool).sum()), "=0", int(accepted["final_issues"].map(bool).sum()) == 0),
            ("재시도 최대 1회 초과", int((candidates["retry_count"] > 1).sum()), "=0", int((candidates["retry_count"] > 1).sum()) == 0),
            ("재시도 실패 후 원문 fallback", float(retry_failed["fallback_to_original"].mean()) if len(retry_failed) else 1.0, "=1.0", bool(retry_failed["fallback_to_original"].all()) if len(retry_failed) else True),
            (
                "질의 가중치 합 오류",
                int((
                    frame.loc[frame["route"].eq("RETRIEVE"), "query_plan_weight_sum"]
                    .sub(1.0).abs() > 1e-9
                ).sum()),
                "=0",
                int((
                    frame.loc[frame["route"].eq("RETRIEVE"), "query_plan_weight_sum"]
                    .sub(1.0).abs() > 1e-9
                ).sum()) == 0,
            ),
        ]
        for gate, value, threshold, passed in gates:
            rows.append({
                "condition": condition,
                "condition_label": CONDITION_LABELS[condition],
                "gate": gate,
                "value": value,
                "threshold": threshold,
                "passed": bool(passed),
            })
    return pd.DataFrame(rows)


def serialize_nested(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()
    for column in output.columns:
        if output[column].map(lambda value: isinstance(value, (list, dict, tuple))).any():
            output[column] = output[column].map(
                lambda value: json.dumps(value, ensure_ascii=False)
                if isinstance(value, (list, dict, tuple)) else value
            )
    return output


In [ ]:
%%writefile kdic_hcx007_resumable_decomposition_core.py
from __future__ import annotations

import email.utils
import hashlib
import json
import random
import time
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Mapping, Sequence

import requests

import kdic_decomposition_quality_core as quality_core
from kdic_decomposition_quality_core import (
    BASELINE,
    CONDITION_LABELS,
    QUALITY,
    QUALITY_RETRY,
    RETRY,
    QualityConfig,
    baseline_json_schema,
    build_quality_messages,
    build_repair_messages,
    content_checks,
    extract_queries,
    quality_json_schema,
    should_semantic_retry,
    validate_baseline_decomposition,
    validate_quality_decomposition,
)
from kdic_lightweight_query_ablation_core import (
    AblationConfig,
    DECOMPOSITION_PROMPT_VERSION,
    _extract_hcx_payload,
    build_decomposition_messages,
    decomposition_json_schema,
    validate_llm_decomposition,
)


HCX_DECOMPOSITION_MODEL = "HCX-007"
HCX_DECOMPOSITION_ENDPOINT = (
    "https://clovastudio.stream.ntruss.com/v3/chat-completions/HCX-007"
)
REDESIGN_PROMPT_VERSION = "KDIC_DECOMPOSITION_QUALITY_V2_HCX007_2026_08_14"
REDESIGN_REPAIR_PROMPT_VERSION = "KDIC_DECOMPOSITION_REPAIR_V2_HCX007_2026_08_14"


@dataclass(frozen=True)
class TransportPolicy:
    request_delay_seconds: float = 1.5
    max_transport_retries: int = 5
    base_backoff_seconds: float = 2.0
    max_backoff_seconds: float = 32.0
    jitter_seconds: float = 0.5
    consecutive_429_cooldown_threshold: int = 3
    cooldown_seconds: float = 60.0
    timeout_seconds: float = 120.0


def _valid_api_key(value: str) -> str:
    key = str(value or "").strip()
    if not key or key.lower().startswith("bearer ") or any(ch.isspace() for ch in key):
        raise ValueError("HCX_API_KEY에는 Bearer 접두사나 공백을 넣지 않습니다.")
    return key


def _cache_key(payload: Mapping[str, Any]) -> str:
    raw = json.dumps(dict(payload), ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


class ValidOnlyJsonlCache:
    """정상 응답만 재사용하고 ERROR 행은 감사 기록으로만 남긴다."""

    def __init__(
        self,
        active_path: str | Path,
        *,
        seed_paths: Sequence[str | Path] = (),
    ) -> None:
        self.active_path = Path(active_path)
        self.rows: dict[str, dict[str, Any]] = {}
        self.origin_by_key: dict[str, str] = {}
        for source in [*map(Path, seed_paths), self.active_path]:
            if not source.is_file():
                continue
            for line in source.read_text(encoding="utf-8").splitlines():
                if not line.strip():
                    continue
                row = json.loads(line)
                key = str(row.get("cache_key") or "")
                if not key or str(row.get("status") or "").upper() == "ERROR":
                    continue
                self.rows[key] = row
                self.origin_by_key[key] = str(source)

    def get(self, key: str) -> dict[str, Any] | None:
        if key not in self.rows:
            return None
        row = dict(self.rows[key])
        row["cache_hit"] = True
        row["actual_api_latency_ms"] = 0.0
        row["cache_origin"] = self.origin_by_key.get(key, "")
        if Path(self.origin_by_key.get(key, "")) != self.active_path:
            promoted = dict(row)
            promoted["promoted_from_seed_cache"] = True
            self.append(promoted, reusable=True)
        return row

    def append(self, row: Mapping[str, Any], *, reusable: bool) -> None:
        payload = dict(row)
        self.active_path.parent.mkdir(parents=True, exist_ok=True)
        with self.active_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(payload, ensure_ascii=False) + "\n")
        key = str(payload.get("cache_key") or "")
        if reusable and key:
            self.rows[key] = payload
            self.origin_by_key[key] = str(self.active_path)


def _retry_after_seconds(response: requests.Response) -> float | None:
    raw = str(response.headers.get("Retry-After") or "").strip()
    if not raw:
        return None
    try:
        return max(0.0, float(raw))
    except ValueError:
        try:
            parsed = email.utils.parsedate_to_datetime(raw)
            if parsed.tzinfo is None:
                parsed = parsed.replace(tzinfo=timezone.utc)
            return max(0.0, (parsed - datetime.now(timezone.utc)).total_seconds())
        except Exception:
            return None


class RobustHCXTransport:
    def __init__(
        self,
        api_key: str,
        *,
        endpoint: str = HCX_DECOMPOSITION_ENDPOINT,
        policy: TransportPolicy | None = None,
        session: requests.Session | None = None,
        sleep_fn: Callable[[float], None] = time.sleep,
        monotonic_fn: Callable[[], float] = time.monotonic,
        random_fn: Callable[[], float] = random.random,
    ) -> None:
        self.api_key = _valid_api_key(api_key)
        self.endpoint = endpoint
        self.policy = policy or TransportPolicy()
        self.session = session or requests.Session()
        self.sleep_fn = sleep_fn
        self.monotonic_fn = monotonic_fn
        self.random_fn = random_fn
        self.last_request_at: float | None = None
        self.consecutive_429 = 0

    def _pace(self) -> float:
        if self.last_request_at is None:
            return 0.0
        remaining = self.policy.request_delay_seconds - (
            self.monotonic_fn() - self.last_request_at
        )
        if remaining > 0:
            self.sleep_fn(remaining)
            return remaining
        return 0.0

    def post_json(self, body: Mapping[str, Any]) -> dict[str, Any]:
        logical_started = self.monotonic_fn()
        total_sleep_seconds = 0.0
        service_latency_ms = 0.0
        attempts = 0
        last_status: int | None = None
        last_error_type = ""
        last_error_message = ""
        last_response_body = ""

        for retry_index in range(self.policy.max_transport_retries + 1):
            total_sleep_seconds += self._pace()
            attempts += 1
            request_started = self.monotonic_fn()
            try:
                response = self.session.post(
                    self.endpoint,
                    headers={
                        "Authorization": f"Bearer {self.api_key}",
                        "Content-Type": "application/json",
                        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
                    },
                    json=dict(body),
                    timeout=self.policy.timeout_seconds,
                )
                self.last_request_at = self.monotonic_fn()
                service_latency_ms += (self.last_request_at - request_started) * 1000
                last_status = int(response.status_code)
                last_response_body = str(response.text or "")[:8000]

                if 200 <= response.status_code < 300:
                    self.consecutive_429 = 0
                    payload, usage = _extract_hcx_payload(response.json())
                    return {
                        "transport_ok": True,
                        "payload": payload,
                        "usage": usage,
                        "http_status": last_status,
                        "transport_attempts": attempts,
                        "service_latency_ms": round(service_latency_ms, 3),
                        "transport_sleep_ms": round(total_sleep_seconds * 1000, 3),
                        "actual_api_latency_ms": round(
                            (self.monotonic_fn() - logical_started) * 1000, 3
                        ),
                        "error_type": "",
                        "error_message": "",
                        "error_response_body": "",
                    }

                last_error_type = "HTTP_ERROR"
                last_error_message = f"HTTP {response.status_code}"
                retryable = response.status_code == 429 or response.status_code >= 500
                if response.status_code == 429:
                    self.consecutive_429 += 1
                else:
                    self.consecutive_429 = 0
                if not retryable or retry_index >= self.policy.max_transport_retries:
                    break

                if (
                    response.status_code == 429
                    and self.consecutive_429
                    >= self.policy.consecutive_429_cooldown_threshold
                ):
                    wait_seconds = self.policy.cooldown_seconds
                    self.consecutive_429 = 0
                else:
                    retry_after = _retry_after_seconds(response)
                    exponential = min(
                        self.policy.max_backoff_seconds,
                        self.policy.base_backoff_seconds * (2**retry_index),
                    )
                    wait_seconds = retry_after if retry_after is not None else exponential
                    wait_seconds += self.random_fn() * self.policy.jitter_seconds
                self.sleep_fn(wait_seconds)
                total_sleep_seconds += wait_seconds
            except Exception as error:
                self.last_request_at = self.monotonic_fn()
                service_latency_ms += (self.last_request_at - request_started) * 1000
                last_error_type = type(error).__name__
                last_error_message = str(error)
                if retry_index >= self.policy.max_transport_retries:
                    break
                wait_seconds = min(
                    self.policy.max_backoff_seconds,
                    self.policy.base_backoff_seconds * (2**retry_index),
                ) + self.random_fn() * self.policy.jitter_seconds
                self.sleep_fn(wait_seconds)
                total_sleep_seconds += wait_seconds

        return {
            "transport_ok": False,
            "payload": {},
            "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0},
            "http_status": last_status,
            "transport_attempts": attempts,
            "service_latency_ms": round(service_latency_ms, 3),
            "transport_sleep_ms": round(total_sleep_seconds * 1000, 3),
            "actual_api_latency_ms": round(
                (self.monotonic_fn() - logical_started) * 1000, 3
            ),
            "error_type": last_error_type or "UNKNOWN_TRANSPORT_ERROR",
            "error_message": last_error_message or "unknown transport error",
            "error_response_body": last_response_body,
        }


def _error_row(
    cache_key: str,
    *,
    question: str,
    model: str,
    prompt_version: str,
    transport: Mapping[str, Any],
) -> dict[str, Any]:
    return {
        "cache_key": cache_key,
        "question": question,
        "model": model,
        "prompt_version": prompt_version,
        "raw_payload": {},
        "status": "ERROR",
        "accepted": False,
        "subqueries": [],
        "candidate_subqueries": [],
        "confidence": 0.0,
        "reason": "",
        "issues": ["LLM_REQUEST_FAILED"],
        "checks": {},
        **dict(transport.get("usage") or {}),
        "effective_api_latency_ms": float(transport.get("actual_api_latency_ms") or 0.0),
        "cache_hit": False,
        **{
            key: transport.get(key)
            for key in (
                "http_status", "transport_attempts", "service_latency_ms",
                "transport_sleep_ms", "actual_api_latency_ms", "error_type",
                "error_message", "error_response_body",
            )
        },
    }


class ResumableBaselineDecomposer:
    def __init__(
        self,
        api_key: str,
        *,
        cache_path: str | Path,
        seed_cache_paths: Sequence[str | Path] = (),
        config: AblationConfig | None = None,
        transport_policy: TransportPolicy | None = None,
        session: requests.Session | None = None,
    ) -> None:
        self.config = config or AblationConfig(
            llm_endpoint=HCX_DECOMPOSITION_ENDPOINT,
            llm_model=HCX_DECOMPOSITION_MODEL,
        )
        if self.config.llm_model != HCX_DECOMPOSITION_MODEL:
            raise ValueError("Baseline 구조화 분해 모델은 HCX-007이어야 합니다.")
        self.cache = ValidOnlyJsonlCache(cache_path, seed_paths=seed_cache_paths)
        self.transport = RobustHCXTransport(
            api_key, endpoint=HCX_DECOMPOSITION_ENDPOINT,
            policy=transport_policy, session=session,
        )

    def _key(self, question: str, expected_businesses: Sequence[str]) -> str:
        return _cache_key({
            "prompt_version": DECOMPOSITION_PROMPT_VERSION,
            "model": self.config.llm_model,
            "question": question,
            "expected_businesses": list(expected_businesses),
            "min_confidence": self.config.llm_min_confidence,
        })

    def decompose(self, question: str, expected_businesses: Sequence[str]) -> dict[str, Any]:
        key = self._key(question, expected_businesses)
        cached = self.cache.get(key)
        if cached is not None:
            return cached
        body = {
            "messages": build_decomposition_messages(question),
            "topP": 0.1,
            "topK": 0,
            "maxCompletionTokens": 700,
            "temperature": 0.0,
            "repetitionPenalty": 1.0,
            "thinking": {"effort": "none"},
            "stop": [],
            "responseFormat": {
                "type": "json",
                "schema": decomposition_json_schema(self.config.max_subqueries),
            },
        }
        transport = self.transport.post_json(body)
        if not transport["transport_ok"]:
            row = _error_row(
                key, question=question, model=self.config.llm_model,
                prompt_version=DECOMPOSITION_PROMPT_VERSION, transport=transport,
            )
            self.cache.append(row, reusable=False)
            return row
        validation = validate_llm_decomposition(
            question,
            transport["payload"],
            expected_businesses=expected_businesses,
            config=self.config,
        )
        row = {
            "cache_key": key,
            "question": question,
            "model": self.config.llm_model,
            "prompt_version": DECOMPOSITION_PROMPT_VERSION,
            "raw_payload": transport["payload"],
            **validation,
            **transport["usage"],
            "effective_api_latency_ms": transport["actual_api_latency_ms"],
            "cache_hit": False,
            **{
                field: transport.get(field)
                for field in (
                    "http_status", "transport_attempts", "service_latency_ms",
                    "transport_sleep_ms", "actual_api_latency_ms", "error_type",
                    "error_message", "error_response_body",
                )
            },
        }
        self.cache.append(row, reusable=True)
        return row


class ResumableQualityCaller:
    def __init__(
        self,
        api_key: str,
        *,
        cache_path: str | Path,
        config: QualityConfig,
        transport_policy: TransportPolicy | None = None,
        session: requests.Session | None = None,
    ) -> None:
        if config.llm_model != HCX_DECOMPOSITION_MODEL:
            raise ValueError("개선 구조화 분해 모델은 HCX-007이어야 합니다.")
        self.config = config
        self.cache = ValidOnlyJsonlCache(cache_path)
        self.transport = RobustHCXTransport(
            api_key, endpoint=HCX_DECOMPOSITION_ENDPOINT,
            policy=transport_policy, session=session,
        )

    def _call(
        self,
        *,
        key_payload: Mapping[str, Any],
        question: str,
        prompt_version: str,
        messages: Sequence[Mapping[str, str]],
        schema: Mapping[str, Any],
        validator: Callable[[Mapping[str, Any]], dict[str, Any]],
    ) -> dict[str, Any]:
        key = _cache_key(key_payload)
        cached = self.cache.get(key)
        if cached is not None:
            return cached
        body = {
            "messages": list(messages),
            "topP": 0.1,
            "topK": 0,
            "maxCompletionTokens": 900,
            "temperature": 0.0,
            "repetitionPenalty": 1.0,
            "thinking": {"effort": "none"},
            "stop": [],
            "responseFormat": {"type": "json", "schema": dict(schema)},
        }
        transport = self.transport.post_json(body)
        if not transport["transport_ok"]:
            row = _error_row(
                key, question=question, model=self.config.llm_model,
                prompt_version=prompt_version, transport=transport,
            )
            self.cache.append(row, reusable=False)
            return row
        validation = validator(transport["payload"])
        row = {
            "cache_key": key,
            "question": question,
            "model": self.config.llm_model,
            "prompt_version": prompt_version,
            "raw_payload": transport["payload"],
            **validation,
            **transport["usage"],
            "effective_api_latency_ms": transport["actual_api_latency_ms"],
            "cache_hit": False,
            **{
                field: transport.get(field)
                for field in (
                    "http_status", "transport_attempts", "service_latency_ms",
                    "transport_sleep_ms", "actual_api_latency_ms", "error_type",
                    "error_message", "error_response_body",
                )
            },
        }
        self.cache.append(row, reusable=True)
        return row

    def quality_first(self, question: str, expected_businesses: Sequence[str]) -> dict[str, Any]:
        return self._call(
            key_payload={
                "prompt_version": REDESIGN_PROMPT_VERSION,
                "model": self.config.llm_model,
                "question": question,
                "expected_businesses": list(expected_businesses),
            },
            question=question,
            prompt_version=REDESIGN_PROMPT_VERSION,
            messages=build_quality_messages(question, expected_businesses),
            schema=quality_json_schema(self.config.max_subqueries),
            validator=lambda payload: validate_quality_decomposition(
                question, payload,
                expected_businesses=expected_businesses,
                config=self.config,
            ),
        )

    def repair(
        self,
        question: str,
        expected_businesses: Sequence[str],
        first_record: Mapping[str, Any],
        *,
        quality_mode: bool,
    ) -> dict[str, Any]:
        issues = list(first_record.get("issues") or [])
        previous_payload = dict(first_record.get("raw_payload") or {})
        if quality_mode:
            schema = quality_json_schema(self.config.max_subqueries)
            validator = lambda payload: validate_quality_decomposition(
                question, payload,
                expected_businesses=expected_businesses,
                config=self.config,
            )
        else:
            schema = baseline_json_schema(self.config.max_subqueries)
            validator = lambda payload: validate_baseline_decomposition(
                question, payload,
                expected_businesses=expected_businesses,
                config=self.config,
            )
        return self._call(
            key_payload={
                "prompt_version": REDESIGN_REPAIR_PROMPT_VERSION,
                "model": self.config.llm_model,
                "question": question,
                "expected_businesses": list(expected_businesses),
                "quality_mode": quality_mode,
                "issues": issues,
                "previous_payload": previous_payload,
            },
            question=question,
            prompt_version=REDESIGN_REPAIR_PROMPT_VERSION,
            messages=build_repair_messages(
                question, expected_businesses, previous_payload, issues,
                quality_mode=quality_mode,
            ),
            schema=schema,
            validator=validator,
        )


def _normalize_baseline_record(
    question: str,
    expected_businesses: Sequence[str],
    record: Mapping[str, Any],
) -> dict[str, Any]:
    output = dict(record)
    output.setdefault(
        "candidate_subqueries",
        extract_queries(record.get("raw_payload") or {}),
    )
    output.setdefault(
        "checks",
        content_checks(question, record.get("raw_payload") or {}, expected_businesses),
    )
    return output


def _condition_record(
    condition: str,
    first: Mapping[str, Any],
    final: Mapping[str, Any],
    *,
    retry_called: bool,
) -> dict[str, Any]:
    final_record = dict(final)
    first_latency = float(first.get("effective_api_latency_ms") or 0.0)
    retry_latency = float(final.get("effective_api_latency_ms") or 0.0) if retry_called else 0.0
    return {
        "condition": condition,
        "condition_label": CONDITION_LABELS[condition],
        "first_status": first.get("status"),
        "first_accepted": bool(first.get("accepted")),
        "first_confidence": float(first.get("confidence") or 0.0),
        "first_issues": list(first.get("issues") or []),
        "first_candidate_subqueries": list(
            first.get("candidate_subqueries") or first.get("subqueries") or []
        ),
        "first_checks": dict(first.get("checks") or {}),
        "first_http_status": first.get("http_status"),
        "first_error_type": first.get("error_type") or "",
        "first_error_message": first.get("error_message") or "",
        "first_error_response_body": first.get("error_response_body") or "",
        "first_transport_attempts": int(first.get("transport_attempts") or 0),
        "first_cache_hit": bool(first.get("cache_hit")),
        "first_cache_origin": first.get("cache_origin") or "",
        "retry_called": retry_called,
        "retry_count": int(retry_called),
        "retry_success": bool(retry_called and final.get("accepted")),
        "retry_status": final.get("status") if retry_called else "NOT_CALLED",
        "retry_issues": list(final.get("issues") or []) if retry_called else [],
        "retry_http_status": final.get("http_status") if retry_called else None,
        "retry_error_type": final.get("error_type") if retry_called else "",
        "retry_transport_attempts": int(final.get("transport_attempts") or 0) if retry_called else 0,
        "final_status": final.get("status"),
        "final_accepted": bool(final.get("accepted")),
        "final_confidence": float(final.get("confidence") or 0.0),
        "final_issues": list(final.get("issues") or []),
        "final_subqueries": list(final.get("subqueries") or []),
        "final_candidate_subqueries": list(
            final.get("candidate_subqueries") or final.get("subqueries") or []
        ),
        "final_checks": dict(final.get("checks") or {}),
        "fallback_to_original": not bool(final.get("accepted")),
        "analysis_api_latency_ms": first_latency + retry_latency,
        "actual_api_latency_ms": float(first.get("actual_api_latency_ms") or 0.0)
        + (float(final.get("actual_api_latency_ms") or 0.0) if retry_called else 0.0),
        "prompt_tokens": int(first.get("prompt_tokens") or 0)
        + (int(final.get("prompt_tokens") or 0) if retry_called else 0),
        "completion_tokens": int(first.get("completion_tokens") or 0)
        + (int(final.get("completion_tokens") or 0) if retry_called else 0),
        "total_tokens": int(first.get("total_tokens") or 0)
        + (int(final.get("total_tokens") or 0) if retry_called else 0),
        "logical_api_request_count": 1 + int(retry_called),
        # 기존 build_condition_case가 읽는 호환 필드입니다. 의미는 HTTP 재시도
        # 횟수가 아니라 첫 구조화 호출 + 선택적 의미 교정 호출 수입니다.
        "api_request_count": 1 + int(retry_called),
        "transport_attempt_count": int(first.get("transport_attempts") or 0)
        + (int(final.get("transport_attempts") or 0) if retry_called else 0),
        "first_raw_payload": dict(first.get("raw_payload") or {}),
        "retry_raw_payload": dict(final_record.get("raw_payload") or {}) if retry_called else {},
    }


def run_resumable_candidate_conditions(
    question: str,
    expected_businesses: Sequence[str],
    *,
    baseline_decomposer: ResumableBaselineDecomposer,
    quality_caller: ResumableQualityCaller,
) -> list[dict[str, Any]]:
    baseline_first = _normalize_baseline_record(
        question,
        expected_businesses,
        baseline_decomposer.decompose(question, expected_businesses),
    )
    quality_first = quality_caller.quality_first(question, expected_businesses)

    if should_semantic_retry(baseline_first):
        baseline_final = quality_caller.repair(
            question, expected_businesses, baseline_first, quality_mode=False
        )
        baseline_retry_called = True
    else:
        baseline_final = baseline_first
        baseline_retry_called = False

    if should_semantic_retry(quality_first):
        quality_final = quality_caller.repair(
            question, expected_businesses, quality_first, quality_mode=True
        )
        quality_retry_called = True
    else:
        quality_final = quality_first
        quality_retry_called = False

    return [
        _condition_record(BASELINE, baseline_first, baseline_first, retry_called=False),
        _condition_record(QUALITY, quality_first, quality_first, retry_called=False),
        _condition_record(RETRY, baseline_first, baseline_final, retry_called=baseline_retry_called),
        _condition_record(
            QUALITY_RETRY, quality_first, quality_final,
            retry_called=quality_retry_called,
        ),
    ]


def component_gate_rows(audit_df: Any) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for condition, frame in audit_df.groupby("condition", sort=False):
        candidates = frame[frame["cross_business_candidate"].astype(bool)]
        first_success = candidates["first_status"].ne("ERROR")
        http_400_count = int(candidates["first_http_status"].eq(400).sum())
        final_error_count = int(candidates["final_status"].eq("ERROR").sum())
        error_latency_missing = int((
            candidates["first_status"].eq("ERROR")
            & candidates["actual_api_latency_ms"].le(0)
        ).sum())
        checks = [
            ("structured_call_success_rate", float(first_success.mean()), 0.995, float(first_success.mean()) >= .995),
            ("candidate_evaluable_count", int(first_success.sum()), len(candidates), int(first_success.sum()) == len(candidates)),
            ("http_400_count", http_400_count, 0, http_400_count == 0),
            ("final_transport_error_count", final_error_count, 0, final_error_count == 0),
            ("error_latency_missing_count", error_latency_missing, 0, error_latency_missing == 0),
            ("retry_limit_exceeded_count", int((candidates["retry_count"] > 1).sum()), 0, int((candidates["retry_count"] > 1).sum()) == 0),
        ]
        for gate, value, threshold, passed in checks:
            rows.append({
                "condition": condition,
                "condition_label": CONDITION_LABELS.get(condition, condition),
                "gate": gate,
                "value": value,
                "threshold": threshold,
                "passed": bool(passed),
            })
    return rows


## 3. M3 Hybrid 7:3 Min-Max + Reranker 검색

In [ ]:
def _normalize_vector(vector: np.ndarray) -> np.ndarray:
    vector = np.asarray(vector, dtype=np.float32)
    norm = float(np.linalg.norm(vector))
    if norm == 0.0:
        raise RuntimeError("질문 임베딩이 영벡터입니다.")
    return vector / norm


def dense_search_by_vector(query_vector: np.ndarray, depth: int = CANDIDATE_DEPTH) -> list[dict[str, Any]]:
    if depth < 1:
        raise ValueError("depth는 1 이상이어야 합니다.")
    response = ES.search(
        index=ES_INDEX_NAME,
        knn={
            "field": "embedding",
            "query_vector": query_vector.tolist(),
            "k": depth,
            "num_candidates": max(depth * 10, 100),
        },
        size=depth,
    )
    results = []
    for rank, hit in enumerate(response["hits"]["hits"], start=1):
        chunk_id = str(hit["_source"]["chunk_id"])
        if chunk_id not in CHUNKS_BY_ID:
            raise RuntimeError(f"Dense 결과의 chunk_id가 원본에 없습니다: {chunk_id}")
        results.append({"chunk_id": chunk_id, "score": float(hit["_score"]), "rank": rank})
    return results

def dense_search(question: str, depth: int = CANDIDATE_DEPTH) -> list[dict[str, Any]]:
    if depth < 1:
        raise ValueError("depth는 1 이상이어야 합니다.")
    query_vector = _normalize_vector(embed_hcx_single(question))
    if query_vector.shape != (DENSE_DIMENSION,):
        raise RuntimeError(f"질문 임베딩 차원 불일치: query={query_vector.shape}, stored={DENSE_DIMENSION}")
    return dense_search_by_vector(query_vector, depth)


def bm25_search(question: str, depth: int = CANDIDATE_DEPTH) -> list[dict[str, Any]]:
    if depth < 1:
        raise ValueError("depth는 1 이상이어야 합니다.")
    response = ES.search(
        index=ES_INDEX_NAME,
        size=depth,
        query={"match": {"search_text": {"query": question}}},
    )
    results = []
    for rank, hit in enumerate(response["hits"]["hits"], start=1):
        chunk_id = str(hit["_source"]["chunk_id"])
        if chunk_id not in CHUNKS_BY_ID:
            raise RuntimeError(f"BM25 결과의 chunk_id가 원본에 없습니다: {chunk_id}")
        results.append({
            "chunk_id": chunk_id,
            "score": float(hit["_score"]),
            "rank": rank,
        })
    return results


def _minmax_by_chunk(results: list[dict[str, Any]]) -> dict[str, float]:
    if not results:
        return {}
    scores = np.asarray([float(row["score"]) for row in results], dtype=np.float64)
    low, high = float(scores.min()), float(scores.max())
    if abs(high - low) <= 1e-12:
        normalized = np.ones_like(scores)
    else:
        normalized = (scores - low) / (high - low)
    return {
        str(row["chunk_id"]): float(score)
        for row, score in zip(results, normalized)
    }


def weighted_minmax(
    dense_results: list[dict[str, Any]],
    bm25_results: list[dict[str, Any]],
    *,
    dense_weight: float = DENSE_WEIGHT,
    bm25_weight: float = BM25_WEIGHT,
    top_k: int = FINAL_TOP_K,
) -> list[dict[str, Any]]:
    if not math.isclose(dense_weight + bm25_weight, 1.0):
        raise ValueError("Dense/BM25 가중치 합은 1이어야 합니다.")
    dense_norm = _minmax_by_chunk(dense_results)
    bm25_norm = _minmax_by_chunk(bm25_results)
    dense_by_id = {str(row["chunk_id"]): row for row in dense_results}
    bm25_by_id = {str(row["chunk_id"]): row for row in bm25_results}
    candidates = []
    for chunk_id in sorted(set(dense_norm) | set(bm25_norm)):
        dense_row = dense_by_id.get(chunk_id)
        bm25_row = bm25_by_id.get(chunk_id)
        score = dense_weight * dense_norm.get(chunk_id, 0.0) + bm25_weight * bm25_norm.get(chunk_id, 0.0)
        candidates.append({
            "chunk_id": chunk_id,
            "minmax_score": float(score),
            "dense_rank": dense_row.get("rank") if dense_row else None,
            "dense_score": dense_row.get("score") if dense_row else None,
            "bm25_rank": bm25_row.get("rank") if bm25_row else None,
            "bm25_score": bm25_row.get("score") if bm25_row else None,
        })
    infinity = float("inf")
    ordered = sorted(candidates, key=lambda row: (
        -row["minmax_score"],
        row["dense_rank"] or infinity,
        row["bm25_rank"] or infinity,
        row["chunk_id"],
    ))
    return [
        {**row, "rank": rank, "chunk": CHUNKS_BY_ID[row["chunk_id"]]}
        for rank, row in enumerate(ordered[:top_k], start=1)
    ]


def hybrid_minmax_search(question: str, *, top_k: int = FINAL_TOP_K) -> list[dict[str, Any]]:
    cleaned = _clean_text(question)
    if not cleaned:
        raise ValueError("검색 질문이 비어 있습니다.")
    dense_results = dense_search(cleaned, CANDIDATE_DEPTH)
    bm25_results = bm25_search(cleaned, CANDIDATE_DEPTH)
    results = weighted_minmax(
        dense_results,
        bm25_results,
        dense_weight=DENSE_WEIGHT,
        bm25_weight=BM25_WEIGHT,
        top_k=top_k,
    )
    if not results:
        raise RuntimeError("Hybrid Min-Max 검색 결과가 없습니다.")
    return results


def fuse_query_results(
    plans: list[dict[str, Any]],
    *,
    top_k: int = FINAL_TOP_K,
    rrf_k: int = QUERY_FUSION_RRF_K,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    if not plans:
        raise ValueError("검색 계획이 없습니다.")
    if not math.isclose(sum(float(plan["weight"]) for plan in plans), 1.0, abs_tol=1e-9):
        raise ValueError("검색 계획 가중치 합은 1이어야 합니다.")
    per_query = []
    fused: dict[str, dict[str, Any]] = {}
    for plan_index, plan in enumerate(plans, start=1):
        started = time.perf_counter()
        hits = hybrid_minmax_search(str(plan["query"]), top_k=CANDIDATE_DEPTH)
        elapsed_ms = (time.perf_counter() - started) * 1000
        per_query.append({**plan, "latency_ms": elapsed_ms, "hits": hits})
        for hit in hits:
            chunk_id = str(hit["chunk_id"])
            row = fused.setdefault(chunk_id, {
                "chunk_id": chunk_id,
                "query_fusion_score": 0.0,
                "best_minmax_score": 0.0,
                "dense_rank": None,
                "bm25_rank": None,
                "matched_queries": [],
                "chunk": hit["chunk"],
            })
            row["query_fusion_score"] += float(plan["weight"]) / (rrf_k + int(hit["rank"]))
            row["best_minmax_score"] = max(row["best_minmax_score"], float(hit["minmax_score"]))
            if row["dense_rank"] is None or (hit["dense_rank"] is not None and hit["dense_rank"] < row["dense_rank"]):
                row["dense_rank"] = hit["dense_rank"]
            if row["bm25_rank"] is None or (hit["bm25_rank"] is not None and hit["bm25_rank"] < row["bm25_rank"]):
                row["bm25_rank"] = hit["bm25_rank"]
            row["matched_queries"].append({
                "plan_index": plan_index,
                "query": plan["query"],
                "source": plan["source"],
                "rank": hit["rank"],
                "weight": plan["weight"],
            })
    ordered = sorted(fused.values(), key=lambda row: (
        -row["query_fusion_score"], -row["best_minmax_score"], row["chunk_id"]
    ))
    final = []
    for rank, row in enumerate(ordered[:top_k], start=1):
        final.append({
            **row,
            "rank": rank,
            "minmax_score": row["best_minmax_score"],
        })
    return final, per_query


print("M3 Hybrid 7:3 Min-Max + Reranker 검색기 준비 완료")

### 3-1. 질문별 검색 세부 레이턴시

In [ ]:
# 상세 검색 레이턴시 버전으로 기존 함수를 재정의합니다.
import time


_V15_LAST_QUERY_TRACE: dict[str, Any] = {}


def hybrid_minmax_search(question: str, *, top_k: int = FINAL_TOP_K) -> list[dict[str, Any]]:
    global _V15_LAST_QUERY_TRACE
    cleaned = _clean_text(question)
    if not cleaned:
        raise ValueError("검색 질문이 비어 있습니다.")
    total_started = time.perf_counter()

    embedding_started = time.perf_counter()
    query_vector = _normalize_vector(embed_hcx_single(cleaned))
    embedding_latency_ms = (time.perf_counter() - embedding_started) * 1000
    if query_vector.shape != (DENSE_DIMENSION,):
        raise RuntimeError(
            f"질문 임베딩 차원 불일치: query={query_vector.shape}, stored={DENSE_DIMENSION}"
        )

    dense_started = time.perf_counter()
    dense_results = dense_search_by_vector(query_vector, CANDIDATE_DEPTH)
    dense_compute_latency_ms = (time.perf_counter() - dense_started) * 1000

    bm25_started = time.perf_counter()
    bm25_results = bm25_search(cleaned, CANDIDATE_DEPTH)
    bm25_latency_ms = (time.perf_counter() - bm25_started) * 1000

    minmax_started = time.perf_counter()
    results = weighted_minmax(
        dense_results,
        bm25_results,
        dense_weight=DENSE_WEIGHT,
        bm25_weight=BM25_WEIGHT,
        top_k=top_k,
    )
    minmax_latency_ms = (time.perf_counter() - minmax_started) * 1000
    total_latency_ms = (time.perf_counter() - total_started) * 1000

    _V15_LAST_QUERY_TRACE = {
        "question": cleaned,
        "embedding_latency_ms": embedding_latency_ms,
        "dense_compute_latency_ms": dense_compute_latency_ms,
        "bm25_latency_ms": bm25_latency_ms,
        "minmax_latency_ms": minmax_latency_ms,
        "query_total_latency_ms": total_latency_ms,
        "dense_candidate_count": len(dense_results),
        "bm25_candidate_count": len(bm25_results),
        "combined_candidate_count": len(results),
    }
    if not results:
        raise RuntimeError("Hybrid Min-Max 검색 결과가 없습니다.")
    return results


def fuse_query_results(
    plans: list[dict[str, Any]],
    *,
    top_k: int = FINAL_TOP_K,
    rrf_k: int = QUERY_FUSION_RRF_K,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    if not plans:
        raise ValueError("검색 계획이 없습니다.")
    if not math.isclose(sum(float(plan["weight"]) for plan in plans), 1.0, abs_tol=1e-9):
        raise ValueError("검색 계획 가중치 합은 1이어야 합니다.")

    per_query = []
    all_hits = []
    for plan_index, plan in enumerate(plans, start=1):
        hits = hybrid_minmax_search(str(plan["query"]), top_k=CANDIDATE_DEPTH)
        trace = dict(_V15_LAST_QUERY_TRACE)
        per_query.append({
            **plan,
            "plan_index": plan_index,
            "latency_ms": trace["query_total_latency_ms"],
            "latency_breakdown_ms": trace,
            "hits": hits,
        })
        all_hits.append((plan_index, plan, hits))

    fusion_started = time.perf_counter()
    fused: dict[str, dict[str, Any]] = {}
    for plan_index, plan, hits in all_hits:
        for hit in hits:
            chunk_id = str(hit["chunk_id"])
            row = fused.setdefault(chunk_id, {
                "chunk_id": chunk_id,
                "query_fusion_score": 0.0,
                "best_minmax_score": 0.0,
                "dense_rank": None,
                "bm25_rank": None,
                "matched_queries": [],
                "chunk": hit["chunk"],
            })
            row["query_fusion_score"] += float(plan["weight"]) / (rrf_k + int(hit["rank"]))
            row["best_minmax_score"] = max(row["best_minmax_score"], float(hit["minmax_score"]))
            if row["dense_rank"] is None or (hit["dense_rank"] is not None and hit["dense_rank"] < row["dense_rank"]):
                row["dense_rank"] = hit["dense_rank"]
            if row["bm25_rank"] is None or (hit["bm25_rank"] is not None and hit["bm25_rank"] < row["bm25_rank"]):
                row["bm25_rank"] = hit["bm25_rank"]
            row["matched_queries"].append({
                "plan_index": plan_index,
                "query": plan["query"],
                "source": plan["source"],
                "rank": hit["rank"],
                "weight": plan["weight"],
            })

    ordered = sorted(fused.values(), key=lambda row: (
        -row["query_fusion_score"], -row["best_minmax_score"], row["chunk_id"]
    ))
    final = [
        {
            **row,
            "rank": rank,
            "minmax_score": row["best_minmax_score"],
        }
        for rank, row in enumerate(ordered[:top_k], start=1)
    ]
    fusion_latency_ms = (time.perf_counter() - fusion_started) * 1000
    for row in per_query:
        row["query_fusion_latency_ms"] = fusion_latency_ms
    return final, per_query


print("상세 검색 레이턴시 측정 준비 완료")

### 3-2. BGE CrossEncoder Reranker

Hybrid 7:3 Min/Max와 다중질의 결합으로 만든 상위 20개 Child 후보를 같은 평가 실험에서 사용한 `BAAI/bge-reranker-v2-m3`로 재정렬합니다. 답변 D안에는 최종 Top-5만 전달합니다.


In [ ]:
%%writefile kdic_v15_context_rerank_core.py
from __future__ import annotations

import re
import time
from typing import Any, Callable, Iterable, Mapping, Sequence

import numpy as np


BUSINESS_LABELS = (
    "예금자보호제도",
    "예금보험금 안내",
    "고객 미수령금 신청",
    "착오송금 반환 신청",
    "채무조정 안내",
    "은닉재산 신고",
)

BUSINESS_ALIASES: dict[str, tuple[str, ...]] = {
    "예금자보호제도": (
        "예금자보호", "보호한도", "보호 대상", "보호대상",
    ),
    "예금보험금 안내": (
        "예금보험금", "보험금 지급", "보험사고",
    ),
    "고객 미수령금 신청": (
        "미수령금", "파산배당금", "개산지급금 정산금",
    ),
    "착오송금 반환 신청": (
        "착오송금", "잘못 송금", "잘못송금", "반환지원",
    ),
    "채무조정 안내": (
        "채무조정", "신용회복", "채무감면",
    ),
    "은닉재산 신고": (
        "은닉재산", "은닉 재산",
    ),
}

INTENT_ONLY_PATTERN = re.compile(
    r"(?:신청|접수|서류|구비서류|준비물|조건|자격|대상|방법|절차|"
    r"기간|기한|금액|한도|조회|상태|연락처|전화번호)"
)
REFERENCE_PATTERN = re.compile(
    r"(?:^|\s)(?:그거|이거|그것|이것|그\s*신청|해당\s*신청|그\s*경우|"
    r"해당\s*경우|그러면|그럼|거기는|거기서)(?:\s|$|[?!.])"
)
SELECTION_PATTERN = re.compile(r"^\s*(\d{1,2})\s*(?:번)?\s*$")


def _clean_text(value: Any) -> str:
    text = str(value or "").replace("\x00", " ")
    return re.sub(r"\s+", " ", text).strip()


def _ordered_unique(values: Iterable[str]) -> list[str]:
    seen: set[str] = set()
    output: list[str] = []
    for value in values:
        cleaned = _clean_text(value)
        if cleaned and cleaned not in seen:
            seen.add(cleaned)
            output.append(cleaned)
    return output


def detect_businesses(text: str) -> list[str]:
    cleaned = _clean_text(text).lower()
    return [
        business
        for business, aliases in BUSINESS_ALIASES.items()
        if any(alias.lower() in cleaned for alias in aliases)
    ]


def _user_messages(previous_turns: Any) -> list[str]:
    if not isinstance(previous_turns, Sequence) or isinstance(previous_turns, (str, bytes)):
        return []
    output: list[str] = []
    for turn in previous_turns:
        if not isinstance(turn, Mapping):
            continue
        role = _clean_text(turn.get("role")).lower()
        if role:
            if role != "user":
                continue
            content = _clean_text(turn.get("content"))
            if content:
                output.append(content)
            continue
        user = _clean_text(turn.get("user") or turn.get("query"))
        if user:
            output.append(user)
    return output


def latest_context_businesses(
    previous_turns: Any,
    *,
    confirmed_businesses: Sequence[str] | None = None,
    detector: Callable[[str], list[str]] = detect_businesses,
) -> list[str]:
    confirmed = _ordered_unique(confirmed_businesses or [])
    if confirmed:
        return confirmed
    for message in reversed(_user_messages(previous_turns)):
        found = _ordered_unique(detector(message))
        if found:
            return found
    return []


def _is_context_dependent(question: str) -> bool:
    compact_length = len(re.sub(r"\s+", "", question))
    short_intent_only = compact_length <= 30 and bool(INTENT_ONLY_PATTERN.search(question))
    has_reference = bool(REFERENCE_PATTERN.search(question))
    return short_intent_only or has_reference


def _clarification_message(candidates: Sequence[str], *, repeated: bool = False) -> str:
    candidates = _ordered_unique(candidates) or list(BUSINESS_LABELS)
    prefix = (
        "아직 어떤 업무를 말씀하시는지 확인하기 어렵습니다."
        if repeated
        else "어떤 업무에 관한 질문인지 확인이 필요합니다."
    )
    lines = [prefix, "", "아래에서 선택하거나 업무명을 직접 입력해 주세요.", ""]
    lines.extend(f"{index}. {business}" for index, business in enumerate(candidates, start=1))
    return "\n".join(lines)


def _pending_payload(
    *,
    original_question: str,
    candidates: Sequence[str],
    clarification_count: int,
) -> dict[str, Any]:
    return {
        "active": True,
        "original_question": original_question,
        "missing_slots": ["business_function"],
        "business_candidates": _ordered_unique(candidates) or list(BUSINESS_LABELS),
        "clarification_count": int(clarification_count),
    }


def resolve_conversational_question(
    question: str,
    *,
    previous_turns: Any = None,
    pending_clarification: Mapping[str, Any] | None = None,
    confirmed_businesses: Sequence[str] | None = None,
    detector: Callable[[str], list[str]] = detect_businesses,
) -> dict[str, Any]:
    """검색 전 문맥을 보수적으로 복원하거나 CLARIFY를 반환한다.

    업무를 추정할 근거가 하나로 수렴하지 않으면 검색을 허용하지 않는다.
    """
    started = time.perf_counter()
    original = _clean_text(question)
    if not original:
        raise ValueError("사용자 질문이 비어 있습니다.")

    explicit_businesses = _ordered_unique(detector(original))
    pending = dict(pending_clarification or {})
    pending_active = bool(pending.get("active"))

    if pending_active:
        candidates = _ordered_unique(pending.get("business_candidates") or BUSINESS_LABELS)
        selected: list[str] = []
        numeric = SELECTION_PATTERN.fullmatch(original)
        if numeric:
            index = int(numeric.group(1)) - 1
            if 0 <= index < len(candidates):
                selected = [candidates[index]]
        if not selected:
            selected = [item for item in explicit_businesses if item in candidates]
        if not selected and len(explicit_businesses) == 1:
            selected = explicit_businesses

        if len(selected) == 1:
            pending_question = _clean_text(pending.get("original_question"))
            is_short_selection = len(re.sub(r"\s+", "", original)) <= 20
            resolved = (
                f"{selected[0]} {pending_question}"
                if pending_question and is_short_selection
                else original
            )
            return {
                "route": "RETRIEVE",
                "original_question": original,
                "resolved_question": resolved,
                "context_used": True,
                "context_businesses": selected,
                "resolution_reason": "PENDING_CLARIFICATION_RESOLVED",
                "clarification_message": "",
                "pending_clarification": None,
                "latency_ms": (time.perf_counter() - started) * 1000,
            }

        count = int(pending.get("clarification_count") or 1) + 1
        return {
            "route": "CLARIFY",
            "original_question": original,
            "resolved_question": "",
            "context_used": False,
            "context_businesses": candidates,
            "resolution_reason": "PENDING_CLARIFICATION_UNRESOLVED",
            "clarification_message": _clarification_message(candidates, repeated=True),
            "pending_clarification": _pending_payload(
                original_question=_clean_text(pending.get("original_question")) or original,
                candidates=candidates,
                clarification_count=count,
            ),
            "escalation_recommended": count >= 2,
            "latency_ms": (time.perf_counter() - started) * 1000,
        }

    if explicit_businesses:
        return {
            "route": "CONTINUE",
            "original_question": original,
            "resolved_question": original,
            "context_used": False,
            "context_businesses": explicit_businesses,
            "resolution_reason": "EXPLICIT_BUSINESS_IN_CURRENT_QUESTION",
            "clarification_message": "",
            "pending_clarification": None,
            "latency_ms": (time.perf_counter() - started) * 1000,
        }

    if not _is_context_dependent(original):
        return {
            "route": "CONTINUE",
            "original_question": original,
            "resolved_question": original,
            "context_used": False,
            "context_businesses": [],
            "resolution_reason": "STANDALONE_OR_BASE_ROUTER_DECISION",
            "clarification_message": "",
            "pending_clarification": None,
            "latency_ms": (time.perf_counter() - started) * 1000,
        }

    context_businesses = latest_context_businesses(
        previous_turns,
        confirmed_businesses=confirmed_businesses,
        detector=detector,
    )
    if len(context_businesses) == 1:
        return {
            "route": "RETRIEVE",
            "original_question": original,
            "resolved_question": f"{context_businesses[0]} {original}",
            "context_used": True,
            "context_businesses": context_businesses,
            "resolution_reason": "UNIQUE_PREVIOUS_BUSINESS",
            "clarification_message": "",
            "pending_clarification": None,
            "latency_ms": (time.perf_counter() - started) * 1000,
        }

    candidates = context_businesses or list(BUSINESS_LABELS)
    reason = "MULTIPLE_PREVIOUS_BUSINESSES" if len(context_businesses) > 1 else "BUSINESS_NOT_SPECIFIED"
    return {
        "route": "CLARIFY",
        "original_question": original,
        "resolved_question": "",
        "context_used": False,
        "context_businesses": context_businesses,
        "resolution_reason": reason,
        "clarification_message": _clarification_message(candidates),
        "pending_clarification": _pending_payload(
            original_question=original,
            candidates=candidates,
            clarification_count=1,
        ),
        "escalation_recommended": False,
        "latency_ms": (time.perf_counter() - started) * 1000,
    }


def _predict_scores(model: Any, pairs: list[list[str]], *, batch_size: int) -> np.ndarray:
    if hasattr(model, "predict"):
        raw = model.predict(
            pairs,
            batch_size=batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
        )
    elif hasattr(model, "compute_score"):
        raw = model.compute_score(pairs, batch_size=batch_size, normalize=True)
    else:
        raise TypeError("Reranker 모델에 predict 또는 compute_score 메서드가 없습니다.")
    scores = np.atleast_1d(np.asarray(raw, dtype=np.float32)).reshape(-1)
    if len(scores) != len(pairs):
        raise RuntimeError(
            f"Reranker 점수 개수 불일치: pairs={len(pairs)}, scores={len(scores)}"
        )
    return scores


def rerank_candidates(
    question: str,
    candidates: Sequence[Mapping[str, Any]],
    *,
    chunks_by_id: Mapping[str, Mapping[str, Any]],
    model: Any,
    text_builder: Callable[[Mapping[str, Any]], str],
    candidate_depth: int = 20,
    final_top_k: int = 5,
    batch_size: int = 8,
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    """Hybrid 상위 후보를 CrossEncoder로 재정렬한다."""
    if candidate_depth < final_top_k or final_top_k < 1:
        raise ValueError("candidate_depth는 final_top_k 이상이어야 합니다.")
    started = time.perf_counter()
    prepared: list[dict[str, Any]] = []
    pairs: list[list[str]] = []
    seen: set[str] = set()
    for base_rank, item in enumerate(candidates[:candidate_depth], start=1):
        chunk_id = _clean_text(item.get("chunk_id"))
        if not chunk_id or chunk_id in seen:
            continue
        seen.add(chunk_id)
        chunk = chunks_by_id.get(chunk_id)
        if chunk is None:
            raise KeyError(f"Reranker 후보 청크가 corpus에 없습니다: {chunk_id}")
        passage = _clean_text(text_builder(chunk))
        if not passage:
            continue
        prepared.append({**dict(item), "pre_rerank_rank": base_rank})
        pairs.append([_clean_text(question), passage])

    if not prepared:
        raise RuntimeError("Reranker에 전달할 유효 후보가 없습니다.")
    scores = _predict_scores(model, pairs, batch_size=batch_size)
    scored = [
        {**row, "reranker_score": float(score)}
        for row, score in zip(prepared, scores)
    ]
    ordered = sorted(
        scored,
        key=lambda row: (
            -float(row["reranker_score"]),
            int(row["pre_rerank_rank"]),
            str(row["chunk_id"]),
        ),
    )
    final = [
        {**row, "rank": rank}
        for rank, row in enumerate(ordered[:final_top_k], start=1)
    ]
    return final, {
        "latency_ms": (time.perf_counter() - started) * 1000,
        "candidate_count": len(prepared),
        "returned_count": len(final),
        "batch_size": int(batch_size),
        "question": _clean_text(question),
    }


In [ ]:
import torch
from sentence_transformers import CrossEncoder

from kdic_v15_context_rerank_core import rerank_candidates

RERANKER_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RERANKER_MODEL = CrossEncoder(
    RERANKER_MODEL_NAME,
    device=RERANKER_DEVICE,
    max_length=RERANKER_MAX_LENGTH,
)

_FUSE_QUERY_RESULTS_BEFORE_RERANKER = fuse_query_results
_LAST_RERANK_TRACE: dict[str, Any] = {}


def _reranker_passage(chunk: dict[str, Any]) -> str:
    return build_dense_structured_v2_text(chunk)[:4_000]


def fuse_query_results(
    plans: list[dict[str, Any]],
    *,
    top_k: int = FINAL_TOP_K,
    rrf_k: int = QUERY_FUSION_RRF_K,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    global _LAST_RERANK_TRACE
    # 먼저 Hybrid/다중질의 결합 상위 20개를 확보합니다.
    candidates, per_query = _FUSE_QUERY_RESULTS_BEFORE_RERANKER(
        plans,
        top_k=RERANKER_CANDIDATE_DEPTH,
        rrf_k=rrf_k,
    )
    original_plan = next(
        (plan for plan in plans if "ORIGINAL" in str(plan.get("source") or "")),
        plans[0],
    )
    rerank_question = str(original_plan.get("query") or "").strip()
    reranked, trace = rerank_candidates(
        rerank_question,
        candidates,
        chunks_by_id=CHUNKS_BY_ID,
        model=RERANKER_MODEL,
        text_builder=_reranker_passage,
        candidate_depth=RERANKER_CANDIDATE_DEPTH,
        final_top_k=top_k,
        batch_size=RERANKER_BATCH_SIZE,
    )
    _LAST_RERANK_TRACE = {
        **trace,
        "model": RERANKER_MODEL_NAME,
        "device": RERANKER_DEVICE,
    }
    for row in per_query:
        row["reranker_latency_ms"] = float(trace["latency_ms"])
        row["reranker_candidate_count"] = int(trace["candidate_count"])
    return reranked, per_query


print({
    "reranker": RERANKER_MODEL_NAME,
    "device": RERANKER_DEVICE,
    "candidate_depth": RERANKER_CANDIDATE_DEPTH,
    "final_top_k": FINAL_TOP_K,
})


### 3-3. Parent-Child Retrieval — Child 검색 후 Parent 문맥 확장

현재 검색 순위 자체는 **Child 청크**로 결정합니다.

```text
Hybrid 7:3 Min-Max
    ↓
다중질의 RRF
    ↓
BGE Reranker
    ↓
Top-5 Child 확정
    ↓
parent_doc_id 기준 Parent 확장
    ↓
Parent Evidence Pack
    ↓
Answer Skeleton → 최종 답변
```

이렇게 구성한 이유는 `Child만 vs Parent 확장` 실험에서 **검색 랭킹은 동일하게 유지하고, 답변 생성에 전달되는 문맥만 바꾸기 위해서**입니다.

- 검색 평가: 기존 Top-5 Child 기준 그대로 수행
- 답변 생성: 같은 Parent의 sibling 청크를 함께 전달
- 같은 Parent에서 Child가 여러 개 검색되면 Parent 문맥은 Evidence Pack에 한 번만 넣어 중복 토큰을 줄임
- `PARENT_CONTEXT_MAX_CHARS=None`은 전체 Parent 확장입니다.
- 이후 컨텍스트 길이 제어 실험을 할 때만 `PARENT_CONTEXT_MAX_CHARS`에 숫자를 넣으면 됩니다.


In [ ]:
import time

_LAST_PARENT_CHILD_TRACE: dict[str, Any] = {}


def _parent_id_for_chunk(chunk: dict[str, Any]) -> str:
    chunk_id = _clean_text(chunk.get("chunk_id"))
    return (
        _clean_text(chunk.get("parent_doc_id"))
        or _clean_text(chunk.get("document_id"))
        or chunk_id
    )


def _select_parent_context_chunks(
    parent_id: str,
    matched_child_ids: list[str],
    *,
    max_chars: int | None = PARENT_CONTEXT_MAX_CHARS,
) -> list[dict[str, Any]]:
    """Parent의 sibling 청크를 문서 순서대로 반환합니다.

    max_chars=None이면 전체 Parent를 사용합니다.
    숫자이면 matched child를 반드시 우선 포함하고, 가장 가까운 sibling부터
    예산 안에서 추가한 뒤 최종 출력은 원문 chunk_index 순서로 정렬합니다.
    """
    children = list(PARENT_CHILDREN_BY_ID.get(parent_id) or [])
    if not children:
        return []

    if max_chars is None:
        return children

    if max_chars <= 0:
        raise ValueError("PARENT_CONTEXT_MAX_CHARS는 None 또는 양수여야 합니다.")

    positions = {
        str(chunk.get("chunk_id") or ""): index
        for index, chunk in enumerate(children)
    }
    matched_positions = [
        positions[chunk_id]
        for chunk_id in matched_child_ids
        if chunk_id in positions
    ]
    if not matched_positions:
        matched_positions = [0]

    def distance(index: int) -> tuple[int, int]:
        return (min(abs(index - anchor) for anchor in matched_positions), index)

    priority = sorted(range(len(children)), key=distance)
    selected_indices: list[int] = []
    used_chars = 0

    # matched child는 예산보다 길더라도 최소 1개는 보존합니다.
    matched_set = set(matched_child_ids)
    for index in priority:
        child = children[index]
        chunk_id = str(child.get("chunk_id") or "")
        content_chars = len(_clean_text(child.get("content")))
        must_include = chunk_id in matched_set or not selected_indices
        if must_include or used_chars + content_chars <= max_chars:
            selected_indices.append(index)
            used_chars += content_chars

    selected_indices.sort()
    return [children[index] for index in selected_indices]


def expand_parent_context(
    search_results: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Reranker Top-K Child를 유지하면서 Parent Evidence 단위만 연결합니다."""
    global _LAST_PARENT_CHILD_TRACE
    started = time.perf_counter()

    if not PARENT_CHILD_ENABLED:
        output = []
        for result in search_results:
            row = dict(result)
            row["parent_doc_id"] = _parent_id_for_chunk(result["chunk"])
            row["parent_evidence_ref"] = f"C{int(result['rank'])}"
            row["parent_context_chunk_ids"] = [str(result["chunk_id"])]
            row["parent_context_chunk_count"] = 1
            row["parent_context_char_count"] = len(_clean_text(result["chunk"].get("content")))
            output.append(row)
        _LAST_PARENT_CHILD_TRACE = {
            "enabled": False,
            "latency_ms": (time.perf_counter() - started) * 1000,
            "matched_child_count": len(search_results),
            "unique_parent_count": len(search_results),
            "expanded_chunk_count": len(search_results),
            "expanded_char_count": sum(row["parent_context_char_count"] for row in output),
        }
        return output

    # Top-K 안에서 같은 Parent가 여러 번 검색되면 하나의 Evidence ref를 공유합니다.
    parent_order: list[str] = []
    matched_by_parent: dict[str, list[str]] = {}
    for result in search_results:
        chunk = result["chunk"]
        parent_id = _parent_id_for_chunk(chunk)
        if parent_id not in matched_by_parent:
            parent_order.append(parent_id)
            matched_by_parent[parent_id] = []
        matched_by_parent[parent_id].append(str(result["chunk_id"]))

    evidence_ref_by_parent = {
        parent_id: f"C{index}"
        for index, parent_id in enumerate(parent_order, start=1)
    }
    selected_by_parent: dict[str, list[dict[str, Any]]] = {
        parent_id: _select_parent_context_chunks(
            parent_id,
            matched_by_parent[parent_id],
            max_chars=PARENT_CONTEXT_MAX_CHARS,
        )
        for parent_id in parent_order
    }

    output: list[dict[str, Any]] = []
    for result in search_results:
        row = dict(result)
        parent_id = _parent_id_for_chunk(result["chunk"])
        selected = selected_by_parent[parent_id]
        row["parent_doc_id"] = parent_id
        row["parent_evidence_ref"] = evidence_ref_by_parent[parent_id]
        row["parent_context_chunk_ids"] = [
            str(chunk.get("chunk_id") or "")
            for chunk in selected
        ]
        row["parent_context_chunk_count"] = len(selected)
        row["parent_context_char_count"] = sum(
            len(_clean_text(chunk.get("content")))
            for chunk in selected
        )
        output.append(row)

    unique_selected = {
        (parent_id, str(chunk.get("chunk_id") or ""))
        for parent_id, chunks in selected_by_parent.items()
        for chunk in chunks
    }
    _LAST_PARENT_CHILD_TRACE = {
        "enabled": True,
        "latency_ms": (time.perf_counter() - started) * 1000,
        "matched_child_count": len(search_results),
        "unique_parent_count": len(parent_order),
        "expanded_chunk_count": len(unique_selected),
        "expanded_char_count": sum(
            len(_clean_text(chunk.get("content")))
            for chunks in selected_by_parent.values()
            for chunk in chunks
        ),
        "parent_context_max_chars": PARENT_CONTEXT_MAX_CHARS,
        "parent_refs": evidence_ref_by_parent,
    }
    return output


def parent_child_markdown(search_results: list[dict[str, Any]]) -> str:
    if not search_results:
        return "### Parent-Child 확장\n\n검색 결과가 없습니다."
    seen: set[str] = set()
    lines = [
        "### Parent-Child 확장",
        "",
        "|Evidence|Parent|매칭 Child|확장 청크 수|확장 문자 수|",
        "|---|---|---|---:|---:|",
    ]
    for result in search_results:
        parent_id = str(result.get("parent_doc_id") or "")
        if parent_id in seen:
            continue
        seen.add(parent_id)
        ref = str(result.get("parent_evidence_ref") or "-")
        matched = [
            str(row["chunk_id"])
            for row in search_results
            if str(row.get("parent_doc_id") or "") == parent_id
        ]
        lines.append(
            f"|{ref}|{parent_id}|{', '.join(matched)}|"
            f"{int(result.get('parent_context_chunk_count') or 0)}|"
            f"{int(result.get('parent_context_char_count') or 0):,}|"
        )
    return "\n".join(lines)


print({
    "parent_child_enabled": PARENT_CHILD_ENABLED,
    "parent_context_max_chars": PARENT_CONTEXT_MAX_CHARS,
    "parent_count": len(PARENT_CHILDREN_BY_ID),
})


## 8. D안 — Top 5에서 Answer Skeleton 생성 후 답변 작성

Top 5는 먼저 규칙 기반 Evidence Pack으로 고정합니다. 첫 번째 HCX-005 호출은 그 Pack을 읽고 다음 최소 구조의 JSON만 생성합니다.

- `answer_type`: 최종 답변의 구성 유형
- `core_answer`: 질문에 대한 직접적인 핵심 결론
- `answer_items`: 반드시 포함할 항목·주장·조건·결과·용어 풀이·세부 근거
- `uncertainties`: Top 5로 확인되지 않는 내용
- `conflicts`: 청크 사이의 충돌
- `coverage_status`: `SUFFICIENT`, `PARTIAL`, `INSUFFICIENT` 중 하나

두 번째 호출은 검증된 Answer Skeleton과 동일 Evidence Pack으로 이해하기 쉬운 기본 설명을 생성합니다. `전문가 설명 보기`는 같은 두 객체와 이미 생성된 기본 설명을 재사용합니다.

현재 `coverage_status`는 관찰·저장용입니다. 임계값이나 답변 차단 로직은 평가 코드로 전환할 때 추가합니다.


In [ ]:
ANSWER_SKELETON_SYSTEM_PROMPT = """
당신은 예금보험공사 공식 문서에서 답변에 필요한 사실 구조만 추출하는 분석기입니다.

반드시 지킬 규칙:
1. 제공된 사용자 질문과 Evidence Pack에 명시된 내용만 사용하세요.
2. 최종 사용자용 문장을 작성하지 말고 Answer Skeleton만 작성하세요.
3. 원문에 없는 사실을 추정하거나 일반상식으로 보완하지 마세요.
4. 질문에 답하는 데 필요한 사실만 남기고 검색 점수와 내부 구현은 제외하세요.
5. 각 claim에는 실제로 그 주장을 뒷받침하는 청크 ID만 연결하세요.
6. 청크끼리 내용이 다르면 임의로 하나를 선택하지 말고 conflicts에 기록하세요.
7. 확인되지 않는 필수 내용은 uncertainties에 기록하세요.
8. 각 answer_item은 질문이 요구한 서로 다른 항목 하나만 표현하고 임의로 합치지 마세요.
9. Evidence Pack에 있는 경우에만 conditions, result, detail_points, term_explanations를 작성하세요.
10. URL과 전화번호는 Skeleton에 넣지 마세요.
11. 반드시 JSON 객체 하나만 출력하고 Markdown 코드 블록이나 설명을 붙이지 마세요.
""".strip()

FINAL_ANSWER_SYSTEM_PROMPT = """
당신은 예금보험공사 공식 문서 기반 질의응답 시스템입니다.

반드시 지킬 규칙:
1. 제공된 Answer Skeleton과 Evidence Pack에 적힌 사실만 사용하여 한국어로 답하세요.
2. 질문에 먼저 직접 답한 뒤 필요한 조건·예외·금액·기간·절차를 설명하세요.
3. Answer Skeleton과 Evidence Pack에 없는 사실을 추정하거나 일반상식으로 보완하지 마세요.
4. 각 주장에는 Skeleton의 evidence_refs에 있는 [C1]~[C5]만 표시하세요. C번호는 검색 Child 순위가 아니라 Evidence Pack의 Parent 근거 그룹을 뜻할 수 있습니다.
5. conflicts가 있으면 한쪽을 임의로 선택하지 말고 차이를 설명하세요.
6. uncertainties 또는 근거 부족 내용은 확인할 수 없다고 명시하세요.
7. URL, 전화번호, 추천 질문, 추천 키워드를 답변 본문에 작성하지 마세요.
8. Answer Skeleton의 answer_items 순서와 개수를 유지하고 항목을 합치거나 제거하지 마세요.
9. 각 answer_item을 독립된 문단 또는 목록 항목으로 작성하고, 그 항목 끝에 해당 evidence_refs를 표시하세요.
10. 검색 점수, Answer Skeleton, Evidence Pack, JSON, 내부 구현을 답변 본문에서 언급하지 마세요.
""".strip()

EASY_STYLE_PROMPT = """
기본 설명 작성 방식:
- 답변을 짧게 줄이는 것보다 사용자가 이해할 수 있게 충분히 설명하는 것을 우선하세요.
- 첫 부분에서 질문에 대한 직접적인 결론을 제시하세요.
- 일반 사용자가 바로 이해할 수 있는 일상적인 표현을 사용하세요.
- 전문용어가 필요하면 공식 용어를 쓰고 같은 문장 안에서 쉬운 뜻을 설명하세요.
- 조건, 예외, 금액, 기간, 신청 절차처럼 판단과 행동에 영향을 주는 사실은 생략하지 마세요.
- 일반 조건과 예외 조건을 분리하고, 절차는 실제 순서대로 번호를 붙이세요.
- 한 문장에 조건을 몰아넣지 말고 필요한 문단과 목록으로 나누세요.
- 사용자를 어린이처럼 대하거나 정보를 과도하게 단순화하지 마세요.
""".strip()

DETAILED_STYLE_PROMPT = """
전문가 설명 작성 방식:
- 기본 설명의 결론, 대상, 조건, 금액, 기간, 절차, 예외를 그대로 유지하세요.
- 기본 설명의 문장을 단순히 길게 늘이거나 같은 말을 반복하지 마세요.
- 각 answer_item을 '무엇인지 → 적용 대상·상황 → 정확한 조건 → 적용 결과' 순서로 상세화하세요.
- Evidence Pack에 실제로 있는 세부 조건, 공식 용어, 적용 범위와 예외 관계를 추가로 설명하세요.
- 비슷한 용어 또는 진행 중·완료 상태의 차이는 근거가 있을 때만 구분하세요.
- 기본 설명에 없더라도 Skeleton과 Evidence Pack에 있는 관련 세부사항은 복원해 설명하세요.
- 근거에 없는 법적 이유, 사례, 수치, 기간, 절차는 추가하지 마세요.
- 필요하면 소제목과 목록을 사용하여 구조적으로 작성하세요.
""".strip()

ALLOWED_COVERAGE_STATUS = {"SUFFICIENT", "PARTIAL", "INSUFFICIENT"}
ALLOWED_ANSWER_TYPES = {
    "INFORMATION_ANSWER",
    "ACTION_GUIDE",
    "ELIGIBILITY_ANSWER",
    "AMOUNT_DEADLINE_ANSWER",
    "COMPARISON_ANSWER",
    "OTHER",
}


def build_evidence_pack(search_results: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """검색 Child를 근거로, Parent-Child가 켜져 있으면 Parent 단위 Evidence Pack을 만듭니다."""
    if not search_results:
        return []

    if not PARENT_CHILD_ENABLED:
        evidence_pack: list[dict[str, Any]] = []
        for result in search_results:
            original_chunk = result["chunk"]
            if not isinstance(original_chunk, dict):
                raise TypeError("원본 청크가 dict가 아닙니다.")
            evidence_pack.append({
                "evidence_ref": f"C{int(result['rank'])}",
                "chunk_id": str(result["chunk_id"]),
                "document_id": _clean_text(original_chunk.get("document_id")),
                "title": _clean_text(original_chunk.get("title")),
                "section_title": _clean_text(original_chunk.get("section_title")),
                "content": _clean_text(original_chunk.get("content")),
                "source_url": _clean_text(original_chunk.get("source_url")),
            })
        return evidence_pack

    by_ref: OrderedDict[str, dict[str, Any]] = OrderedDict()
    for result in search_results:
        original_chunk = result["chunk"]
        if not isinstance(original_chunk, dict):
            raise TypeError("원본 청크가 dict가 아닙니다.")

        evidence_ref = str(result.get("parent_evidence_ref") or "").strip()
        if not evidence_ref:
            raise RuntimeError("Parent-Child 확장 결과에 parent_evidence_ref가 없습니다.")
        parent_id = str(result.get("parent_doc_id") or _parent_id_for_chunk(original_chunk))
        context_ids = list(result.get("parent_context_chunk_ids") or [str(result["chunk_id"])])

        row = by_ref.setdefault(evidence_ref, {
            "evidence_ref": evidence_ref,
            "parent_doc_id": parent_id,
            "document_id": _clean_text(original_chunk.get("document_id")) or parent_id,
            "title": _clean_text(original_chunk.get("title")),
            "source_url": _clean_text(original_chunk.get("source_url")),
            "matched_child_ids": [],
            "matched_child_ranks": [],
            "context_chunks": [],
        })
        row["matched_child_ids"].append(str(result["chunk_id"]))
        row["matched_child_ranks"].append(int(result["rank"]))

        if not row["context_chunks"]:
            context_chunks = []
            for chunk_id in context_ids:
                chunk = CHUNKS_BY_ID.get(str(chunk_id))
                if chunk is None:
                    raise KeyError(f"Parent context 청크가 corpus에 없습니다: {chunk_id}")
                context_chunks.append({
                    "chunk_id": str(chunk.get("chunk_id") or ""),
                    "chunk_index": int(chunk.get("chunk_index") or 0),
                    "title": _clean_text(chunk.get("title")),
                    "section_title": _clean_text(chunk.get("section_title")),
                    "heading_path": chunk.get("heading_path") or [],
                    "content": _clean_text(chunk.get("content")),
                })
            row["context_chunks"] = context_chunks

    for row in by_ref.values():
        row["matched_child_ids"] = list(dict.fromkeys(row["matched_child_ids"]))
        row["matched_child_ranks"] = sorted(set(row["matched_child_ranks"]))
        row["context_chunk_count"] = len(row["context_chunks"])
        row["context_char_count"] = sum(
            len(_clean_text(chunk.get("content")))
            for chunk in row["context_chunks"]
        )
    return list(by_ref.values())


def format_evidence_pack(evidence_pack: list[dict[str, Any]]) -> str:
    return json.dumps(evidence_pack, ensure_ascii=False, indent=2, default=str)


def _extract_json_object(text: str) -> dict[str, Any]:
    cleaned = str(text or "").strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        value = json.loads(cleaned)
    except json.JSONDecodeError:
        decoder = json.JSONDecoder()
        start = cleaned.find("{")
        if start < 0:
            raise ValueError("HCX 출력에서 JSON 객체를 찾지 못했습니다.")
        try:
            value, _ = decoder.raw_decode(cleaned[start:])
        except json.JSONDecodeError as error:
            raise ValueError(f"Answer Skeleton JSON 파싱 실패: {error}") from error
    if not isinstance(value, dict):
        raise TypeError("Answer Skeleton 최상위 값은 JSON 객체여야 합니다.")
    return value


def _as_clean_string_list(value: Any) -> list[str]:
    """HCX가 문자열 하나 또는 배열을 반환해도 문자열 배열로 정규화합니다."""
    if value is None:
        return []
    if isinstance(value, (str, int, float)):
        values = [value]
    elif isinstance(value, (list, tuple, set)):
        values = list(value)
    else:
        return []

    output: list[str] = []
    seen: set[str] = set()
    for item in values:
        text = _clean_text(item)
        if text and text not in seen:
            seen.add(text)
            output.append(text)
    return output


def _normalize_term_explanations(value: Any) -> list[dict[str, str]]:
    if isinstance(value, dict):
        value = [
            {"term": term, "plain_explanation": explanation}
            for term, explanation in value.items()
        ]
    if not isinstance(value, list):
        return []
    output: list[dict[str, str]] = []
    seen: set[tuple[str, str]] = set()
    for item in value:
        if not isinstance(item, dict):
            continue
        term = _clean_text(item.get("term") or item.get("name"))
        explanation = _clean_text(
            item.get("plain_explanation")
            or item.get("explanation")
            or item.get("meaning")
        )
        key = (term, explanation)
        if term and explanation and key not in seen:
            seen.add(key)
            output.append({"term": term, "plain_explanation": explanation})
    return output


def _first_present(mapping: dict[str, Any], keys: tuple[str, ...]) -> Any:
    for key in keys:
        value = mapping.get(key)
        if value not in (None, "", []):
            return value
    return None


def _flatten_evidence_values(value: Any) -> list[str]:
    """근거가 문자열·배열·객체 중 어느 형태로 와도 식별자 후보를 꺼냅니다."""
    if value is None:
        return []
    if isinstance(value, dict):
        values: list[str] = []
        for key in (
            "chunk_id", "evidence_ref", "ref", "id", "citation",
            "chunk_ids", "evidence_refs",
        ):
            values.extend(_flatten_evidence_values(value.get(key)))
        return values
    if isinstance(value, (list, tuple, set)):
        values = []
        for item in value:
            values.extend(_flatten_evidence_values(item))
        return values

    text = _clean_text(value)
    if not text:
        return []
    # "[C1, C2]" 또는 "C1 / C2" 같은 한 문자열도 분리합니다.
    refs = re.findall(r"(?i)(?:\[\s*)?C\s*0*(\d+)(?:\s*\])?", text)
    if refs:
        return [f"C{int(number)}" for number in refs]
    return [text.strip("[](){} \t\r\n\"'")]


def _lexical_units(text: str) -> set[str]:
    cleaned = re.sub(r"[^0-9A-Za-z가-힣]", "", _clean_text(text).lower())
    if not cleaned:
        return set()
    units = {cleaned[index:index + 2] for index in range(max(1, len(cleaned) - 1))}
    units.update(re.findall(r"[0-9A-Za-z가-힣]{2,}", _clean_text(text).lower()))
    return units


def _infer_evidence_chunk_ids(
    claim: str,
    search_results: list[dict[str, Any]],
    evidence_pack: list[dict[str, Any]] | None = None,
) -> list[str]:
    """근거 ID 누락 시 주장과 가장 겹치는 실제 Evidence 청크 하나를 연결합니다."""
    claim_units = _lexical_units(claim)
    if not claim_units:
        return []

    candidates: list[tuple[str, str]] = []
    if evidence_pack:
        for evidence in evidence_pack:
            context_chunks = evidence.get("context_chunks")
            if isinstance(context_chunks, list):
                for chunk in context_chunks:
                    if not isinstance(chunk, dict):
                        continue
                    chunk_id = _clean_text(chunk.get("chunk_id"))
                    candidate_text = " ".join(
                        _clean_text(value)
                        for value in (
                            chunk.get("title"),
                            chunk.get("section_title"),
                            chunk.get("content"),
                        )
                        if value
                    )
                    if chunk_id and candidate_text:
                        candidates.append((chunk_id, candidate_text))
            else:
                chunk_id = _clean_text(evidence.get("chunk_id"))
                candidate_text = " ".join(
                    _clean_text(value)
                    for value in (
                        evidence.get("title"),
                        evidence.get("section_title"),
                        evidence.get("content"),
                    )
                    if value
                )
                if chunk_id and candidate_text:
                    candidates.append((chunk_id, candidate_text))

    if not candidates:
        for result in search_results:
            original = result.get("chunk") or {}
            chunk_id = str(result.get("chunk_id") or "")
            candidate_text = " ".join(
                _clean_text(value)
                for value in (
                    original.get("title"),
                    original.get("section_title"),
                    original.get("content"),
                )
                if value
            )
            if chunk_id and candidate_text:
                candidates.append((chunk_id, candidate_text))

    best_chunk_id = ""
    best_score = 0.0
    for chunk_id, candidate_text in candidates:
        candidate_units = _lexical_units(candidate_text)
        if not candidate_units:
            continue
        score = len(claim_units & candidate_units) / max(1, len(claim_units))
        if score > best_score:
            best_score = score
            best_chunk_id = chunk_id

    return [best_chunk_id] if best_chunk_id and best_score >= 0.08 else []


def validate_answer_skeleton(
    raw_skeleton: dict[str, Any],
    search_results: list[dict[str, Any]],
    evidence_pack: list[dict[str, Any]] | None = None,
) -> dict[str, Any]:
    chunk_to_ref: dict[str, str] = {}
    representative_chunk_by_ref: dict[str, str] = {}

    for evidence in evidence_pack or []:
        ref = _clean_text(evidence.get("evidence_ref")).upper()
        if not ref:
            continue
        candidate_ids: list[str] = []
        context_chunks = evidence.get("context_chunks")
        if isinstance(context_chunks, list):
            candidate_ids.extend(
                _clean_text(chunk.get("chunk_id"))
                for chunk in context_chunks
                if isinstance(chunk, dict)
            )
        candidate_ids.extend(_as_clean_string_list(evidence.get("matched_child_ids")))
        legacy_chunk_id = _clean_text(evidence.get("chunk_id"))
        if legacy_chunk_id:
            candidate_ids.append(legacy_chunk_id)

        for chunk_id in candidate_ids:
            if chunk_id:
                chunk_to_ref[chunk_id] = ref
        if candidate_ids:
            representative_chunk_by_ref.setdefault(ref, next(
                chunk_id for chunk_id in candidate_ids if chunk_id
            ))

    if not chunk_to_ref:
        for result in search_results:
            chunk_id = str(result["chunk_id"])
            ref = _clean_text(result.get("parent_evidence_ref")) or f"C{int(result['rank'])}"
            chunk_to_ref[chunk_id] = ref.upper()
            representative_chunk_by_ref.setdefault(ref.upper(), chunk_id)

    ref_to_chunk = dict(representative_chunk_by_ref)
    chunk_casefold = {chunk_id.casefold(): chunk_id for chunk_id in chunk_to_ref}
    validation_warnings: list[str] = []

    answer_type = _clean_text(raw_skeleton.get("answer_type")).upper()
    if answer_type not in ALLOWED_ANSWER_TYPES:
        answer_type = "OTHER"

    coverage_status = _clean_text(raw_skeleton.get("coverage_status")).upper()
    if coverage_status not in ALLOWED_COVERAGE_STATUS:
        coverage_status = "PARTIAL"

    core_answer = _clean_text(
        _first_present(
            raw_skeleton,
            ("core_answer", "answer", "summary", "conclusion", "direct_answer"),
        )
    )

    raw_items = _first_present(
        raw_skeleton,
        ("answer_items", "items", "key_points", "claims", "facts"),
    )
    if isinstance(raw_items, dict):
        raw_items = [raw_items]
    if not isinstance(raw_items, list):
        raw_items = []

    normalized_items: list[dict[str, Any]] = []
    rejected_reasons: list[str] = []
    for index, raw_item in enumerate(raw_items, start=1):
        if isinstance(raw_item, str):
            item: dict[str, Any] = {"claim": raw_item}
        elif isinstance(raw_item, dict):
            item = raw_item
        else:
            rejected_reasons.append(f"A{index}: 객체나 문자열이 아님")
            continue

        topic = _clean_text(
            _first_present(item, ("topic", "title", "category", "name"))
        ) or "핵심 내용"
        claim = _clean_text(
            _first_present(
                item,
                ("claim", "statement", "fact", "content", "answer", "key_fact"),
            )
        )
        if not claim:
            rejected_reasons.append(f"A{index}: 주장 텍스트가 비어 있음")
            continue

        evidence_value = _first_present(
            item,
            (
                "evidence_chunk_ids", "evidence_refs", "citations", "sources",
                "chunk_ids", "evidence_ids", "evidence",
            ),
        )
        resolved_chunk_ids: list[str] = []
        for raw_value in _flatten_evidence_values(evidence_value):
            token = re.sub(r"\s+", "", raw_value.strip("[](){} \t\r\n\"'"))
            upper_token = token.upper()
            if token in chunk_to_ref:
                resolved_chunk_ids.append(token)
            elif token.casefold() in chunk_casefold:
                resolved_chunk_ids.append(chunk_casefold[token.casefold()])
            elif upper_token in ref_to_chunk:
                resolved_chunk_ids.append(ref_to_chunk[upper_token])

        resolved_chunk_ids = list(dict.fromkeys(resolved_chunk_ids))
        if not resolved_chunk_ids:
            resolved_chunk_ids = _infer_evidence_chunk_ids(claim, search_results, evidence_pack)
            if resolved_chunk_ids:
                validation_warnings.append(
                    f"A{index}: 근거 ID가 없어 텍스트 중첩으로 "
                    f"{resolved_chunk_ids[0]}을 연결함"
                )
        if not resolved_chunk_ids:
            rejected_reasons.append(f"A{index}: Top 5에서 연결 가능한 근거가 없음")
            continue

        normalized_items.append({
            "item_id": f"A{len(normalized_items) + 1}",
            "topic": topic,
            "claim": claim,
            "conditions": _as_clean_string_list(
                _first_present(item, ("conditions", "condition", "requirements"))
            ),
            "result": _clean_text(
                _first_present(item, ("result", "outcome", "effect"))
            ),
            "detail_points": _as_clean_string_list(
                _first_present(item, ("detail_points", "details", "additional_details"))
            ),
            "term_explanations": _normalize_term_explanations(
                _first_present(item, ("term_explanations", "terms", "glossary"))
            ),
            "evidence_chunk_ids": resolved_chunk_ids,
            "evidence_refs": list(dict.fromkeys(
                chunk_to_ref[chunk_id] for chunk_id in resolved_chunk_ids
            )),
        })

    if not core_answer and normalized_items:
        core_answer = normalized_items[0]["claim"]
        validation_warnings.append("core_answer가 없어 첫 answer_item의 claim으로 보정함")

    if not normalized_items and core_answer:
        inferred_ids = _infer_evidence_chunk_ids(core_answer, search_results, evidence_pack)
        if not inferred_ids and chunk_to_ref:
            # 마지막 복구 경로: 특정 청크 하나를 임의 지정하지 않고
            # 각 Evidence 그룹의 대표 청크를 연결합니다.
            inferred_ids = list(dict.fromkeys(representative_chunk_by_ref.values()))
            validation_warnings.append(
                "answer_items 복구를 위해 core_answer에 전체 Evidence 그룹을 연결함"
            )
        if inferred_ids:
            normalized_items.append({
                "item_id": "A1",
                "topic": "핵심 답변",
                "claim": core_answer,
                "conditions": [],
                "result": "",
                "detail_points": [],
                "term_explanations": [],
                "evidence_chunk_ids": inferred_ids,
                "evidence_refs": list(dict.fromkeys(
                    chunk_to_ref[chunk_id] for chunk_id in inferred_ids
                )),
            })
            validation_warnings.append(
                "유효한 answer_items가 없어 core_answer로 최소 항목을 복구함"
            )

    if rejected_reasons:
        validation_warnings.extend(rejected_reasons)

    if not core_answer:
        raise ValueError(
            "HCX Skeleton에 core_answer와 복구 가능한 answer_item이 모두 없습니다. "
            f"원본 키={list(raw_skeleton.keys())}"
        )
    if not normalized_items:
        raise ValueError(
            "HCX Skeleton의 주장과 Top 5 근거를 연결하지 못했습니다. "
            f"검증 내역={validation_warnings}"
        )

    return {
        "answer_type": answer_type,
        "core_answer": core_answer,
        "answer_items": normalized_items,
        "uncertainties": _as_clean_string_list(raw_skeleton.get("uncertainties")),
        "conflicts": _as_clean_string_list(raw_skeleton.get("conflicts")),
        "coverage_status": coverage_status,
        "validation_warnings": validation_warnings,
    }

def _remove_model_generated_urls(answer: str) -> str:
    answer = re.sub(r"\[([^\]]+)\]\(https?://[^)]+\)", r"\1", answer)
    answer = re.sub(r"https?://[^\s)\]}>]+", "", answer)
    answer = re.sub(r"[ \t]+\n", "\n", answer)
    answer = re.sub(r"\n{3,}", "\n\n", answer)
    return answer.strip()


def _remove_invalid_chunk_citations(answer: str, result_count: int) -> str:
    allowed = {f"C{index}" for index in range(1, result_count + 1)}

    def replace(match: re.Match[str]) -> str:
        citation = f"C{match.group(1)}"
        return match.group(0) if citation in allowed else ""

    answer = re.sub(r"\[C(\d+)\]", replace, answer)
    answer = re.sub(r"[ \t]+\n", "\n", answer)
    answer = re.sub(r"\n{3,}", "\n\n", answer)
    return answer.strip()


def generate_answer_skeleton(
    question: str,
    search_results: list[dict[str, Any]],
    evidence_pack: list[dict[str, Any]],
) -> dict[str, Any]:
    if not search_results:
        raise ValueError("Answer Skeleton을 생성할 검색 결과가 없습니다.")

    evidence_pack_text = format_evidence_pack(evidence_pack)
    user_prompt = f"""
    [사용자 질문]
    {_clean_text(question)}

    [Evidence Pack JSON]
    {evidence_pack_text}

    [출력 JSON 스키마]
    {{
      "answer_type": "INFORMATION_ANSWER | ACTION_GUIDE | ELIGIBILITY_ANSWER | AMOUNT_DEADLINE_ANSWER | COMPARISON_ANSWER | OTHER",
      "core_answer": "질문에 대한 핵심 결론",
      "answer_items": [
        {{
          "item_id": "A1",
          "topic": "대상·조건·금액·기한·절차·예외 등",
          "claim": "답변에 반드시 포함할 원문 기반 사실",
          "conditions": ["이 주장이 적용되는 원문 기반 조건"],
          "result": "이 조건이 적용될 때의 원문 기반 결과",
          "detail_points": ["전문가 설명에서 사용할 원문 기반 세부사항"],
          "term_explanations": [
            {{
              "term": "원문에 나온 공식 용어",
              "plain_explanation": "Evidence Pack에 근거가 있는 쉬운 풀이"
            }}
          ],
          "evidence_chunk_ids": ["Evidence Pack의 context_chunks 안에 있는 실제 chunk_id"]
        }}
      ],
      "uncertainties": ["Top 5로 확인할 수 없는 필수 내용"],
      "conflicts": ["청크 사이에서 충돌하는 내용"],
      "coverage_status": "SUFFICIENT | PARTIAL | INSUFFICIENT"
    }}

    JSON 객체 하나만 출력하세요.
    """.strip()

    try:
        response = HCX_CLIENT.chat.completions.create(
            model=HCX_CHAT_MODEL,
            messages=[
                {"role": "system", "content": ANSWER_SKELETON_SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.0,
            max_tokens=2000,
        )
    except BadRequestError as error:
        raise RuntimeError(
            "HCX가 Answer Skeleton 생성 요청을 거부했습니다. Top 5 전체 길이가 "
            f"모델 입력 한도를 넘었는지 확인하세요. 원인={error}"
        ) from error

    content = response.choices[0].message.content
    if not content or not content.strip():
        raise RuntimeError("HCX Answer Skeleton 출력이 비어 있습니다.")
    return validate_answer_skeleton(_extract_json_object(content), search_results, evidence_pack)


def generate_answer_from_skeleton(
    question: str,
    answer_skeleton: dict[str, Any],
    evidence_pack: list[dict[str, Any]],
    *,
    result_count: int,
    answer_mode: Literal["easy", "detailed"] = "easy",
    basic_answer: str | None = None,
) -> str:
    if answer_mode not in {"easy", "detailed"}:
        raise ValueError(f"지원하지 않는 answer_mode: {answer_mode}")

    style_prompt = EASY_STYLE_PROMPT if answer_mode == "easy" else DETAILED_STYLE_PROMPT
    max_tokens = 1200 if answer_mode == "easy" else 1800
    skeleton_text = json.dumps(answer_skeleton, ensure_ascii=False, indent=2)
    evidence_pack_text = format_evidence_pack(evidence_pack)
    basic_answer_section = ""
    if answer_mode == "detailed":
        if not _clean_text(basic_answer):
            raise ValueError("전문가 설명 생성에는 이미 생성된 기본 설명이 필요합니다.")
        basic_answer_section = f"""

        [이미 생성된 기본 설명]
        {_clean_text(basic_answer)}
        """.rstrip()
    user_prompt = f"""
    [사용자 질문]
    {_clean_text(question)}

    [Answer Skeleton JSON]
    {skeleton_text}

    [동일 Evidence Pack JSON]
    {evidence_pack_text}
    {basic_answer_section}

    [출력 방식]
    {style_prompt}

    위 Answer Skeleton과 Evidence Pack의 근거 범위 안에서 최종 답변을 작성하세요.
    """.strip()

    response = HCX_CLIENT.chat.completions.create(
        model=HCX_CHAT_MODEL,
        messages=[
            {"role": "system", "content": FINAL_ANSWER_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
        max_tokens=max_tokens,
    )
    answer = response.choices[0].message.content
    if not answer or not answer.strip():
        raise RuntimeError("HCX 최종 답변 본문이 비어 있습니다.")
    answer = _remove_model_generated_urls(answer.strip())
    return _remove_invalid_chunk_citations(answer, result_count)


print("D안 Answer Skeleton 및 최종 답변 생성기 준비 완료")


In [ ]:
def _ordered_unique_strings(values: Iterable[Any]) -> list[str]:
    seen: set[str] = set()
    ordered: list[str] = []
    for value in values:
        text = str(value or "").strip()
        if text and text not in seen:
            seen.add(text)
            ordered.append(text)
    return ordered


def build_rule_based_sources(search_results: list[dict[str, Any]]) -> list[dict[str, Any]]:
    by_url: OrderedDict[str, dict[str, Any]] = OrderedDict()
    for result in search_results:
        chunk = result["chunk"]
        rank = int(result["rank"])
        evidence_ref = (
            str(result.get("parent_evidence_ref") or "").strip()
            or f"C{rank}"
        )
        for key, source_type in (
            ("source_url", "공식 페이지"),
            ("official_download_url", "공식 첨부파일"),
        ):
            url = str(chunk.get(key) or "").strip()
            if not url:
                continue
            row = by_url.setdefault(url, {
                "url": url,
                "source_type": source_type,
                "title": (
                    str(chunk.get("title") or "").strip()
                    or str(chunk.get("section_title") or "").strip()
                    or str(chunk.get("document_id") or "").strip()
                ),
                "chunk_citations": [],
            })
            citation = evidence_ref
            if citation not in row["chunk_citations"]:
                row["chunk_citations"].append(citation)
    return list(by_url.values())


def sources_to_markdown(sources: list[dict[str, Any]]) -> str:
    if not sources:
        return "### 출처\n\nTop 5 청크에서 공식 URL을 확인하지 못했습니다."
    lines = ["### 출처", ""]
    for source in sources:
        citations = ", ".join(source["chunk_citations"])
        title = str(source["title"] or "공식 출처").replace("[", "").replace("]", "")
        lines.append(
            f"- [{title}]({source['url']}) — {source['source_type']} ({citations})"
        )
    return "\n".join(lines)


def retrieval_table_markdown(results: list[dict[str, Any]]) -> str:
    lines = [
        "### 검색 결과",
        "",
        "|순위|청크 ID|Dense 순위|BM25 순위|Weighted RRF|제목 / 소제목|",
        "|---:|---|---:|---:|---:|---|",
    ]
    for result in results:
        chunk = result["chunk"]
        dense_rank = result["dense_rank"] if result["dense_rank"] is not None else "-"
        bm25_rank = result["bm25_rank"] if result["bm25_rank"] is not None else "-"
        title = " / ".join(
            part
            for part in [
                str(chunk.get("title") or "").replace("|", "\\|"),
                str(chunk.get("section_title") or "").replace("|", "\\|"),
            ]
            if part
        )
        lines.append(
            f"|{result['rank']}|{result['chunk_id']}|{dense_rank}|{bm25_rank}|"
            f"{result['rrf_score']:.6f}|{title}|"
        )
    return "\n".join(lines)


## 4. V1.5 → 검색 → 답변 D 통합

In [ ]:
import time
import uuid

from kdic_decomposition_quality_core import BASELINE
from kdic_hcx007_resumable_decomposition_core import (
    HCX_DECOMPOSITION_ENDPOINT,
    ResumableBaselineDecomposer,
    TransportPolicy,
    _condition_record,
    _normalize_baseline_record,
)
from kdic_lightweight_query_ablation_core import AblationConfig, analyze_common


v15_transport_policy = TransportPolicy(
    request_delay_seconds=0.0,
    max_transport_retries=4,
    base_backoff_seconds=5.0,
    max_backoff_seconds=120.0,
    jitter_seconds=1.0,
    consecutive_429_cooldown_threshold=1,
    cooldown_seconds=65.0,
    timeout_seconds=HCX_REQUEST_TIMEOUT,
)
v15_decomposition_config = AblationConfig(
    llm_model=HCX_DECOMPOSITION_MODEL,
    llm_endpoint=HCX_DECOMPOSITION_ENDPOINT,
    llm_timeout_seconds=HCX_REQUEST_TIMEOUT,
    llm_min_confidence=V15_MIN_CONFIDENCE,
    max_subqueries=V15_MAX_SUBQUERIES,
)
V15_DECOMPOSER = ResumableBaselineDecomposer(
    HCX_API_KEY,
    cache_path=V15_CACHE_PATH,
    seed_cache_paths=[],
    config=v15_decomposition_config,
    transport_policy=v15_transport_policy,
)


def _route_response(route: str, common: dict[str, Any]) -> str:
    if route == "DIRECT_RESPONSE":
        return "안녕하세요. 예금보험공사 관련 제도와 신청 절차에 관해 질문해 주세요."
    if route == "OUT_OF_SCOPE":
        return "이 챗봇은 예금자보호, 예금보험금, 고객 미수령금, 착오송금 반환지원, 채무조정, 은닉재산 신고 관련 질문에 답변합니다."
    missing = common.get("missing_information") or []
    detail = " / ".join(str(value) for value in missing if str(value).strip())
    if detail:
        return f"정확한 안내를 위해 정보가 더 필요합니다: {detail}"
    return "어떤 업무에 관한 질문인지 선택해 주세요: 예금자보호, 예금보험금, 고객 미수령금, 착오송금 반환지원, 채무조정, 은닉재산 신고."


def analyze_v15_chat_query(question: str, previous_turns: Any = None) -> dict[str, Any]:
    started = time.perf_counter()
    common = analyze_common(
        f"CHAT_{uuid.uuid4().hex[:12]}",
        question,
        previous_turns=previous_turns,
    )
    route = str(common["route"])
    businesses = list(dict.fromkeys((common.get("complexity") or {}).get("businesses") or []))
    cross_candidate = bool(
        route == "RETRIEVE"
        and common.get("complex_candidate")
        and len(businesses) >= 2
    )
    record = None
    decomposition_latency_ms = 0.0
    if cross_candidate:
        decomposition_started = time.perf_counter()
        first = _normalize_baseline_record(
            common["normalized_question"],
            businesses,
            V15_DECOMPOSER.decompose(common["normalized_question"], businesses),
        )
        record = _condition_record(BASELINE, first, first, retry_called=False)
        decomposition_latency_ms = (time.perf_counter() - decomposition_started) * 1000

    accepted = bool((record or {}).get("final_accepted"))
    subqueries = list((record or {}).get("final_subqueries") or []) if accepted else []
    if route != "RETRIEVE":
        plans = []
    elif subqueries:
        sub_weight = V15_SUBQUERY_TOTAL_WEIGHT / len(subqueries)
        plans = [{
            "query": common["original_question"],
            "weight": V15_ORIGINAL_WEIGHT,
            "source": "ORIGINAL_ANCHOR",
        }]
        plans.extend({
            "query": query,
            "weight": sub_weight,
            "source": "DECOMPOSED",
        } for query in subqueries)
    else:
        plans = [{
            "query": common["original_question"],
            "weight": 1.0,
            "source": "ORIGINAL",
        }]

    return {
        "route": route,
        "route_reasons": common.get("route_reasons") or [],
        "businesses": businesses,
        "complexity": (common.get("complexity") or {}).get("question_type", "NONE"),
        "cross_business_candidate": cross_candidate,
        "decomposition_called": cross_candidate,
        "decomposition_accepted": accepted,
        "decomposition_status": (record or {}).get("final_status", "NOT_CALLED"),
        "decomposition_issues": (record or {}).get("final_issues") or [],
        "subqueries": subqueries,
        "fallback_to_original": bool(cross_candidate and not accepted),
        "plans": plans,
        "route_response": _route_response(route, common) if route != "RETRIEVE" else "",
        "routing_latency_ms": float(common.get("common_latency_ms") or 0.0),
        "rule_latency_ms": float(common.get("rule_latency_ms") or 0.0),
        "decomposition_latency_ms": decomposition_latency_ms,
        "analysis_wall_latency_ms": (time.perf_counter() - started) * 1000,
    }


def retrieval_table_markdown(results: list[dict[str, Any]]) -> str:
    lines = [
        "### 검색 결과",
        "",
        "|순위|청크 ID|Dense 순위|BM25 순위|Min-Max 최고점|질의결합점수|제목 / 소제목|",
        "|---:|---|---:|---:|---:|---:|---|",
    ]
    for result in results:
        chunk = result["chunk"]
        dense_rank = result.get("dense_rank") or "-"
        bm25_rank = result.get("bm25_rank") or "-"
        title = " / ".join(
            part for part in [
                str(chunk.get("title") or "").replace("|", "\\|"),
                str(chunk.get("section_title") or "").replace("|", "\\|"),
            ] if part
        )
        lines.append(
            f"|{result['rank']}|{result['chunk_id']}|{dense_rank}|{bm25_rank}|"
            f"{float(result.get('minmax_score') or 0):.6f}|"
            f"{float(result.get('query_fusion_score') or 0):.6f}|{title}|"
        )
    return "\n".join(lines)


def analysis_markdown(analysis: dict[str, Any]) -> str:
    plans = analysis.get("plans") or []
    plan_text = "<br>".join(
        f"{index}. {plan['source']} · {plan['weight']:.3f} · {plan['query']}"
        for index, plan in enumerate(plans, start=1)
    ) or "검색 계획 없음"
    return (
        "### V1.5 질의분석\n\n"
        "|항목|결과|\n|---|---|\n"
        f"|최종 경로|{analysis['route']}|\n"
        f"|탐지 업무|{', '.join(analysis['businesses']) or '-'}|\n"
        f"|복합 후보|{analysis['complexity']}|\n"
        f"|교차업무 분해 호출|{analysis['decomposition_called']}|\n"
        f"|분해 승인|{analysis['decomposition_accepted']}|\n"
        f"|원문 fallback|{analysis['fallback_to_original']}|\n"
        f"|검색 계획|{plan_text}|"
    )


def latency_markdown(latency: dict[str, float]) -> str:
    return (
        "### 단계별 지연시간\n\n"
        "|단계|지연시간|\n|---|---:|\n"
        + "\n".join(f"|{key}|{value:,.1f}ms|" for key, value in latency.items())
    )


def ask_v15(
    question: str,
    *,
    previous_turns: Any = None,
    show_analysis: bool = True,
    show_retrieval: bool = True,
    show_answer_skeleton: bool = False,
) -> dict[str, Any]:
    global _LAST_PARENT_CHILD_TRACE
    total_started = time.perf_counter()
    _LAST_PARENT_CHILD_TRACE = {}
    analysis = analyze_v15_chat_query(question, previous_turns=previous_turns)
    if analysis["route"] != "RETRIEVE":
        result = {
            "answer_method": "V1.5_ROUTE_ONLY",
            "question": question,
            "route": analysis["route"],
            "basic_answer": analysis["route_response"],
            "analysis": analysis,
            "search_results": [],
            "sources": [],
            "latency_ms": {
                "질의분석": analysis["analysis_wall_latency_ms"],
                "전체": (time.perf_counter() - total_started) * 1000,
            },
        }
        display(Markdown(f"## 답변\n\n{result['basic_answer']}"))
        if show_analysis:
            display(Markdown(analysis_markdown(analysis)))
        display(Markdown(latency_markdown(result["latency_ms"])))
        return result

    search_started = time.perf_counter()
    search_results, per_query = fuse_query_results(analysis["plans"])
    child_search_latency_ms = (time.perf_counter() - search_started) * 1000

    parent_started = time.perf_counter()
    search_results = expand_parent_context(search_results)
    parent_child_latency_ms = (time.perf_counter() - parent_started) * 1000
    # Hybrid + 다중질의 결합 + Reranker + Parent 확장을 검색 단계 전체로 봅니다.
    search_latency_ms = child_search_latency_ms + parent_child_latency_ms

    evidence_started = time.perf_counter()
    evidence_pack = build_evidence_pack(search_results)
    evidence_latency_ms = (time.perf_counter() - evidence_started) * 1000

    skeleton_started = time.perf_counter()
    answer_skeleton = generate_answer_skeleton(question, search_results, evidence_pack)
    skeleton_latency_ms = (time.perf_counter() - skeleton_started) * 1000

    answer_started = time.perf_counter()
    basic_answer = generate_answer_from_skeleton(
        question,
        answer_skeleton,
        evidence_pack,
        result_count=len(evidence_pack),
        answer_mode="easy",
    )
    answer_latency_ms = (time.perf_counter() - answer_started) * 1000
    sources = build_rule_based_sources(search_results)
    latency = {
        "질의분석": analysis["analysis_wall_latency_ms"],
        "검색": search_latency_ms,
        "Parent-Child 확장": parent_child_latency_ms,
        "Evidence Pack": evidence_latency_ms,
        "Answer Skeleton": skeleton_latency_ms,
        "최종 답변": answer_latency_ms,
        "전체": (time.perf_counter() - total_started) * 1000,
    }
    result = {
        "answer_method": "V1.5_M3_7_3_MINMAX_BGE_RERANKER_PARENT_CHILD_D_ANSWER_SKELETON",
        "question": question,
        "route": analysis["route"],
        "basic_answer": basic_answer,
        "easy_answer": basic_answer,
        "detailed_answer": None,
        "analysis": analysis,
        "search_plans": analysis["plans"],
        "per_query_search": per_query,
        "search_results": search_results,
        "parent_child": dict(_LAST_PARENT_CHILD_TRACE),
        "evidence_pack": evidence_pack,
        "answer_skeleton": answer_skeleton,
        "sources": sources,
        "raw_top5_text": format_evidence_pack(evidence_pack),
        "raw_parent_evidence_text": format_evidence_pack(evidence_pack),
        "latency_ms": latency,
    }
    display(Markdown(f"## 기본 설명\n\n{basic_answer}"))
    display(Markdown(sources_to_markdown(sources)))
    if show_analysis:
        display(Markdown(analysis_markdown(analysis)))
    if show_answer_skeleton:
        display(JSON(answer_skeleton, expanded=False))
    if show_retrieval:
        display(Markdown(retrieval_table_markdown(search_results)))
        display(Markdown(parent_child_markdown(search_results)))
    display(Markdown(latency_markdown(latency)))
    return result


print("ask_v15() 통합 파이프라인 준비 완료")


### 4-1. 질의분석부터 답변까지 상세 레이턴시

In [ ]:
_ask_v15_coarse_latency = ask_v15


def per_query_latency_markdown(rows: list[dict[str, Any]]) -> str:
    if not rows:
        return "### 질의별 검색 레이턴시\n\n검색을 실행하지 않았습니다."
    lines = [
        "### 질의별 검색 레이턴시",
        "",
        "|계획|출처|가중치|질문 임베딩|Dense 계산|BM25|Min-Max|질의 전체|검색 질의|",
        "|---:|---|---:|---:|---:|---:|---:|---:|---|",
    ]
    for row in rows:
        trace = row.get("latency_breakdown_ms") or {}
        query = str(row.get("query") or "").replace("|", "\\|")
        lines.append(
            f"|{row.get('plan_index', '-')}|{row.get('source', '-')}|{float(row.get('weight') or 0):.3f}|"
            f"{float(trace.get('embedding_latency_ms') or 0):,.1f}ms|"
            f"{float(trace.get('dense_compute_latency_ms') or 0):,.1f}ms|"
            f"{float(trace.get('bm25_latency_ms') or 0):,.1f}ms|"
            f"{float(trace.get('minmax_latency_ms') or 0):,.1f}ms|"
            f"{float(trace.get('query_total_latency_ms') or 0):,.1f}ms|{query}|"
        )
    return "\n".join(lines)


def detailed_latency_markdown(result: dict[str, Any]) -> str:
    latency = result.get("latency_ms") or {}
    rows = result.get("per_query_latency") or []
    analysis = result.get("analysis") or {}
    values = [
        ("질의분석", "규칙 라우팅", float(analysis.get("routing_latency_ms") or 0)),
        ("질의분석", "규칙 복합판정·분해", float(analysis.get("rule_latency_ms") or 0)),
        ("질의분석", "HCX-007 구조화 분해", float(analysis.get("decomposition_latency_ms") or 0)),
        ("질의분석", "질의분석 전체 Wall", float(analysis.get("analysis_wall_latency_ms") or 0)),
        ("검색", "질문 임베딩 합계", sum(float(r.get("embedding_latency_ms") or 0) for r in rows)),
        ("검색", "Dense 계산 합계", sum(float(r.get("dense_compute_latency_ms") or 0) for r in rows)),
        ("검색", "BM25 합계", sum(float(r.get("bm25_latency_ms") or 0) for r in rows)),
        ("검색", "Min-Max 합계", sum(float(r.get("minmax_latency_ms") or 0) for r in rows)),
        ("검색", "다중 질의 RRF 결합", float(rows[0].get("query_fusion_latency_ms") or 0) if rows else 0.0),
        ("검색", "BGE Reranker", float((_LAST_RERANK_TRACE or {}).get("latency_ms") or 0)),
        ("검색", "Parent-Child 문맥 확장", float((_LAST_PARENT_CHILD_TRACE or {}).get("latency_ms") or 0)),
        ("검색", "검색 전체 Wall", float(latency.get("검색") or 0)),
        ("답변", "Evidence Pack", float(latency.get("Evidence Pack") or 0)),
        ("답변", "Answer Skeleton · HCX-005", float(latency.get("Answer Skeleton") or 0)),
        ("답변", "최종 답변 · HCX-005", float(latency.get("최종 답변") or 0)),
        ("전체", "질문 입력부터 최종 완료", float(latency.get("전체") or 0)),
    ]
    lines = [
        "### 상세 단계별 레이턴시",
        "",
        "하위 단계는 상위 `전체 Wall` 구간에 포함되므로 서로 중복해 더하지 않습니다.",
        "",
        "|구간|측정 단계|지연시간|",
        "|---|---|---:|",
    ]
    lines.extend(f"|{group}|{stage}|{value:,.1f}ms|" for group, stage, value in values)
    return "\n".join(lines)


def ask_v15(
    question: str,
    *,
    previous_turns: Any = None,
    show_analysis: bool = True,
    show_retrieval: bool = True,
    show_answer_skeleton: bool = False,
) -> dict[str, Any]:
    result = _ask_v15_coarse_latency(
        question,
        previous_turns=previous_turns,
        show_analysis=show_analysis,
        show_retrieval=show_retrieval,
        show_answer_skeleton=show_answer_skeleton,
    )
    context_latency_ms = float((_LAST_CONTEXT_TRACE or {}).get("latency_ms") or 0)
    reranker_latency_ms = float((_LAST_RERANK_TRACE or {}).get("latency_ms") or 0)
    parent_child_latency_ms = float((_LAST_PARENT_CHILD_TRACE or {}).get("latency_ms") or 0)
    result["latency_ms"]["문맥 해석"] = context_latency_ms
    result["latency_ms"]["Reranker"] = reranker_latency_ms
    result["latency_ms"]["Parent-Child 확장"] = parent_child_latency_ms
    result["latency_ms"]["전체"] = float(result["latency_ms"].get("전체") or 0) + context_latency_ms
    per_query_latency = []
    for row in result.get("per_query_search") or []:
        trace = dict(row.get("latency_breakdown_ms") or {})
        per_query_latency.append({
            "plan_index": row.get("plan_index"),
            "source": row.get("source"),
            "weight": float(row.get("weight") or 0),
            "query": row.get("query"),
            "embedding_latency_ms": float(trace.get("embedding_latency_ms") or 0),
            "dense_compute_latency_ms": float(trace.get("dense_compute_latency_ms") or 0),
            "bm25_latency_ms": float(trace.get("bm25_latency_ms") or 0),
            "minmax_latency_ms": float(trace.get("minmax_latency_ms") or 0),
            "query_total_latency_ms": float(trace.get("query_total_latency_ms") or 0),
            "query_fusion_latency_ms": float(row.get("query_fusion_latency_ms") or 0),
        })
    result["per_query_latency"] = per_query_latency
    result["detailed_latency_ms"] = {
        "context_resolution_ms": context_latency_ms,
        "routing_rule_ms": float(result["analysis"].get("routing_latency_ms") or 0),
        "complexity_rule_ms": float(result["analysis"].get("rule_latency_ms") or 0),
        "hcx007_decomposition_ms": float(result["analysis"].get("decomposition_latency_ms") or 0),
        "analysis_wall_ms": float(result["analysis"].get("analysis_wall_latency_ms") or 0),
        "embedding_sum_ms": sum(row["embedding_latency_ms"] for row in per_query_latency),
        "dense_compute_sum_ms": sum(row["dense_compute_latency_ms"] for row in per_query_latency),
        "bm25_sum_ms": sum(row["bm25_latency_ms"] for row in per_query_latency),
        "minmax_sum_ms": sum(row["minmax_latency_ms"] for row in per_query_latency),
        "query_fusion_ms": per_query_latency[0]["query_fusion_latency_ms"] if per_query_latency else 0.0,
        "reranker_ms": reranker_latency_ms,
        "parent_child_ms": parent_child_latency_ms,
        "search_wall_ms": float(result["latency_ms"].get("검색") or 0),
        "evidence_pack_ms": float(result["latency_ms"].get("Evidence Pack") or 0),
        "answer_skeleton_hcx005_ms": float(result["latency_ms"].get("Answer Skeleton") or 0),
        "final_answer_hcx005_ms": float(result["latency_ms"].get("최종 답변") or 0),
        "total_wall_ms": float(result["latency_ms"].get("전체") or 0),
    }
    display(Markdown(detailed_latency_markdown(result)))
    display(Markdown(per_query_latency_markdown(per_query_latency)))
    return result


print("ask_v15() 상세 레이턴시 래퍼 준비 완료")


### 4-2. 문맥 복원과 보수적 CLARIFY 게이트

`신청할 때 서류는?`처럼 업무가 빠진 짧은 질문은 바로 검색하지 않습니다. 직전 업무가 하나면 독립질의로 복원하고, 없거나 둘 이상이면 선택지를 제시한 뒤 사용자의 보충 답변을 원래 질문과 결합합니다.


In [ ]:
from kdic_v15_context_rerank_core import resolve_conversational_question

_ASK_V15_BEFORE_CONTEXT = ask_v15
_LAST_CONTEXT_TRACE: dict[str, Any] = {}


def context_resolution_markdown(resolution: dict[str, Any]) -> str:
    original = str(resolution.get("original_question") or "").replace("|", chr(92) + "|")
    resolved = str(resolution.get("resolved_question") or "-").replace("|", chr(92) + "|")
    businesses = ", ".join(resolution.get("context_businesses") or []) or "-"
    return (
        "### 대화 문맥 처리\n\n"
        "|항목|값|\n|---|---|\n"
        f"|사용자 원문|{original}|\n"
        f"|검색용 독립질의|{resolved}|\n"
        f"|문맥 사용|{bool(resolution.get('context_used'))}|\n"
        f"|문맥 업무|{businesses}|\n"
        f"|판단 근거|{resolution.get('resolution_reason')}|"
    )


def retrieval_table_markdown(results: list[dict[str, Any]]) -> str:
    lines = [
        "### Reranker 적용 검색 결과",
        "",
        "|최종 순위|이전 순위|Child 청크 ID|Evidence|Parent|Reranker|질의결합점수|Min-Max|제목 / 섹션|",
        "|---:|---:|---|---|---|---:|---:|---:|---|",
    ]
    for result in results:
        chunk = result["chunk"]
        title = " / ".join(
            part for part in [
                str(chunk.get("title") or "").replace("|", chr(92) + "|"),
                str(chunk.get("section_title") or "").replace("|", chr(92) + "|"),
            ] if part
        )
        lines.append(
            f"|{result['rank']}|{result.get('pre_rerank_rank', '-')}|{result['chunk_id']}|"
            f"{result.get('parent_evidence_ref', '-')}|{result.get('parent_doc_id', '-')}|"
            f"{float(result.get('reranker_score') or 0):.6f}|"
            f"{float(result.get('query_fusion_score') or 0):.6f}|"
            f"{float(result.get('minmax_score') or 0):.6f}|{title}|"
        )
    return "\n".join(lines)


def ask_v15(
    question: str,
    *,
    previous_turns: Any = None,
    conversation_state: dict[str, Any] | None = None,
    show_analysis: bool = True,
    show_retrieval: bool = True,
    show_answer_skeleton: bool = False,
) -> dict[str, Any]:
    global _LAST_CONTEXT_TRACE, _LAST_RERANK_TRACE, _LAST_PARENT_CHILD_TRACE
    total_started = time.perf_counter()
    state = conversation_state if isinstance(conversation_state, dict) else {}
    turns = previous_turns if previous_turns is not None else state.get("turns", [])[-6:]
    resolution = resolve_conversational_question(
        question,
        previous_turns=turns,
        pending_clarification=state.get("pending_clarification"),
        confirmed_businesses=state.get("confirmed_businesses"),
    )
    _LAST_CONTEXT_TRACE = dict(resolution)
    _LAST_RERANK_TRACE = {}
    _LAST_PARENT_CHILD_TRACE = {}

    if resolution["route"] == "CLARIFY":
        state["pending_clarification"] = resolution["pending_clarification"]
        answer = resolution["clarification_message"]
        total_latency_ms = (time.perf_counter() - total_started) * 1000
        result = {
            "answer_method": "V1.5_CONTEXT_CLARIFY",
            "question": question,
            "original_question": question,
            "resolved_question": "",
            "route": "CLARIFY",
            "basic_answer": answer,
            "easy_answer": answer,
            "analysis": {
                "route": "CLARIFY",
                "businesses": resolution.get("context_businesses") or [],
                "resolution_reason": resolution["resolution_reason"],
                "routing_latency_ms": resolution["latency_ms"],
                "rule_latency_ms": 0.0,
                "decomposition_latency_ms": 0.0,
                "analysis_wall_latency_ms": resolution["latency_ms"],
            },
            "context_resolution": resolution,
            "pending_clarification": resolution["pending_clarification"],
            "search_plans": [],
            "search_results": [],
            "sources": [],
            "latency_ms": {
                "문맥 해석": float(resolution["latency_ms"]),
                "전체": total_latency_ms,
            },
            "detailed_latency_ms": {
                "context_resolution_ms": float(resolution["latency_ms"]),
                "reranker_ms": 0.0,
                "parent_child_ms": 0.0,
                "total_wall_ms": total_latency_ms,
            },
        }
        display(Markdown(f"## 추가 정보 필요\n\n{answer}"))
        display(Markdown(context_resolution_markdown(resolution)))
        return result

    state["pending_clarification"] = None
    resolved_question = resolution["resolved_question"]
    result = _ASK_V15_BEFORE_CONTEXT(
        resolved_question,
        previous_turns=turns,
        show_analysis=show_analysis,
        show_retrieval=show_retrieval,
        show_answer_skeleton=show_answer_skeleton,
    )
    result["question"] = question
    result["original_question"] = question
    result["resolved_question"] = resolved_question
    result["context_resolution"] = resolution
    result["pending_clarification"] = None
    result["reranker"] = dict(_LAST_RERANK_TRACE)
    result["parent_child"] = dict(_LAST_PARENT_CHILD_TRACE)
    if result.get("route") == "RETRIEVE":
        businesses = list((result.get("analysis") or {}).get("businesses") or [])
        if businesses:
            state["confirmed_businesses"] = businesses
    result["latency_ms"]["전체"] = (time.perf_counter() - total_started) * 1000
    result["detailed_latency_ms"]["total_wall_ms"] = result["latency_ms"]["전체"]
    display(Markdown(context_resolution_markdown(resolution)))
    return result


print("문맥 복원 + CLARIFY 상태 관리 준비 완료")


## 5. 간이 챗봇 입력창

### 상세 레이턴시 필드 검사

In [ ]:
# API 호출 없이 출력 필드 구조만 검사합니다.
_latency_schema_required = {
    "context_resolution_ms",
    "routing_rule_ms",
    "complexity_rule_ms",
    "hcx007_decomposition_ms",
    "analysis_wall_ms",
    "embedding_sum_ms",
    "dense_compute_sum_ms",
    "bm25_sum_ms",
    "minmax_sum_ms",
    "query_fusion_ms",
    "reranker_ms",
    "search_wall_ms",
    "evidence_pack_ms",
    "answer_skeleton_hcx005_ms",
    "final_answer_hcx005_ms",
    "total_wall_ms",
}
print("상세 레이턴시 필드 수:", len(_latency_schema_required))

In [ ]:
def launch_v15_chat() -> dict[str, Any]:
    import html

    question_input = widgets.Textarea(
        placeholder="예: 예금자보호 한도와 착오송금 반환지원 신청 방법을 알려줘.",
        description="질문",
        layout=widgets.Layout(width="100%", height="92px"),
        style={"description_width": "55px"},
    )
    send_button = widgets.Button(description="질문 전송", button_style="success", icon="paper-plane")
    clear_button = widgets.Button(description="대화 지우기", icon="trash")
    status = widgets.HTML("질문을 입력한 뒤 <b>질문 전송</b>을 누르세요.")
    output = widgets.Output(layout=widgets.Layout(width="100%"))
    state = {
        "results": [],
        "turns": [],
        "busy": False,
        "pending_clarification": None,
        "confirmed_businesses": [],
    }

    def submit(_button=None):
        if state["busy"]:
            return
        question = _clean_text(question_input.value)
        if not question:
            status.value = '<span style="color:#c62828"><b>질문을 입력하세요.</b></span>'
            return
        state["busy"] = True
        send_button.disabled = True
        clear_button.disabled = True
        status.value = '<span style="color:#1565c0"><b>질의분석·검색·답변 생성 중...</b></span>'
        try:
            with output:
                display(Markdown(f"---\n\n## 사용자 질문\n\n{question}"))
                result = ask_v15(
                    question,
                    previous_turns=state["turns"][-6:],
                    conversation_state=state,
                )
            state["results"].append(result)
            state["turns"].extend([
                {"role": "user", "content": question},
                {"role": "assistant", "content": result["basic_answer"]},
            ])
            question_input.value = ""
            status.value = "답변 생성 완료. 다음 질문을 입력할 수 있습니다."
        except Exception as error:
            with output:
                display(Markdown(
                    "### 오류\n\n"
                    + f"`{type(error).__name__}: {html.escape(str(error))}`"
                ))
            status.value = '<span style="color:#c62828"><b>오류가 발생했습니다.</b></span>'
        finally:
            state["busy"] = False
            send_button.disabled = False
            clear_button.disabled = False

    def clear(_button=None):
        state["results"].clear()
        state["turns"].clear()
        state["pending_clarification"] = None
        state["confirmed_businesses"].clear()
        output.clear_output()
        status.value = "대화를 지웠습니다."

    send_button.on_click(submit)
    clear_button.on_click(clear)
    display(widgets.VBox([
        question_input,
        widgets.HBox([send_button, clear_button]),
        status,
        output,
    ]))
    return state


chat_state = launch_v15_chat()

## 6. 함수로 직접 질문하기

```python
result = ask_v15("예금은 얼마까지 보호되나요?")
```

결과 객체의 주요 항목은 `route`, `analysis`, `search_plans`, `search_results`, `answer_skeleton`, `basic_answer`, `latency_ms`입니다.

## 12. Elasticsearch 상태 진단

검색 오류가 발생했을 때 아래 셀을 실행하면 서버, 플러그인, 인덱스, 문서 수, Nori 분석 결과를 한 번에 확인할 수 있습니다.


In [ ]:
def elasticsearch_diagnostics() -> dict[str, Any]:
    info = ES.info()
    nodes = ES.nodes.info(metric="plugins")
    plugins = sorted({
        str(plugin.get("name") or "")
        for node in nodes["nodes"].values()
        for plugin in node.get("plugins", [])
    })
    index_exists = bool(ES.indices.exists(index=ES_INDEX_NAME))
    document_count = int(ES.count(index=ES_INDEX_NAME)["count"]) if index_exists else None
    tokens = []
    if index_exists:
        tokens = [
            token["token"]
            for token in ES.indices.analyze(
                index=ES_INDEX_NAME,
                analyzer=ES_ANALYZER_NAME,
                text="착오송금 반환지원",
            )["tokens"]
        ]
    return {
        "connected": True,
        "url": ES_URL,
        "version": info["version"]["number"],
        "cluster_name": info["cluster_name"],
        "analysis_nori_installed": "analysis-nori" in plugins,
        "plugins": plugins,
        "index_name": ES_INDEX_NAME,
        "index_exists": index_exists,
        "document_count": document_count,
        "expected_document_count": len(CHUNKS),
        "nori_none_tokens": tokens,
    }


display(JSON(elasticsearch_diagnostics(), expanded=True))
